In [1]:
import subprocess, os, psutil, time

print("Proses Java yang sedang jalan:")
total = 0
pids = []
for p in psutil.process_iter(["pid", "name", "memory_info", "create_time"]):
    try:
        if "java" in (p.info["name"] or "").lower():
            mb = p.info["memory_info"].rss / 1e6
            total += mb
            pids.append(p.info["pid"])
            umur = (time.time() - p.info["create_time"]) / 60
            print(f"  PID {p.info['pid']:<8} {mb:>10,.0f} MB   umur {umur:>6.1f} menit")
    except Exception:
        pass

print(f"\nTotal RAM dipegang Java: {total:,.0f} MB dari {len(pids)} proses")

if len(pids) > 0:
    print("\nMematikan semua proses Java ...")
    subprocess.run(["taskkill", "/F", "/IM", "java.exe"],
                   capture_output=True, text=True)
    time.sleep(5)
    sisa = sum(1 for p in psutil.process_iter(["name"])
               if "java" in (p.info["name"] or "").lower())
    print(f"  sisa proses Java: {sisa}")

vm = psutil.virtual_memory()
print(f"\nRAM total   : {vm.total/1e9:.1f} GB")
print(f"RAM terpakai: {vm.used/1e9:.1f} GB")
print(f"RAM bebas   : {vm.available/1e9:.1f} GB")
if vm.available / 1e9 < 50:
    print("  !! Masih < 50 GB bebas. Tutup browser/aplikasi lain.")

Proses Java yang sedang jalan:

Total RAM dipegang Java: 0 MB dari 0 proses

RAM total   : 68.4 GB
RAM terpakai: 17.8 GB
RAM bebas   : 50.6 GB


# Influenza A Host Tropism — Big Data Pipeline (REVISI: profil hemat memori)

Revisi dari versi sebelumnya. Perubahan: heap JVM diperkecil, partisi diperkecil, setiap tahap berat diakhiri pembersihan memori eksplisit, dan seluruh loop iteratif di-checkpoint ke disk agar dapat dilanjutkan bila terhenti.

Pemetaan lapisan: Data Sources -> Ingestion -> Storage -> Platform Management -> Analytics Engines -> Data Warehouse -> Visualization. Security Layer dan Monitoring Layer melintasi seluruh lapisan.

## LAPISAN 0 — BOOTSTRAP / DEPENDENCY MANAGEMENT

In [2]:
import sys, subprocess

PAKET = ["pyspark", "pandas", "numpy", "scipy", "matplotlib", "scikit-learn",
         "pyarrow", "requests", "biopython", "networkx", "plotly", "psutil"]
for p in PAKET:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

import importlib
for p in ["pyspark","pandas","numpy","scipy","matplotlib","sklearn",
          "pyarrow","requests","Bio","networkx","plotly","psutil"]:
    try:
        m = importlib.import_module(p)
        print(f"  {p:<12} {getattr(m,'__version__','ok')}")
    except Exception as e:
        print(f"  {p:<12} GAGAL {e}")

  pyspark      3.5.9
  pandas       2.3.2
  numpy        1.26.4
  scipy        1.16.3
  matplotlib   3.10.7
  sklearn      1.7.2
  pyarrow      23.0.1
  requests     2.32.5
  Bio          1.86
  networkx     3.5
  plotly       6.5.2
  psutil       7.1.3


In [4]:
import os, sys, re, gzip, json, time, math, hashlib, random, warnings, platform
import itertools, collections, gc
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import psutil

warnings.filterwarnings("ignore")

BASE   = Path(r"D:\BDA")
LAKE, STAGE, MART   = BASE / "lake", BASE / "stage", BASE / "mart"
SEQ, OUT, TMP       = BASE / "sequences", BASE / "output", BASE / "spark_tmp"
LOGDIR, AUDIT, CKPT = BASE / "spark_events", BASE / "audit", BASE / "checkpoint"
for d in (BASE, LAKE, STAGE, MART, SEQ, OUT, TMP, LOGDIR, AUDIT, CKPT):
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    "run_id":         datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ"),
    "mode":           "UJI",
    "subsampel":      40000,       # turunkan ke 15000 bila masih berat
    "k_mer":          10,
    "vocab":          1 << 18,
    "n_fold":         5,
    "seed":           42,
    "ambang_jaccard": 0.95,
    "n_hash_lsh":     5,
    "lsh_top_k":      30,
    "lsh_n_blok":     40,
    # ---------- PROFIL HEMAT MEMORI ----------
    # Heap kecil memaksa Spark menumpahkan ke disk lebih awal.
    # Lebih lambat sedikit, tapi JVM tidak pernah dibunuh OS.
    "spark_master":   "local[4]",
    "driver_memory":  "12g",
    "shuffle_part":   64,          # untuk k-mer, ML, mart
    "part_graf":      32,          # untuk operasi graf iteratif
    "ncbi_batch":     200,
    "ncbi_delay":     0.34,
    "taxid":          11320,
}
SEGMEN = {"HA": "4", "NA": "6"}
K, VOCAB, N_FOLD, SEED = CFG["k_mer"], CFG["vocab"], CFG["n_fold"], CFG["seed"]
FOLDS = ["fold_acak", "fold_spasial", "fold_temporal", "fold_cluster"]
random.seed(SEED); np.random.seed(SEED)

vm = psutil.virtual_memory()
print(json.dumps(CFG, indent=2))
print(f"\nHost {platform.node()} | Python {sys.version.split()[0]} | CPU {os.cpu_count()}")
print(f"RAM {vm.total/1e9:.1f} GB total, {vm.available/1e9:.1f} GB bebas")
heap_rencana = int(CFG["driver_memory"].rstrip("g"))
print(f"Heap rencana {heap_rencana} GB -> margin "
      f"{vm.available/1e9 - heap_rencana:.1f} GB")
print("\nProfil HEMAT aktif: heap kecil, partisi kecil, checkpoint ke disk.")

{
  "run_id": "run_20260827T152621Z",
  "mode": "UJI",
  "subsampel": 40000,
  "k_mer": 10,
  "vocab": 262144,
  "n_fold": 5,
  "seed": 42,
  "ambang_jaccard": 0.95,
  "n_hash_lsh": 5,
  "lsh_top_k": 30,
  "lsh_n_blok": 40,
  "spark_master": "local[4]",
  "driver_memory": "22g",
  "shuffle_part": 600,
  "ncbi_batch": 200,
  "ncbi_delay": 0.34,
  "taxid": 11320
}
Host Willy-Boen | Python 3.11.14 | CPU 28
RAM 68.4 GB total, 50.7 GB bebas


## LAPISAN SECURITY — AUDIT TRAIL, PSEUDONIMISASI, KONTROL AKSES, PEMINDAIAN PII, DATA QUALITY GATE

In [5]:
AUDIT_LOG = AUDIT / (CFG["run_id"] + ".jsonl")

def audit(peristiwa, **rincian):
    rec = {"ts": datetime.now(timezone.utc).isoformat(),
           "run_id": CFG["run_id"], "peristiwa": peristiwa,
           "aktor": os.environ.get("USERNAME", "unknown"), **rincian}
    with open(AUDIT_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
    return rec

SALT = hashlib.sha256(CFG["run_id"].encode()).digest()

def pseudonim(nilai):
    if nilai is None or (isinstance(nilai, float) and nilai != nilai):
        return None
    return hashlib.blake2b(str(nilai).encode(), key=SALT, digest_size=12).hexdigest()

KLASIFIKASI = {
    "accession":        ("PUBLIC",     "Identifier GenBank publik"),
    "seq":              ("PUBLIC",     "Sekuens nukleotida publik"),
    "inang":            ("PUBLIC",     "Kategori spesies inang"),
    "benua":            ("PUBLIC",     "Geografis kasar"),
    "tahun":            ("PUBLIC",     "Tahun koleksi"),
    "serotype":         ("PUBLIC",     "Subtipe HA/NA"),
    "segmen":           ("PUBLIC",     "Nomor segmen"),
    "negara":           ("INTERNAL",   "Geografis menengah"),
    "strain":           ("INTERNAL",   "Dapat memuat lokasi granular"),
    "collection_date":  ("INTERNAL",   "Tanggal presisi hari"),
    "isolation_source": ("RESTRICTED", "Konteks klinis"),
    "lat_lon":          ("RESTRICTED", "Koordinat presisi"),
}
PERAN = {"publik": {"PUBLIC"},
         "peneliti": {"PUBLIC", "INTERNAL"},
         "pengelola": {"PUBLIC", "INTERNAL", "RESTRICTED"}}
PERAN_AKTIF = "peneliti"

def terapkan_kontrol_akses(df, peran=PERAN_AKTIF):
    izin = PERAN[peran]
    buang, samar = [], []
    for kol in df.columns:
        kelas = KLASIFIKASI.get(kol, ("PUBLIC", ""))[0]
        if kelas not in izin:
            (buang if kelas == "RESTRICTED" else samar).append(kol)
    hasil = df.drop(columns=buang, errors="ignore").copy()
    for kol in samar:
        hasil[kol] = hasil[kol].map(pseudonim)
    audit("kontrol_akses", peran=peran, dibuang=buang, dipseudonimkan=samar)
    return hasil

POLA_PII = {
    "email":     re.compile(r"[\w\.-]+@[\w\.-]+\.\w+"),
    "telepon":   re.compile(r"\+?\d[\d\s\-]{8,}\d"),
    "koordinat": re.compile(r"[-+]?\d{1,3}\.\d{4,}\s*,\s*[-+]?\d{1,3}\.\d{4,}"),
    "nik":       re.compile(r"\b\d{16}\b"),
}

def pindai_pii(df, n_sampel=20000):
    sub = df.head(n_sampel)
    temuan = {}
    for kol in sub.columns:
        if kol == "seq" or sub[kol].dtype != object:
            continue
        teks = sub[kol].dropna().astype(str)
        for nama, pola in POLA_PII.items():
            n = int(teks.str.contains(pola, regex=True, na=False).sum())
            if n:
                temuan[kol + ":" + nama] = n
    audit("pindai_pii", temuan=temuan, n_sampel=int(len(sub)))
    return temuan

def gerbang_kualitas(df, aturan, nama_tahap):
    lapor, lolos = [], True
    for nama, fn, wajib in aturan:
        try:
            nilai = fn(df); ok = bool(nilai)
        except Exception as ex:
            nilai, ok = "ERROR " + str(ex), False
        if wajib and not ok:
            lolos = False
        lapor.append({"tahap": nama_tahap, "aturan": nama,
                      "hasil": str(nilai), "lolos": ok, "wajib": wajib})
    dfl = pd.DataFrame(lapor)
    audit("gerbang_kualitas", tahap=nama_tahap, lolos=lolos)
    print(dfl.to_string(index=False))
    if not lolos:
        print("  !! GERBANG KUALITAS GAGAL: " + nama_tahap)
    return lolos, dfl

audit("mulai_run", cfg={k: str(v) for k, v in CFG.items()})
print("Audit log :", AUDIT_LOG)
print("Peran     :", PERAN_AKTIF, "->", sorted(PERAN[PERAN_AKTIF]))
print()
print(pd.DataFrame([{"kolom": k, "klasifikasi": v[0], "catatan": v[1]}
                    for k, v in KLASIFIKASI.items()]).to_string(index=False))

Audit log : D:\BDA\audit\run_20260827T152621Z.jsonl
Peran     : peneliti -> ['INTERNAL', 'PUBLIC']

           kolom klasifikasi                      catatan
       accession      PUBLIC    Identifier GenBank publik
             seq      PUBLIC    Sekuens nukleotida publik
           inang      PUBLIC       Kategori spesies inang
           benua      PUBLIC              Geografis kasar
           tahun      PUBLIC                Tahun koleksi
        serotype      PUBLIC                Subtipe HA/NA
          segmen      PUBLIC                 Nomor segmen
          negara    INTERNAL           Geografis menengah
          strain    INTERNAL Dapat memuat lokasi granular
 collection_date    INTERNAL         Tanggal presisi hari
isolation_source  RESTRICTED               Konteks klinis
         lat_lon  RESTRICTED            Koordinat presisi


## LAPISAN MONITORING — INSTRUMENTASI TAHAP BATCH

In [5]:
import psutil

MONITOR = []

class Tahap:
    def __init__(self, nama, lapisan):
        self.nama, self.lapisan = nama, lapisan
    def __enter__(self):
        self.t0 = time.time()
        self.m0 = psutil.virtual_memory().used / 1e9
        psutil.cpu_percent(interval=None)
        print("[" + self.lapisan + "] " + self.nama + " ... mulai")
        audit("tahap_mulai", tahap=self.nama, lapisan=self.lapisan)
        return self
    def __exit__(self, *a):
        dt = time.time() - self.t0
        rec = {"lapisan": self.lapisan, "tahap": self.nama,
               "detik": round(dt, 2), "menit": round(dt / 60, 3),
               "mem_delta_gb": round(psutil.virtual_memory().used/1e9 - self.m0, 2),
               "mem_total_gb": round(psutil.virtual_memory().used/1e9, 2),
               "cpu_pct": psutil.cpu_percent(interval=None),
               "status": "GAGAL" if a[0] else "OK"}
        MONITOR.append(rec)
        audit("tahap_selesai", **rec)
        print(f"[{self.lapisan}] {self.nama} -> {dt/60:.2f} menit, "
              f"mem {rec['mem_total_gb']} GB, status {rec['status']}")
        return False

def simpan_monitor():
    dfm = pd.DataFrame(MONITOR)
    if len(dfm):
        dfm.to_csv(OUT / "monitoring_tahap.csv", index=False)
    return dfm

print("Monitoring aktif.")

Monitoring aktif.


## PEMANDU EKSEKUSI | status pipeline & titik lanjut

In [ ]:
# ============================================================
# PEMANDU — sel mana yang perlu dijalankan berikutnya
# Membaca keadaan nyata di memori dan di disk.
# ============================================================
def status_pipeline():
    g = globals()
    TAHAP = [
        ("import + CFG",       "sel 3, 4",   lambda: "CFG" in g),
        ("security",           "sel 6",      lambda: "audit" in g),
        ("monitoring",         "sel 8",      lambda: "Tahap" in g),
        ("fungsi normalisasi", "sel 18",     lambda: "petakan_benua" in g),
        ("winutils",           "sel 24",     lambda: "WINUTILS_OK" in g),
        ("SparkSession",       "sel 26",     lambda: "spark" in g and "bersihkan" in g),
        ("data HA + k-mer",    "sel 28, 29", lambda: "kmer_df" in g and "N" in g),
        ("TF-IDF + fitur",     "sel 31",     lambda: "N_FITUR" in g),
        ("graf LSH",           "sel 33",     lambda: "sisi" in g and "n_sisi" in g),
        ("komponen",           "sel 34",     lambda: "n_klaster" in g),
        ("4 kolom fold",       "sel 36",     lambda: "kerja" in g),
        ("ML Spark",           "sel 39",     lambda: "per_fold" in g),
        ("laporan klasifikasi","sel 41",     lambda: "lap" in g),
        ("statistik",          "sel 43, 44", lambda: "df_uji" in g),
        ("sklearn",            "sel 46",     lambda: "sk_per_fold" in g),
        ("banding mesin",      "sel 48",     lambda: "ringkas_mesin" in g),
        ("validasi biologis",  "sel 50-54",  lambda: "df_enrich" in g),
        ("zoonotik + NA",      "sel 56, 57", lambda: "df_na" in g),
        ("restore graf",       "sel 59",     lambda: "sisi2" in g),
        ("derajat & komponen", "sel 60",     lambda: "simpul_df" in g),
        ("PageRank",           "sel 61",     lambda: "pr" in g),
        ("asortativitas",      "sel 62",     lambda: "GRAF_STAT" in g),
        ("G1 property graph",  "sel 64",     lambda: "vertices" in g),
        ("G2 centrality",      "sel 66",     lambda: "pusat_pdf" in g and "sent" in g),
        ("G3 community",       "sel 68",     lambda: "KOM_STAT" in g),
        ("G4 connectivity",    "sel 70",     lambda: "KONEKTIVITAS" in g),
        ("G5 path",            "sel 72",     lambda: "PATH_STAT" in g),
        ("G6 sintesis",        "sel 74",     lambda: "GRAF_LENGKAP" in g),
        ("mart",               "sel 76",     lambda: "uk_komp_pdf" in g),
        ("benchmark",          "sel 78",     lambda: "df_bench" in g),
        ("event log",          "sel 80",     lambda: "df_tahap" in g),
        ("figur",              "sel 82",     lambda: (OUT/"ringkasan_visual.png").exists()),
        ("dashboard",          "sel 84",     lambda: (OUT/"dashboard.html").exists()),
        ("ringkasan akhir",    "sel 86, 87", lambda: (OUT/"ringkasan_akhir.csv").exists()),
    ]

    print("=" * 68)
    print(" STATUS PIPELINE")
    print("=" * 68)
    berikutnya = None
    for nama, sel, cek in TAHAP:
        try:
            ok = bool(cek())
        except Exception:
            ok = False
        tanda = "OK " if ok else "-- "
        if not ok and berikutnya is None:
            berikutnya, tanda = (nama, sel), ">> "
        print(f"  {tanda} {nama:<22} {sel}")
    print("=" * 68)
    if berikutnya:
        print(f"  JALANKAN BERIKUTNYA: {berikutnya[1]}   ({berikutnya[0]})")
    else:
        print("  Seluruh pipeline selesai.")

    sub = CFG["subsampel"] or "penuh"
    amb = int(CFG["ambang_jaccard"] * 100)
    print()
    print("Checkpoint di disk (memungkinkan melompati tahap):")
    for nama, pth in [
        ("stage bersih",   STAGE / "flua_seq_bersih.parquet"),
        ("k-mer",          STAGE / f"kmer_ha_k{CFG['k_mer']}_{sub}.parquet"),
        ("vektor biner",   STAGE / "vek_biner.parquet"),
        ("graf sisi",      STAGE / f"graf_kemiripan_j{amb}_k{CFG['lsh_top_k']}.parquet"),
        ("sisi dua arah",  STAGE / "sisi_dua_arah.parquet"),
        ("label komponen", STAGE / "lp_iter"),
        ("fold lengkap",   STAGE / "fold_lengkap.parquet"),
        ("PageRank",       STAGE / "pr_iter"),
        ("eigenvector",    STAGE / "ev_iter"),
        ("centrality",     STAGE / "centrality.parquet"),
    ]:
        ada = pth.exists() and any(pth.rglob("*.parquet"))
        mb = (sum(f.stat().st_size for f in pth.rglob("*.parquet")) / 1e6) if ada else 0
        print(f"  {'ADA ' if ada else '--  '} {nama:<16} {mb:>9,.1f} MB")

    vm = psutil.virtual_memory()
    print(f"\nRAM bebas: {vm.available/1e9:.1f} GB")
    if "DF_TERLACAK" in g:
        print(f"DataFrame ter-cache: {[n for n, _ in g['DF_TERLACAK']] or 'tidak ada'}")
    return berikutnya


status_pipeline()


In [6]:
# ============================================================
# RESUME CEPAT — lompati LAPISAN 1-3 bila artefaknya sudah ada
# ============================================================
LEWATI_INGESTION = True     # set False bila ingin menjalankan ulang dari NCBI

ARTEFAK = {
    "metadata mentah":   LAKE / "metadata_mentah.parquet",
    "stage bersih":      STAGE / "flua_seq_bersih.parquet",
    "stage berpartisi":  STAGE / "flua_partisi",
    "FASTA HA":          SEQ / "flua_HA.fasta.gz",
    "FASTA NA":          SEQ / "flua_NA.fasta.gz",
}

print("=" * 62)
print(" STATUS ARTEFAK LAPISAN 1-3")
print("=" * 62)
lengkap = True
for nama, p in ARTEFAK.items():
    if p.exists():
        mb = (sum(f.stat().st_size for f in p.rglob("*")) / 1e6
              if p.is_dir() else p.stat().st_size / 1e6)
        print(f"  {nama:<20} ADA    {mb:>10,.1f} MB")
    else:
        print(f"  {nama:<20} TIDAK ADA")
        lengkap = False

st_ing = {}
if (CKPT / "ingest_state.json").exists():
    st_ing = json.loads((CKPT / "ingest_state.json").read_text(encoding="utf-8"))
    print()
    for k_, v_ in st_ing.items():
        print(f"  {k_:<20} {v_.get('cakupan_pct', 0)}% "
              f"{'SELESAI' if v_.get('selesai') else 'BELUM'}")

print()
if lengkap and LEWATI_INGESTION:
    print("  -> LAPISAN 1, 2, dan 3 BISA DILEWATI.")
    print("     Lanjut langsung ke sel LAPISAN 4 (bangun_spark).")
    print("     Yang tetap harus dijalankan tiap restart kernel:")
    print("       sel 3, 4 (import + CFG), sel 6 (SECURITY), sel 8 (MONITORING),")
    print("       sel winutils, sel pra-flight, lalu LAPISAN 4.")
    LEWATI_L123 = True
else:
    print("  -> Artefak belum lengkap. Jalankan LAPISAN 1-3.")
    LEWATI_L123 = False

 STATUS ARTEFAK LAPISAN 1-3
  metadata mentah      ADA          31.6 MB
  stage bersih         ADA         157.4 MB
  stage berpartisi     ADA       1,700.2 MB
  FASTA HA             ADA          38.5 MB
  FASTA NA             ADA          28.0 MB

  meta_HA              100.0% SELESAI
  meta_NA              100.0% SELESAI

  -> LAPISAN 1, 2, dan 3 BISA DILEWATI.
     Lanjut langsung ke sel LAPISAN 4 (bangun_spark).
     Yang tetap harus dijalankan tiap restart kernel:
       sel 3, 4 (import + CFG), sel 6 (SECURITY), sel 8 (MONITORING),
       sel winutils, sel pra-flight, lalu LAPISAN 4.


## LAPISAN 1 — DATA SOURCES | NCBI Entrez: FASTA unstructured + metadata semi-terstruktur

In [7]:
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
SESI = requests.Session()
SESI.headers.update({"User-Agent": "BDA-Influenza-Pipeline/1.0"})


def json_aman(teks):
    """NCBI kadang mengirim control character mentah yang tidak di-escape."""
    try:
        return json.loads(teks, strict=False)
    except json.JSONDecodeError:
        bersih = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", teks)
        return json.loads(bersih, strict=False)


def eutils_get(endpoint, params, percobaan=5, timeout=120):
    for i in range(percobaan):
        try:
            r = SESI.get(EUTILS + "/" + endpoint, params=params, timeout=timeout)
            r.raise_for_status()
            time.sleep(CFG["ncbi_delay"])
            return r
        except Exception as ex:
            tunggu = min(2 ** i, 30)
            print(f"    retry {i+1}/{percobaan} {type(ex).__name__} tunggu {tunggu}s")
            time.sleep(tunggu)
    raise RuntimeError("eutils gagal: " + endpoint)


with Tahap("probe sumber data", "DATA SOURCES"):
    inv = []
    for seg, sid in SEGMEN.items():
        r = eutils_get("esearch.fcgi", {
            "db": "nuccore",
            "term": f'txid{CFG["taxid"]}[Organism] AND {sid}[Segment]',
            "retmode": "json", "retmax": 0})
        n = int(json_aman(r.text)["esearchresult"]["count"])
        inv.append({"segmen": seg, "segment_id": sid, "jumlah": n})
        print(f"  segmen {seg} (id {sid}): {n:,}")
    df_inv = pd.DataFrame(inv)
    df_inv["volume_estimasi_mb"] = (df_inv["jumlah"] * 1700 / 1e6).round(1)
    df_inv.to_csv(OUT / "inventaris_sumber.csv", index=False)
    audit("inventaris_sumber", data=df_inv.to_dict("records"))

print()
print(df_inv.to_string(index=False))

[DATA SOURCES] probe sumber data ... mulai
  segmen HA (id 4): 362,909
  segmen NA (id 6): 287,444
[DATA SOURCES] probe sumber data -> 0.02 menit, mem 19.57 GB, status OK

segmen segment_id  jumlah  volume_estimasi_mb
    HA          4  362909               616.9
    NA          6  287444               488.7


## LAPISAN 2 — INGESTION LAYER | Batch harvest berjendela + checkpoint idempotent

In [8]:
BATCH_GAGAL = []
INGEST_STATE = CKPT / "ingest_state.json"


def muat_state():
    if INGEST_STATE.exists():
        try:
            return json.loads(INGEST_STATE.read_text(encoding="utf-8"))
        except Exception:
            pass
    return {}


def simpan_state(st):
    INGEST_STATE.write_text(json.dumps(st, indent=2), encoding="utf-8")


def harvest_metadata(segmen, segment_id, paksa=False):
    ckpt = CKPT / ("meta_" + segmen + ".jsonl")
    st = muat_state()
    kunci = f"meta_{segmen}"

    # --- gerbang cepat: kalau sudah pernah tuntas, langsung keluar ---
    if not paksa and st.get(kunci, {}).get("selesai"):
        info = st[kunci]
        print(f"  {segmen}: SUDAH SELESAI ({info['tersimpan']:,}/{info['total']:,}"
              f" = {info['cakupan_pct']}%) -- dilewati")
        return info["total"], info["tersimpan"]

    # hitung baris tanpa parsing JSON: jauh lebih cepat
    n_baris = 0
    if ckpt.exists():
        with open(ckpt, "rb") as f:
            n_baris = sum(1 for _ in f)
        print(f"  {segmen}: checkpoint berisi {n_baris:,} baris")

    r = eutils_get("esearch.fcgi", {
        "db": "nuccore",
        "term": f'txid{CFG["taxid"]}[Organism] AND {segment_id}[Segment]',
        "usehistory": "y", "retmode": "json", "retmax": 0})
    js = json_aman(r.text)["esearchresult"]
    total, webenv, qkey = int(js["count"]), js["webenv"], js["querykey"]
    print(f"  {segmen}: target NCBI {total:,}")

    # kalau checkpoint sudah >= 98% target, anggap tuntas
    if not paksa and n_baris >= 0.98 * total:
        st[kunci] = {"total": total, "tersimpan": n_baris, "selesai": True,
                     "cakupan_pct": round(100 * n_baris / max(total, 1), 2),
                     "waktu": datetime.now(timezone.utc).isoformat()}
        simpan_state(st)
        print(f"  {segmen}: cakupan {st[kunci]['cakupan_pct']}% -> ditandai SELESAI")
        return total, n_baris

    # baru di sini muat uid ke memori (mahal, hanya bila memang perlu lanjut)
    sudah = set()
    if ckpt.exists():
        with open(ckpt, encoding="utf-8") as f:
            for baris in f:
                try:
                    sudah.add(json.loads(baris)["uid"])
                except Exception:
                    pass
        print(f"  {segmen}: {len(sudah):,} uid unik dimuat")

    # lanjut dari offset terakhir, bukan dari nol
    mulai = st.get(kunci, {}).get("offset", 0)
    B = CFG["ncbi_batch"]
    print(f"  {segmen}: lanjut dari retstart={mulai:,}")

    with open(ckpt, "a", encoding="utf-8") as fh:
        for start in range(mulai, total, B):
            if start % (B * 25) == 0:
                print(f"    {start:,}/{total:,} ({100*start/max(total,1):.1f}%)")
                st[kunci] = {"total": total, "tersimpan": len(sudah),
                             "offset": start, "selesai": False,
                             "cakupan_pct": round(100*len(sudah)/max(total,1), 2)}
                simpan_state(st)

            res = None
            for strategi in ["json", "json_kecil", "xml"]:
                try:
                    if strategi == "json":
                        rs = eutils_get("esummary.fcgi", {
                            "db": "nuccore", "query_key": qkey, "WebEnv": webenv,
                            "retstart": start, "retmax": B, "retmode": "json"})
                        res = json_aman(rs.text).get("result", {})
                    elif strategi == "json_kecil":
                        res = {"uids": []}
                        for sub in range(start, min(start + B, total), 20):
                            rs = eutils_get("esummary.fcgi", {
                                "db": "nuccore", "query_key": qkey, "WebEnv": webenv,
                                "retstart": sub, "retmax": 20, "retmode": "json"})
                            try:
                                d = json_aman(rs.text).get("result", {})
                            except Exception:
                                continue
                            for u in d.get("uids", []):
                                res["uids"].append(u)
                                res[u] = d[u]
                    else:
                        import xml.etree.ElementTree as ET
                        rs = eutils_get("esummary.fcgi", {
                            "db": "nuccore", "query_key": qkey, "WebEnv": webenv,
                            "retstart": start, "retmax": B, "retmode": "xml"})
                        akar = ET.fromstring(re.sub(
                            r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", rs.text))
                        res = {"uids": []}
                        for doc in akar.findall(".//DocSum"):
                            uid = doc.findtext("Id")
                            item = {it.get("Name"): (it.text or "")
                                    for it in doc.findall("Item")}
                            res["uids"].append(uid)
                            res[uid] = {
                                "accessionversion": item.get("AccessionVersion", uid),
                                "title": item.get("Title", ""),
                                "slen": item.get("Slen"),
                                "subtype": item.get("SubType", ""),
                                "subname": item.get("SubName", "")}
                    break
                except Exception as ex:
                    print(f"    batch {start} strategi={strategi} gagal: "
                          f"{type(ex).__name__}")
                    res = None

            if res is None:
                BATCH_GAGAL.append({"segmen": segmen, "retstart": start, "retmax": B})
                audit("batch_gagal", segmen=segmen, retstart=start)
                continue

            for uid in res.get("uids", []):
                if uid in sudah:
                    continue
                rec = res[uid]
                fh.write(json.dumps({
                    "uid": uid,
                    "accession": rec.get("accessionversion", uid),
                    "title": rec.get("title", ""),
                    "slen": rec.get("slen"),
                    "subtype": rec.get("subtype", ""),
                    "subname": rec.get("subname", ""),
                    "segmen": segmen}, ensure_ascii=False) + "\n")
                sudah.add(uid)

    st[kunci] = {"total": total, "tersimpan": len(sudah), "offset": total,
                 "selesai": len(sudah) >= 0.98 * total,
                 "cakupan_pct": round(100 * len(sudah) / max(total, 1), 2),
                 "waktu": datetime.now(timezone.utc).isoformat()}
    simpan_state(st)
    audit("ingest_metadata", segmen=segmen, total=total, tersimpan=len(sudah))
    return total, len(sudah)


with Tahap("harvest metadata batch", "INGESTION"):
    lapor_ingest = []
    for seg, sid in SEGMEN.items():
        tot, sim = harvest_metadata(seg, sid)
        lapor_ingest.append({"segmen": seg, "target": tot, "tersimpan": sim,
                             "cakupan_pct": round(100 * sim / max(tot, 1), 2)})
    df_ing = pd.DataFrame(lapor_ingest)
    df_ing.to_csv(OUT / "cakupan_ingestion.csv", index=False)

print()
print(df_ing.to_string(index=False))
print(f"\nBatch gagal permanen: {len(BATCH_GAGAL)}")

[INGESTION] harvest metadata batch ... mulai
  HA: SUDAH SELESAI (362,843/362,843 = 100.0%) -- dilewati
  NA: SUDAH SELESAI (287,378/287,378 = 100.0%) -- dilewati
[INGESTION] harvest metadata batch -> 0.00 menit, mem 19.57 GB, status OK

segmen  target  tersimpan  cakupan_pct
    HA  362843     362843        100.0
    NA  287378     287378        100.0

Batch gagal permanen: 0


In [9]:
def unduh_fasta(segmen, segment_id):
    keluar = SEQ / ("flua_" + segmen + ".fasta.gz")
    ckpt   = CKPT / ("fasta_" + segmen + ".state.json")
    state  = {"offset": 0, "total": None, "webenv": None, "qkey": None}
    if ckpt.exists():
        state.update(json.loads(ckpt.read_text()))

    if state["webenv"] is None:
        r = eutils_get("esearch.fcgi", {
            "db": "nuccore",
            "term": f'txid{CFG["taxid"]}[Organism] AND {segment_id}[Segment]',
            "usehistory": "y", "retmode": "json", "retmax": 0})
        js = json_aman(r.text)["esearchresult"]
        state.update({"total": int(js["count"]), "webenv": js["webenv"],
                      "qkey": js["querykey"]})
        ckpt.write_text(json.dumps(state))

    total, B = state["total"], 500
    mode = "ab" if state["offset"] > 0 and keluar.exists() else "wb"
    print(f"  {segmen}: lanjut offset {state['offset']:,}/{total:,}")

    with gzip.open(keluar, mode) as fo:
        while state["offset"] < total:
            if state["offset"] % (B * 20) == 0:
                print(f"    {state['offset']:,}/{total:,} "
                      f"({100*state['offset']/max(total,1):.1f}%)")
            try:
                r = eutils_get("efetch.fcgi", {
                    "db": "nuccore", "query_key": state["qkey"],
                    "WebEnv": state["webenv"], "retstart": state["offset"],
                    "retmax": B, "rettype": "fasta", "retmode": "text"},
                    timeout=300)
                fo.write(r.text.encode("utf-8")); fo.flush()
            except Exception as ex:
                print(f"    offset {state['offset']} gagal {ex}")
                time.sleep(10); continue
            state["offset"] += B
            ckpt.write_text(json.dumps(state))
    audit("ingest_fasta", segmen=segmen, total=total,
          ukuran_mb=round(keluar.stat().st_size / 1e6, 1))
    return keluar


with Tahap("unduh FASTA batch", "INGESTION"):
    for seg, sid in SEGMEN.items():
        p = unduh_fasta(seg, sid)
        print(f"  {seg}: {p.stat().st_size/1e6:.1f} MB")

[INGESTION] unduh FASTA batch ... mulai
  HA: lanjut offset 363,000/362,843
  HA: 38.5 MB
  NA: lanjut offset 287,500/287,378
  NA: 28.0 MB
[INGESTION] unduh FASTA batch -> 0.00 menit, mem 19.59 GB, status OK


## LAPISAN 3 — STORAGE LAYER | Zona lake -> stage -> mart, Parquet kolumnar berpartisi (setara HDFS)

In [10]:
import sys, subprocess
for p in ["pycountry", "pycountry-convert"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

import pycountry
from pycountry_convert import country_alpha2_to_continent_code

# ============================================================
# BENUA — otomatis via ISO 3166 (pycountry), tanpa daftar manual
# ============================================================
KODE_BENUA = {"AS": "Asia", "EU": "Eropa", "NA": "Amerika Utara",
              "SA": "Amerika Selatan", "AF": "Afrika", "OC": "Oseania",
              "AN": "Antartika"}

CACHE_NEGARA = STAGE / "cache_negara.json"

def muat_cache(path):
    """Baca cache JSON dengan aman; kembalikan dict kosong bila rusak."""
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as ex:
        rusak = path.with_suffix(path.suffix + ".rusak")
        try:
            path.replace(rusak)
            print(f"  cache {path.name} rusak ({type(ex).__name__}), "
                  f"dipindah ke {rusak.name}, dibangun ulang")
        except Exception:
            print(f"  cache {path.name} rusak, diabaikan")
        return {}

def simpan_cache(path, data):
    path.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")

_pn = muat_cache(CACHE_NEGARA)


def resolve_negara(nama):
    """Nama negara bebas -> (nama ISO, alpha_2). Berlapis, tanpa daftar manual."""
    if not isinstance(nama, str) or not nama.strip():
        return None, None
    kunci = nama.split(":")[0].strip()
    if not kunci:
        return None, None
    if kunci in _pn:
        v = _pn[kunci]
        return (v[0], v[1]) if v else (None, None)

    hasil = None
    for cara in ("alpha_2", "alpha_3", "name", "official_name", "common_name"):
        try:
            c = pycountry.countries.get(**{cara: kunci})
            if c:
                hasil = c
                break
        except Exception:
            pass
    if hasil is None:
        try:
            hasil = pycountry.countries.lookup(kunci)
        except LookupError:
            pass
    if hasil is None:
        try:
            hasil = pycountry.countries.search_fuzzy(kunci)[0]
        except Exception:
            hasil = None

    _pn[kunci] = [hasil.name, hasil.alpha_2] if hasil else None
    return (hasil.name, hasil.alpha_2) if hasil else (None, None)


def petakan_benua(nama):
    _, a2 = resolve_negara(nama)
    if not a2:
        return "Tidak terpetakan"
    try:
        return KODE_BENUA.get(country_alpha2_to_continent_code(a2),
                              "Tidak terpetakan")
    except KeyError:
        return "Tidak terpetakan"


# ============================================================
# INANG — otomatis via NCBI Taxonomy lineage, bukan regex kata kunci
# ============================================================
CACHE_INANG = STAGE / "cache_inang.json"
_pi = muat_cache(CACHE_INANG)

# Tiga baris ini menggantikan seluruh daftar POLA_INANG manual
KLAD_INANG = [("Homo sapiens", "manusia"),
              ("Aves",         "unggas"),
              ("Suidae",       "babi")]


def resolve_inang_batch(daftar_nama, ukuran=100):
    """Resolusi nama inang -> kategori lewat lineage taksonomi NCBI."""
    import xml.etree.ElementTree as ET
    belum = [n for n in daftar_nama
             if isinstance(n, str) and n.strip() and n not in _pi]
    print(f"  perlu resolusi taksonomi: {len(belum):,} nama unik")

    for i in range(0, len(belum), ukuran):
        blok = belum[i:i + ukuran]
        if i % (ukuran * 10) == 0:
            print(f"    {i:,}/{len(belum):,}")

        peta_uid = {}
        for nm in blok:
            bersih = re.split(r"[;(]", nm)[0].strip()
            try:
                r = eutils_get("esearch.fcgi", {
                    "db": "taxonomy", "term": bersih,
                    "retmode": "json", "retmax": 1})
                ids = json_aman(r.text)["esearchresult"].get("idlist", [])
                if ids:
                    peta_uid[ids[0]] = peta_uid.get(ids[0], []) + [nm]
                else:
                    _pi[nm] = "lain"
            except Exception:
                _pi[nm] = "lain"

        if not peta_uid:
            continue

        try:
            r = eutils_get("efetch.fcgi", {
                "db": "taxonomy", "id": ",".join(peta_uid),
                "retmode": "xml"}, timeout=180)
            akar = ET.fromstring(re.sub(
                r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", r.text))
            for tx in akar.findall(".//Taxon"):
                uid = tx.findtext("TaxId")
                if uid not in peta_uid:
                    continue
                garis = ((tx.findtext("Lineage") or "") + ";" +
                         (tx.findtext("ScientificName") or ""))
                kategori = "lain"
                for klad, label in KLAD_INANG:
                    if klad in garis:
                        kategori = label
                        break
                for nm in peta_uid[uid]:
                    _pi[nm] = kategori
        except Exception as ex:
            print(f"    efetch taxonomy gagal: {type(ex).__name__}")
            for lst in peta_uid.values():
                for nm in lst:
                    _pi.setdefault(nm, "lain")

    simpan_cache(CACHE_INANG, _pi)
    return _pi


def petakan_inang(v):
    if not isinstance(v, str):
        return "lain"
    return _pi.get(v, "lain")


# ============================================================
# SISA FUNGSI NORMALISASI
# ============================================================
def ambil_tahun(v):
    if not isinstance(v, str):
        return None
    m = re.search(r"(19|20)\d{2}", v)
    return int(m.group(0)) if m else None


def ambil_serotype(judul, sero):
    for kandidat in (sero, judul):
        if isinstance(kandidat, str):
            m = re.search(r"\bH\d{1,2}N\d{1,2}\b", kandidat)
            if m:
                return m.group(0)
    return None


def parse_subname(subtype, subname):
    if not subtype:
        return {}
    return dict(zip(str(subtype).split("|"), str(subname).split("|")))


def baca_fasta_gz(path):
    rec, h, buf = {}, None, []
    with gzip.open(path, "rt", errors="ignore") as f:
        for baris in f:
            baris = baris.rstrip()
            if not baris:
                continue
            if baris.startswith(">"):
                if h: rec[h] = "".join(buf)
                h, buf = baris[1:].split()[0], []
            else:
                buf.append(baris.upper())
        if h: rec[h] = "".join(buf)
    return rec


AMBIGU = set("NRYKMSWBDHV")
print("Normalisasi otomatis siap: ISO 3166 (benua) + NCBI Taxonomy (inang).")

Normalisasi otomatis siap: ISO 3166 (benua) + NCBI Taxonomy (inang).


In [11]:
with Tahap("normalisasi metadata", "STORAGE"):
    baris = []
    for seg in SEGMEN:
        p = CKPT / ("meta_" + seg + ".jsonl")
        if not p.exists():
            continue
        with open(p, encoding="utf-8") as f:
            for ln in f:
                try: baris.append(json.loads(ln))
                except Exception: pass
    meta = pd.DataFrame(baris).drop_duplicates("accession")
    kv = meta.apply(lambda r: parse_subname(r["subtype"], r["subname"]), axis=1)
    for kol in ["country", "host", "collection_date", "strain", "serotype",
                "segment", "isolation_source", "lat_lon"]:
        meta[kol] = kv.map(lambda d: d.get(kol))

    # --- benua: resolusi hanya atas nama negara UNIK ---
    negara_unik = meta["country"].dropna().unique().tolist()
    print(f"  negara unik: {len(negara_unik):,}")
    peta_benua_unik = {n: petakan_benua(n) for n in negara_unik}
    peta_iso_unik   = {n: resolve_negara(n)[0] for n in negara_unik}
    simpan_cache(CACHE_NEGARA, _pn)

    meta["benua"]  = meta["country"].map(peta_benua_unik).fillna("Tidak terpetakan")
    meta["negara"] = meta["country"].map(peta_iso_unik)

    # --- inang: resolusi taksonomi atas nama UNIK ---
    inang_unik = meta["host"].dropna().unique().tolist()
    print(f"  host unik: {len(inang_unik):,}")
    resolve_inang_batch(inang_unik)
    meta["inang"] = meta["host"].map(petakan_inang).fillna("lain")

    meta["tahun"]    = meta["collection_date"].map(ambil_tahun)
    meta["serotype"] = meta.apply(
        lambda r: ambil_serotype(r["title"], r["serotype"]), axis=1)

    meta.to_parquet(LAKE / "metadata_mentah.parquet", index=False)

    tak_terpeta = (meta.loc[meta["benua"] == "Tidak terpetakan", "country"]
                   .value_counts().head(25))
    host_lain = meta.loc[meta["inang"] == "lain", "host"].value_counts().head(25)

    audit("normalisasi",
          record=int(len(meta)),
          benua_pct=round(100 * float((meta["benua"] != "Tidak terpetakan").mean()), 2),
          inang_pct=round(100 * float((meta["inang"] != "lain").mean()), 2))

print(f"  record               : {len(meta):,}")
print(f"  benua terpetakan     : {100*(meta['benua']!='Tidak terpetakan').mean():.2f}%")
print(f"  inang terklasifikasi : {100*(meta['inang']!='lain').mean():.2f}%")
print(f"  PII                  : {pindai_pii(meta) or 'tidak ada'}")
print()
print(meta["benua"].value_counts().to_string())
print()
print(meta["inang"].value_counts().to_string())
print()
print("25 negara tak terpetakan teratas:")
print(tak_terpeta.to_string())
print()
print("25 host 'lain' teratas:")
print(host_lain.to_string())

[STORAGE] normalisasi metadata ... mulai
  negara unik: 4,647
  host unik: 6,508
  perlu resolusi taksonomi: 0 nama unik
[STORAGE] normalisasi metadata -> 0.30 menit, mem 21.35 GB, status OK
  record               : 610,847
  benua terpetakan     : 97.51%
  inang terklasifikasi : 90.42%
  PII                  : {'uid:telepon': 19996, 'title:telepon': 3988, 'subname:telepon': 15602, 'collection_date:telepon': 15513, 'strain:telepon': 3981}

benua
Amerika Utara       365103
Asia                139387
Eropa                49831
Oseania              15307
Tidak terpetakan     15213
Amerika Selatan      14189
Afrika               11817

inang
manusia    323166
unggas     168121
babi        61070
lain        58490

25 negara tak terpetakan teratas:
country
Turkey                                                            543
Turkey: Izmir                                                     113
Antarctica                                                         52
Turkey: Ankara               

In [12]:
with Tahap("bangun tabel terkurasi", "STORAGE"):
    bagian = []
    for seg in SEGMEN:
        p = SEQ / ("flua_" + seg + ".fasta.gz")
        if not p.exists():
            print(f"  FASTA {seg} tidak ada")
            continue
        seqs = baca_fasta_gz(p)
        d = pd.DataFrame({"accession": list(seqs.keys()),
                          "seq": list(seqs.values())})
        d["segmen"] = seg
        bagian.append(d)
        print(f"  {seg}: {len(d):,} sekuens dibaca")

    fasta = pd.concat(bagian, ignore_index=True)
    fasta["panjang"] = fasta["seq"].str.len()

    kolom_meta = ["accession","segmen","negara","benua","inang","tahun",
                  "serotype","strain","collection_date","isolation_source"]
    gab = fasta.merge(meta[kolom_meta], on=["accession","segmen"], how="inner")
    print(f"  setelah join: {len(gab):,}")

    gab["frac_ambigu"] = gab["seq"].map(
        lambda s: sum(1 for c in s if c in AMBIGU) / max(len(s), 1))

    RENTANG = {"HA": (1500, 1900), "NA": (1250, 1600)}
    simpan = []
    for seg, (lo, hi) in RENTANG.items():
        sub = gab[(gab["segmen"] == seg) &
                  gab["panjang"].between(lo, hi) &
                  (gab["frac_ambigu"] <= 0.05)]
        simpan.append(sub)
        print(f"  {seg} lolos filter: {len(sub):,}")

    bersih = pd.concat(simpan, ignore_index=True).drop_duplicates("accession")
    bersih["tahun"] = bersih["tahun"].fillna(0).astype(int)
    audit("kurasi", masuk=int(len(gab)), keluar=int(len(bersih)))

[STORAGE] bangun tabel terkurasi ... mulai
  HA: 362,842 sekuens dibaca
  NA: 287,378 sekuens dibaca
  setelah join: 610,823
  HA lolos filter: 237,559
  NA lolos filter: 203,304
[STORAGE] bangun tabel terkurasi -> 0.45 menit, mem 22.43 GB, status OK


In [13]:
ATURAN = [
 ("baris > 100.000",        lambda d: len(d) > 100_000, True),
 ("kolom inang ada",        lambda d: "inang" in d.columns, True),
 ("kelas inang >= 3",       lambda d: d["inang"].nunique() >= 3, True),
 ("duplikat accession = 0", lambda d: int(d["accession"].duplicated().sum()) == 0, True),
 ("ambigu <= 5%",           lambda d: float(d["frac_ambigu"].max()) <= 0.05, True),
 ("dua segmen",             lambda d: d["segmen"].nunique() == 2, True),
 ("tahun valid > 90%",      lambda d: float((d["tahun"] > 1900).mean()) > 0.90, False),
 ("benua terpetakan > 90%", lambda d: float((d["benua"] != "Tidak terpetakan").mean()) > 0.90, False),
]

with Tahap("gerbang kualitas data", "SECURITY"):
    lolos, lap_kualitas = gerbang_kualitas(bersih, ATURAN, "stage")
    lap_kualitas.to_csv(OUT / "laporan_kualitas.csv", index=False)

[SECURITY] gerbang kualitas data ... mulai
tahap                 aturan hasil  lolos  wajib
stage        baris > 100.000  True   True   True
stage        kolom inang ada  True   True   True
stage       kelas inang >= 3  True   True   True
stage duplikat accession = 0  True   True   True
stage           ambigu <= 5%  True   True   True
stage             dua segmen  True   True   True
stage      tahun valid > 90%  True   True  False
stage benua terpetakan > 90%  True   True  False
[SECURITY] gerbang kualitas data -> 0.00 menit, mem 22.45 GB, status OK


In [14]:
with Tahap("tulis Parquet berpartisi", "STORAGE"):
    tujuan = STAGE / "flua_seq_bersih.parquet"
    bersih.to_parquet(tujuan, index=False, compression="snappy")

    part = STAGE / "flua_partisi"
    bersih.to_parquet(part, index=False, compression="snappy",
                      partition_cols=["segmen", "benua"])

    n_file  = sum(1 for _ in part.rglob("*.parquet"))
    mb_flat = tujuan.stat().st_size / 1e6
    mb_part = sum(f.stat().st_size for f in part.rglob("*.parquet")) / 1e6
    print(f"  flat    : {mb_flat:,.1f} MB")
    print(f"  partisi : {mb_part:,.1f} MB dalam {n_file} file")
    audit("tulis_storage", baris=int(len(bersih)), file_partisi=int(n_file))

print()
print(bersih.groupby(["segmen","inang"]).size().unstack(fill_value=0).to_string())
print()
print(bersih.groupby("benua").size().sort_values(ascending=False).to_string())

[STORAGE] tulis Parquet berpartisi ... mulai
  flat    : 157.4 MB
  partisi : 1,854.7 MB dalam 168 file
[STORAGE] tulis Parquet berpartisi -> 0.03 menit, mem 22.92 GB, status OK

inang    babi   lain  manusia  unggas
segmen                               
HA      27110  18993   129442   62014
NA      24409  16160   118900   43835

benua
Amerika Utara       294539
Asia                 77195
Eropa                32700
Oseania              13598
Afrika                8392
Amerika Selatan       7819
Tidak terpetakan      6620


## LAPISAN 4 — HADOOP PLATFORM MANAGEMENT LAYER | Apache Spark sebagai pengganti MapReduce

In [15]:
# ============================================================
# SETUP WINUTILS — jalankan SEBELUM sel LAPISAN 4
# Wajib kernel baru: Kernel -> Restart Kernel
# ============================================================
import os, sys, urllib.request, shutil, subprocess
from pathlib import Path
import pyspark

HADOOP_DIR = Path(r"D:\hadoop")
BIN = HADOOP_DIR / "bin"
BIN.mkdir(parents=True, exist_ok=True)

# Spark 3.5 dibundel dengan Hadoop 3.3.4; biner 3.3.x saling kompatibel
VERSI = ["hadoop-3.3.6", "hadoop-3.3.5", "hadoop-3.3.4", "hadoop-3.3.1"]
BERKAS = ["winutils.exe", "hadoop.dll"]
BASE = "https://raw.githubusercontent.com/cdarlint/winutils/master"

def unduh(nama):
    tujuan = BIN / nama
    if tujuan.exists() and tujuan.stat().st_size > 1000:
        print(f"  {nama:<14} sudah ada ({tujuan.stat().st_size/1024:.0f} KB)")
        return True
    for v in VERSI:
        url = f"{BASE}/{v}/bin/{nama}"
        try:
            with urllib.request.urlopen(url, timeout=120) as r, \
                 open(tujuan, "wb") as f:
                shutil.copyfileobj(r, f)
            if tujuan.stat().st_size > 1000:
                print(f"  {nama:<14} OK dari {v} ({tujuan.stat().st_size/1024:.0f} KB)")
                return True
        except Exception as ex:
            print(f"    {v} gagal: {type(ex).__name__}")
    return False

print("Mengunduh winutils ...")
sukses = all(unduh(b) for b in BERKAS)

if sukses:
    os.environ["HADOOP_HOME"]   = str(HADOOP_DIR)
    os.environ["hadoop.home.dir"] = str(HADOOP_DIR)
    os.environ["PATH"] = str(BIN) + os.pathsep + os.environ.get("PATH", "")
    WINUTILS_OK = True

    r = subprocess.run([str(BIN / "winutils.exe")],
                       capture_output=True, text=True)
    print(f"\n  uji winutils: {'jalan' if 'Usage' in (r.stdout + r.stderr) else 'gagal eksekusi'}")
else:
    WINUTILS_OK = False
    print("\n  Unduhan gagal. Event log akan dinonaktifkan otomatis.")
    print("  Alternatif manual: unduh winutils.exe dan hadoop.dll dari")
    print("  https://github.com/cdarlint/winutils (folder hadoop-3.3.5/bin)")
    print(f"  lalu taruh di {BIN}")

# JAVA_HOME
if not os.environ.get("JAVA_HOME"):
    try:
        w = subprocess.run(["where", "java"], capture_output=True, text=True)
        jp = Path(w.stdout.strip().splitlines()[0]).parent.parent
        if (jp / "bin" / "java.exe").exists():
            os.environ["JAVA_HOME"] = str(jp)
            print(f"\n  JAVA_HOME diset: {jp}")
    except Exception:
        print("\n  JAVA_HOME tidak bisa dideteksi otomatis (biasanya tidak masalah)")

os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"]        = "127.0.0.1"

print(f"\nHADOOP_HOME : {os.environ.get('HADOOP_HOME', 'TIDAK DISET')}")
print(f"WINUTILS_OK : {WINUTILS_OK}")

Mengunduh winutils ...
  winutils.exe   sudah ada (116 KB)
  hadoop.dll     sudah ada (77 KB)

  uji winutils: jalan

HADOOP_HOME : D:\hadoop
WINUTILS_OK : True


In [16]:
# ============================================================
# PRA-FLIGHT: bersihkan JVM zombie + cek RAM
# ============================================================
import subprocess, psutil, time

sisa = []
for pr in psutil.process_iter(["pid", "name", "memory_info", "create_time"]):
    try:
        if "java" in (pr.info["name"] or "").lower():
            sisa.append((pr.info["pid"],
                         pr.info["memory_info"].rss / 1e6,
                         (time.time() - pr.info["create_time"]) / 60))
    except Exception:
        pass

if sisa:
    print("Proses Java yang masih hidup:")
    for pid, mb, umur in sisa:
        print(f"  PID {pid:<8} {mb:>10,.0f} MB   umur {umur:>6.1f} menit")
    print("\nMematikan ...")
    subprocess.run(["taskkill", "/F", "/IM", "java.exe"], capture_output=True)
    time.sleep(5)
else:
    print("Tidak ada proses Java tersisa.")

vm = psutil.virtual_memory()
print(f"\nRAM total   : {vm.total/1e9:.1f} GB")
print(f"RAM bebas   : {vm.available/1e9:.1f} GB")
heap_rencana = int(CFG["driver_memory"].rstrip("g"))
print(f"Heap rencana: {heap_rencana} GB")
margin = vm.available/1e9 - heap_rencana
print(f"Margin      : {margin:.1f} GB "
      + ("(aman)" if margin >= 12 else "!! turunkan driver_memory"))
if sisa:
    print("\n-> Java baru dimatikan. RESTART KERNEL sebelum lanjut ke LAPISAN 4.")


Tidak ada proses Java tersisa.

RAM total   : 68.4 GB
RAM bebas   : 45.5 GB
Heap rencana: 22 GB
Margin      : 23.5 GB (aman)


In [17]:
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"]        = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"]   = (
    "--driver-memory " + CFG["driver_memory"] +
    " --conf spark.driver.maxResultSize=2g pyspark-shell")

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import (IntegerType, StringType, DoubleType, LongType,
                               StructType, StructField)

EVENTLOG_AKTIF = bool(globals().get("WINUTILS_OK", False))


def bangun_spark(master=None, app="influenza-bda-revisi"):
    b = (SparkSession.builder
        .appName(app)
        .master(master or CFG["spark_master"])
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.local.dir", str(TMP))
        .config("spark.sql.shuffle.partitions", str(CFG["shuffle_part"]))
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.network.timeout", "1200s")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.kryoserializer.buffer.max", "512m")
        # --- hemat: tumpahkan ke disk lebih awal ---
        .config("spark.memory.fraction", "0.5")
        .config("spark.memory.storageFraction", "0.2")
        .config("spark.sql.files.maxPartitionBytes", "16m")
        .config("spark.sql.autoBroadcastJoinThreshold", "-1")
        .config("spark.shuffle.compress", "true")
        .config("spark.shuffle.spill.compress", "true")
        .config("spark.rdd.compress", "true")
        # --- worker Python dibatasi ketat ---
        .config("spark.python.worker.memory", "512m")
        .config("spark.python.worker.reuse", "true")
        .config("spark.executor.cores", "4")
        .config("spark.driver.extraJavaOptions",
                "-Djava.net.preferIPv4Stack=true "
                "-Dio.netty.tryReflectionSetAccessible=true "
                "-XX:+UseG1GC -XX:MaxGCPauseMillis=200 "
                "-XX:+ExitOnOutOfMemoryError")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.sql.execution.arrow.maxRecordsPerBatch", "2000"))
    if EVENTLOG_AKTIF:
        b = (b.config("spark.eventLog.enabled", "true")
              .config("spark.eventLog.dir", LOGDIR.as_uri()))
    return b.getOrCreate()


# ============================================================
# HELPER MEMORI — dipakai di setiap tahap berat
# ============================================================
DF_TERLACAK = []


def lacak(df, nama):
    """Cache sebuah DataFrame dan catat agar bisa dibebaskan belakangan."""
    df.cache(); df.count()
    DF_TERLACAK.append((nama, df))
    return df


def bersihkan(kecuali=(), tegas=False, catatan=""):
    """Bebaskan cache Spark + garbage collect Python. Panggil di akhir tiap tahap."""
    dibuang = []
    for item in list(DF_TERLACAK):
        nama, df = item
        if nama in kecuali:
            continue
        try:
            df.unpersist(blocking=True)
            dibuang.append(nama)
        except Exception:
            pass
        try:
            DF_TERLACAK.remove(item)
        except ValueError:
            pass
    if tegas:
        try:
            spark.catalog.clearCache()
        except Exception:
            pass
    gc.collect()
    vm = psutil.virtual_memory()
    sisa = [n for n, _ in DF_TERLACAK]
    print(f"  [bersih{(' ' + catatan) if catatan else ''}] "
          f"dibebaskan {len(dibuang)}"
          f"{' ' + str(dibuang) if dibuang else ''}"
          f"{' | tetap: ' + str(sisa) if sisa else ''}"
          f" | RAM bebas {vm.available/1e9:.1f} GB")


def cp(df, nama, paksa=False):
    """Checkpoint ke Parquet lalu baca ulang. Memotong lineage sepenuhnya.
    Wajib dipakai di dalam loop iteratif (PageRank, LPA, k-core, BFS)."""
    path = STAGE / "cp" / nama
    path.parent.mkdir(parents=True, exist_ok=True)
    if paksa or not (path.exists() and any(path.rglob("*.parquet"))):
        df.write.mode("overwrite").parquet(str(path))
    return spark.read.parquet(str(path))


def mode_graf():
    spark.conf.set("spark.sql.shuffle.partitions", str(CFG["part_graf"]))
    print(f"  mode GRAF: shuffle partitions -> {CFG['part_graf']}")


def mode_data():
    spark.conf.set("spark.sql.shuffle.partitions", str(CFG["shuffle_part"]))
    print(f"  mode DATA: shuffle partitions -> {CFG['shuffle_part']}")


spark = bangun_spark()
spark.sparkContext.setLogLevel("WARN")

heap = spark.sparkContext._jvm.java.lang.Runtime.getRuntime().maxMemory()
vm = psutil.virtual_memory()
SPARK_VERSI = spark.version
print("Spark     :", SPARK_VERSI)
print(f"Heap JVM  : {heap/1e9:.1f} GB")
print("Master    :", spark.sparkContext.master)
print("Partisi   :", CFG["shuffle_part"], "(graf:", CFG["part_graf"], ")")
print("Spark UI  :", spark.sparkContext.uiWebUrl)
print("Event log :", LOGDIR if EVENTLOG_AKTIF else "NONAKTIF (winutils tidak ada)")
print(f"RAM bebas : {vm.available/1e9:.1f} GB | margin {(vm.available-heap)/1e9:.1f} GB")
print("Helper    : lacak(df,nama)  bersihkan()  cp(df,nama)  mode_graf()  mode_data()")
audit("spark_start", versi=SPARK_VERSI, heap_gb=round(heap/1e9, 1),
      profil="hemat")

Spark      : 3.5.9
Heap JVM   : 23.6 GB
Master     : local[4]
Partisi    : 600
Spark UI   : http://127.0.0.1:4040
Event log  : D:\BDA\spark_events
RAM bebas  : 45.1 GB dari 68.4 GB
  Margin di luar heap: 21.4 GB (aman)


{'ts': '2026-08-27T05:26:18.779017+00:00',
 'run_id': 'run_20260827T052524Z',
 'peristiwa': 'spark_start',
 'aktor': 'Willy Boen',
 'versi': '3.5.9',
 'master': 'local[4]',
 'heap_gb': 23.6,
 'eventlog': True,
 'ram_bebas_gb': 45.1}

## LAPISAN 5 — BATCH PROCESSING & FEATURE ENGINEERING | Ekstraksi k-mer via Spark SQL native (tanpa UDF Python)

In [18]:
with Tahap("muat stage ke Spark", "PLATFORM"):
    stage_all = spark.read.parquet(str(STAGE / "flua_seq_bersih.parquet"))

    ha = (stage_all.filter((F.col("segmen") == "HA") &
                           F.col("inang").isNotNull() &
                           (F.col("inang") != "lain")))

    n_ha_penuh = ha.count()
    if CFG["subsampel"] and CFG["subsampel"] < n_ha_penuh:
        # sampel berstrata per inang agar proporsi kelas terjaga
        frak = CFG["subsampel"] / n_ha_penuh
        pecahan = {r["inang"]: min(frak * 1.0, 1.0)
                   for r in ha.select("inang").distinct().collect()}
        ha = ha.sampleBy("inang", fractions=pecahan, seed=7)
        print(f"  subsampel berstrata: {CFG['subsampel']:,} dari {n_ha_penuh:,}")

    ha = ha.withColumn("idx", F.monotonically_increasing_id()) \
           .repartition(CFG["shuffle_part"]).cache()
    N = ha.count()

print(f"  HA populasi penuh : {n_ha_penuh:,}")
print(f"  HA dipakai        : {N:,}")
print(f"  partisi           : {ha.rdd.getNumPartitions()}")
ha.groupBy("inang").count().orderBy(F.desc("count")).show()

[PLATFORM] muat stage ke Spark ... mulai
  subsampel berstrata: 40,000 dari 218,566
[PLATFORM] muat stage ke Spark -> 0.10 menit, mem 25.15 GB, status OK
  HA populasi penuh : 218,566
  HA dipakai        : 39,856
  partisi           : 600
+-------+-----+
|  inang|count|
+-------+-----+
|manusia|23506|
| unggas|11407|
|   babi| 4943|
+-------+-----+



In [19]:
KMER_PATH = STAGE / f"kmer_ha_k{K}_{'penuh' if not CFG['subsampel'] else CFG['subsampel']}.parquet"

with Tahap("ekstraksi k-mer batch", "BATCH PROCESSING"):
    sudah_ada = KMER_PATH.exists() and any(KMER_PATH.rglob("*.parquet"))

    if sudah_ada:
        print(f"  checkpoint ditemukan: {KMER_PATH.name}")
        print("  -> lewati komputasi, baca ulang dari disk")
    else:
        print(f"  menulis k-mer ke {KMER_PATH.name} ...")
        (ha.withColumn("kmers",
               F.expr(f"filter(transform(sequence(0, length(seq) - {K}), "
                      f"i -> substring(seq, i+1, {K})), x -> x rlike '^[ACGT]+$')"))
           .withColumn("kmers_unik", F.array_distinct(F.col("kmers")))
           .select("idx", "accession", "inang", "benua", "negara", "tahun",
                   "serotype", "panjang", "kmers", "kmers_unik")
           .filter(F.size("kmers_unik") > 50)
           .write.mode("overwrite").parquet(str(KMER_PATH)))
        print("  selesai ditulis")

    # dibaca ulang dari disk: tidak menahan seluruh array di heap
    kmer_df = spark.read.parquet(str(KMER_PATH))
    n_kmer = kmer_df.count()

    stat = kmer_df.select(
        F.sum(F.size("kmers")).alias("tot"),
        F.avg(F.size("kmers")).alias("rata"),
        F.avg(F.size("kmers_unik")).alias("rata_unik")).first()

    mb_kmer = sum(f.stat().st_size for f in KMER_PATH.rglob("*.parquet")) / 1e6

VOLUME = {
    "sekuens_populasi": int(n_ha_penuh),
    "sekuens_dianalisis": int(n_kmer),
    "k": K,
    "token_kmer_total": int(stat["tot"]),
    "token_kmer_per_sekuens": round(float(stat["rata"]), 1),
    "kmer_unik_per_sekuens": round(float(stat["rata_unik"]), 1),
    "ruang_fitur": VOCAB,
    "sel_matriks_teoretis": int(n_kmer) * VOCAB,
    "partisi": kmer_df.rdd.getNumPartitions(),
    "ukuran_kmer_parquet_mb": round(mb_kmer, 1),
}
print()
for k_, v_ in VOLUME.items():
    print(f"  {k_:<24}: {v_:,}" if isinstance(v_, int) else f"  {k_:<24}: {v_}")
json.dump(VOLUME, open(OUT / "volume_data.json", "w"), indent=2)
audit("volume_batch", **VOLUME)

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("kmer_df",), catatan="k-mer")


[BATCH PROCESSING] ekstraksi k-mer batch ... mulai
  checkpoint ditemukan: kmer_ha_k10_40000.parquet
  -> lewati komputasi, baca ulang dari disk
[BATCH PROCESSING] ekstraksi k-mer batch -> 0.03 menit, mem 25.43 GB, status OK

  sekuens_populasi        : 218,566
  sekuens_dianalisis      : 39,856
  k                       : 10
  token_kmer_total        : 67,822,368
  token_kmer_per_sekuens  : 1701.7
  kmer_unik_per_sekuens   : 1698.5
  ruang_fitur             : 262,144
  sel_matriks_teoretis    : 10,448,011,264
  partisi                 : 86
  ukuran_kmer_parquet_mb  : 468.7


{'ts': '2026-08-27T05:26:28.464988+00:00',
 'run_id': 'run_20260827T052524Z',
 'peristiwa': 'volume_batch',
 'aktor': 'Willy Boen',
 'sekuens_populasi': 218566,
 'sekuens_dianalisis': 39856,
 'k': 10,
 'token_kmer_total': 67822368,
 'token_kmer_per_sekuens': 1701.7,
 'kmer_unik_per_sekuens': 1698.5,
 'ruang_fitur': 262144,
 'sel_matriks_teoretis': 10448011264,
 'partisi': 86,
 'ukuran_kmer_parquet_mb': 468.7}

## LAPISAN 6 — ANALYTICS ENGINE A: TEXT ANALYTICS | CountVectorizer + IDF atas token k-mer

In [20]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (CountVectorizer, IDF, StringIndexer,
                                MinHashLSH, UnivariateFeatureSelector)

with Tahap("vektorisasi TF-IDF", "ANALYTICS/TEXT"):
    indexer = StringIndexer(inputCol="inang", outputCol="label")
    cvz     = CountVectorizer(inputCol="kmers", outputCol="tf",
                              vocabSize=VOCAB, minDF=5.0)
    idf     = IDF(inputCol="tf", outputCol="fitur")

    fitur_model = Pipeline(stages=[indexer, cvz, idf]).fit(kmer_df)
    LABEL_NAMA  = list(fitur_model.stages[0].labels)
    KOSAKATA    = list(fitur_model.stages[1].vocabulary)
    N_FITUR     = len(KOSAKATA)
    N_KELAS     = len(LABEL_NAMA)

    ds_fitur = fitur_model.transform(kmer_df).select(
        "idx","accession","fitur","label","inang","benua","negara",
        "tahun","serotype").cache()
    ds_fitur.count()

print(f"  kosakata k-mer : {N_FITUR:,}")
print(f"  kelas          : {LABEL_NAMA}")

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("kmer_df", "ds_fitur"), catatan="TF-IDF")


[ANALYTICS/TEXT] vektorisasi TF-IDF ... mulai
[ANALYTICS/TEXT] vektorisasi TF-IDF -> 0.58 menit, mem 30.01 GB, status OK
  kosakata k-mer : 210,221
  kelas          : ['manusia', 'unggas', 'babi']


## LAPISAN 7 — ANALYTICS ENGINE B: SIMILARITY / GRAPH CONSTRUCTION | MinHash LSH

In [21]:
from pyspark.ml.feature import CountVectorizer, MinHashLSH

N_HASH = CFG["n_hash_lsh"]
AMBANG = CFG["ambang_jaccard"]
TOP_K  = CFG["lsh_top_k"]
N_BLOK = CFG["lsh_n_blok"]

VEK_PATH = STAGE / "vek_biner.parquet"
SISI_DIR = STAGE / f"sisi_blok_j{int(AMBANG*100)}_k{TOP_K}_h{N_HASH}"
SISI_PATH = STAGE / f"graf_kemiripan_j{int(AMBANG*100)}_k{TOP_K}.parquet"
SISI_DIR.mkdir(parents=True, exist_ok=True)

with Tahap("vektorisasi biner untuk LSH", "ANALYTICS/GRAPH"):
    if VEK_PATH.exists() and any(VEK_PATH.rglob("*.parquet")):
        print("  checkpoint vektor ditemukan, baca ulang")
    else:
        cvz_bin = CountVectorizer(inputCol="kmers_unik", outputCol="vek",
                                  vocabSize=VOCAB, minDF=2.0, binary=True)
        cvz_bin_model = cvz_bin.fit(kmer_df)
        print(f"  kosakata biner: {len(cvz_bin_model.vocabulary):,}")

        @F.udf(returnType=IntegerType())
        def nnz(v):
            try:
                return int(v.numNonzeros())
            except Exception:
                return 0

        (cvz_bin_model.transform(kmer_df)
         .withColumn("nnz", nnz(F.col("vek")))
         .filter(F.col("nnz") > 0)
         .select("idx", "accession", "inang", "benua", "tahun", "serotype", "vek")
         .write.mode("overwrite").parquet(str(VEK_PATH)))
        print("  vektor ditulis ke disk")

with Tahap("MinHash LSH berblok (resumable)", "ANALYTICS/GRAPH"):
    vek_df = (spark.read.parquet(str(VEK_PATH))
              .withColumn("blok", F.abs(F.crc32(F.col("accession"))) % N_BLOK)
              .cache())
    n_vek = vek_df.count()
    print(f"  simpul : {n_vek:,} dibagi {N_BLOK} blok (~{n_vek//N_BLOK:,}/blok)")
    print(f"  param  : numHashTables={N_HASH}, Jaccard>={AMBANG}, topK={TOP_K}")

    mh_model = MinHashLSH(inputCol="vek", outputCol="hash",
                          numHashTables=N_HASH, seed=SEED).fit(vek_df)
    hash_penuh = mh_model.transform(vek_df).cache()
    hash_penuh.count()

    w_topk = Window.partitionBy("src").orderBy(F.desc("kemiripan"))
    t_mulai, blok_gagal = time.time(), []

    for b in range(N_BLOK):
        tujuan = SISI_DIR / f"blok={b}"
        if tujuan.exists() and any(tujuan.rglob("*.parquet")):
            print(f"  blok {b:>3}/{N_BLOK}  sudah ada, lewati")
            continue
        t0 = time.time()
        try:
            potongan = hash_penuh.filter(F.col("blok") == b)
            (mh_model.approxSimilarityJoin(
                    potongan, hash_penuh,
                    threshold=1.0 - AMBANG, distCol="jarak")
             .select(F.col("datasetA.idx").alias("src"),
                     F.col("datasetB.idx").alias("dst"),
                     F.col("datasetA.accession").alias("src_acc"),
                     F.col("datasetB.accession").alias("dst_acc"),
                     F.col("datasetA.inang").alias("src_inang"),
                     F.col("datasetB.inang").alias("dst_inang"),
                     F.col("datasetA.benua").alias("src_benua"),
                     F.col("datasetB.benua").alias("dst_benua"),
                     F.col("datasetA.tahun").alias("src_tahun"),
                     F.col("datasetB.tahun").alias("dst_tahun"),
                     (1.0 - F.col("jarak")).alias("kemiripan"))
             .filter(F.col("src") != F.col("dst"))
             .withColumn("rk", F.row_number().over(w_topk))
             .filter(F.col("rk") <= TOP_K).drop("rk")
             .write.mode("overwrite").parquet(str(tujuan)))
            n_b = spark.read.parquet(str(tujuan)).count()
            sisa_menit = ((time.time()-t_mulai)/max(b+1,1))*(N_BLOK-b-1)/60
            print(f"  blok {b:>3}/{N_BLOK}  {n_b:>9,} sisi  "
                  f"{(time.time()-t0)/60:>5.2f} mnt  "
                  f"(sisa ~{sisa_menit:.0f} mnt)")
            audit("lsh_blok", blok=b, sisi=int(n_b))
        except Exception as ex:
            blok_gagal.append(b)
            print(f"  blok {b:>3}/{N_BLOK}  GAGAL {type(ex).__name__}: {str(ex)[:90]}")
            audit("lsh_blok_gagal", blok=b, error=type(ex).__name__)

    if blok_gagal:
        print(f"\n  !! {len(blok_gagal)} blok gagal: {blok_gagal}")
        print("     Jalankan ulang sel ini; blok yang sudah jadi akan dilewati.")

with Tahap("gabung & kanonikalisasi sisi", "ANALYTICS/GRAPH"):
    sisi_mentah = spark.read.parquet(str(SISI_DIR))
    (sisi_mentah
     .withColumn("lo", F.least("src", "dst"))
     .withColumn("hi", F.greatest("src", "dst"))
     .withColumn("tetap", F.col("src") == F.col("lo"))
     .select(
         F.col("lo").alias("src"), F.col("hi").alias("dst"),
         F.when(F.col("tetap"), F.col("src_acc")).otherwise(F.col("dst_acc")).alias("src_acc"),
         F.when(F.col("tetap"), F.col("dst_acc")).otherwise(F.col("src_acc")).alias("dst_acc"),
         F.when(F.col("tetap"), F.col("src_inang")).otherwise(F.col("dst_inang")).alias("src_inang"),
         F.when(F.col("tetap"), F.col("dst_inang")).otherwise(F.col("src_inang")).alias("dst_inang"),
         F.when(F.col("tetap"), F.col("src_benua")).otherwise(F.col("dst_benua")).alias("src_benua"),
         F.when(F.col("tetap"), F.col("dst_benua")).otherwise(F.col("src_benua")).alias("dst_benua"),
         F.when(F.col("tetap"), F.col("src_tahun")).otherwise(F.col("dst_tahun")).alias("src_tahun"),
         F.when(F.col("tetap"), F.col("dst_tahun")).otherwise(F.col("src_tahun")).alias("dst_tahun"),
         F.col("kemiripan"))
     .dropDuplicates(["src", "dst"])
     .write.mode("overwrite").parquet(str(SISI_PATH)))

    sisi = spark.read.parquet(str(SISI_PATH)).cache()
    n_sisi = sisi.count()

print()
print(f"  simpul        : {n_vek:,}")
print(f"  sisi          : {n_sisi:,}")
print(f"  derajat rata2 : {2*n_sisi/max(n_vek,1):.1f}")
print(f"  densitas      : {n_sisi/max(n_vek*(n_vek-1)/2,1):.3e}")
audit("graf_dibangun", simpul=int(n_vek), sisi=int(n_sisi),
      n_hash=N_HASH, ambang=AMBANG, top_k=TOP_K, n_blok=N_BLOK)

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("kmer_df", "ds_fitur", "sisi"), catatan="graf")


[ANALYTICS/GRAPH] vektorisasi biner untuk LSH ... mulai
  checkpoint vektor ditemukan, baca ulang
[ANALYTICS/GRAPH] vektorisasi biner untuk LSH -> 0.00 menit, mem 30.04 GB, status OK
[ANALYTICS/GRAPH] MinHash LSH berblok (resumable) ... mulai
  simpul : 39,856 dibagi 40 blok (~996/blok)
  param  : numHashTables=5, Jaccard>=0.95, topK=30
  blok   0/40  sudah ada, lewati
  blok   1/40  sudah ada, lewati
  blok   2/40  sudah ada, lewati
  blok   3/40  sudah ada, lewati
  blok   4/40  sudah ada, lewati
  blok   5/40  sudah ada, lewati
  blok   6/40  sudah ada, lewati
  blok   7/40  sudah ada, lewati
  blok   8/40  sudah ada, lewati
  blok   9/40  sudah ada, lewati
  blok  10/40  sudah ada, lewati
  blok  11/40  sudah ada, lewati
  blok  12/40  sudah ada, lewati
  blok  13/40  sudah ada, lewati
  blok  14/40  sudah ada, lewati
  blok  15/40  sudah ada, lewati
  blok  16/40  sudah ada, lewati
  blok  17/40  sudah ada, lewati
  blok  18/40  sudah ada, lewati
  blok  19/40  sudah ada, lewati
 

{'ts': '2026-08-27T05:27:11.004061+00:00',
 'run_id': 'run_20260827T052524Z',
 'peristiwa': 'graf_dibangun',
 'aktor': 'Willy Boen',
 'simpul': 39856,
 'sisi': 381004,
 'n_hash': 5,
 'ambang': 0.95,
 'top_k': 30,
 'n_blok': 40}

In [22]:
LP_DIR   = STAGE / "lp_iter"
SISI2_PATH = STAGE / "sisi_dua_arah.parquet"
LP_DIR.mkdir(parents=True, exist_ok=True)

# graf kecil: 600 partisi jauh berlebihan dan justru memicu OOM
PART_GRAF = 64
part_asli = spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.shuffle.partitions", str(PART_GRAF))

with Tahap("siapkan sisi dua arah", "ANALYTICS/GRAPH"):
    if SISI2_PATH.exists() and any(SISI2_PATH.rglob("*.parquet")):
        print("  checkpoint sisi dua arah ditemukan")
    else:
        (sisi.select("src", "dst")
         .union(sisi.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
         .distinct()
         .repartition(PART_GRAF, "src")
         .write.mode("overwrite").parquet(str(SISI2_PATH)))
    sisi2 = spark.read.parquet(str(SISI2_PATH)).cache()
    n_arah = sisi2.count()
    print(f"  sisi dua arah: {n_arah:,}")

with Tahap("connected components (label propagation)", "ANALYTICS/GRAPH"):
    MAKS_ITER = 25
    TOLERANSI = 0.0005          # berhenti bila < 0,05% simpul berubah

    # lanjutkan dari iterasi terakhir yang tersimpan
    ada = sorted(int(d.name.split("=")[1]) for d in LP_DIR.glob("iter=*")
                 if any(d.rglob("*.parquet")))
    mulai = (max(ada) + 1) if ada else 0

    if mulai == 0:
        (vek_df.select(F.col("idx").alias("id"))
         .withColumn("komp", F.col("id"))
         .withColumn("berubah", F.lit(0))
         .repartition(PART_GRAF, "id")
         .write.mode("overwrite").parquet(str(LP_DIR / "iter=0")))
        mulai = 1
        print("  inisialisasi label selesai")
    else:
        print(f"  lanjut dari iterasi {mulai}")

    label_kmp = spark.read.parquet(str(LP_DIR / f"iter={mulai-1}")) \
                     .select("id", "komp")
    n_simpul = label_kmp.count()

    for it in range(mulai, MAKS_ITER + 1):
        tujuan = LP_DIR / f"iter={it}"
        if tujuan.exists() and any(tujuan.rglob("*.parquet")):
            label_kmp = spark.read.parquet(str(tujuan)).select("id", "komp")
            print(f"  iterasi {it:>2}: sudah ada, lewati")
            continue

        t0 = time.time()
        pesan = (sisi2.join(label_kmp, sisi2.src == label_kmp.id)
                      .groupBy("dst").agg(F.min("komp").alias("komp_min")))

        (label_kmp.join(pesan, label_kmp.id == pesan.dst, "left")
         .select("id",
                 F.col("komp").alias("komp_lama"),
                 F.least(F.col("komp"),
                         F.coalesce(F.col("komp_min"), F.col("komp"))).alias("komp"))
         .withColumn("berubah",
                     (F.col("komp") != F.col("komp_lama")).cast("int"))
         .drop("komp_lama")
         .repartition(PART_GRAF, "id")
         .write.mode("overwrite").parquet(str(tujuan)))

        # baca ulang dari disk: lineage terpotong total
        hasil = spark.read.parquet(str(tujuan))
        berubah = hasil.agg(F.sum("berubah").alias("n")).first()["n"] or 0
        label_kmp = hasil.select("id", "komp")

        rasio = berubah / max(n_simpul, 1)
        print(f"  iterasi {it:>2}: {berubah:>7,} berubah "
              f"({100*rasio:.3f}%)  {(time.time()-t0)/60:.2f} menit")
        audit("lp_iterasi", iterasi=it, berubah=int(berubah))

        if rasio < TOLERANSI:
            print(f"  konvergen pada iterasi {it}")
            break

    n_klaster = label_kmp.select("komp").distinct().count()

spark.conf.set("spark.sql.shuffle.partitions", part_asli)

RASIO_PSEUDO = n_vek / max(n_klaster, 1)
print()
print(f"  simpul               : {n_vek:,}")
print(f"  klaster identitas    : {n_klaster:,}")
print(f"  rasio pseudoreplikasi: {RASIO_PSEUDO:.2f}x")
audit("klaster", n_klaster=int(n_klaster), rasio=round(RASIO_PSEUDO, 2))

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("kmer_df", "ds_fitur", "sisi", "sisi2"), catatan="komponen")


[ANALYTICS/GRAPH] siapkan sisi dua arah ... mulai
  checkpoint sisi dua arah ditemukan
  sisi dua arah: 762,008
[ANALYTICS/GRAPH] siapkan sisi dua arah -> 0.01 menit, mem 31.55 GB, status OK
[ANALYTICS/GRAPH] connected components (label propagation) ... mulai
  lanjut dari iterasi 17
  iterasi 17:       0 berubah (0.000%)  0.02 menit
  konvergen pada iterasi 17
[ANALYTICS/GRAPH] connected components (label propagation) -> 0.03 menit, mem 31.53 GB, status OK

  simpul               : 39,856
  klaster identitas    : 12,554
  rasio pseudoreplikasi: 3.17x


{'ts': '2026-08-27T05:27:13.307671+00:00',
 'run_id': 'run_20260827T052524Z',
 'peristiwa': 'klaster',
 'aktor': 'Willy Boen',
 'n_klaster': 12554,
 'rasio': 3.17}

## LAPISAN 8 — PARTISI EVALUASI | Empat skema fold: acak, blok spasial, blok temporal, blok klaster

In [23]:
with Tahap("bangun 4 kolom fold", "BATCH PROCESSING"):
    kerja = (ds_fitur.join(label_kmp.withColumnRenamed("id", "idx"), "idx").cache())
    kerja.count()

    kerja = kerja.withColumn("fold_acak",
                             (F.rand(seed=SEED) * N_FOLD).cast(IntegerType()))

    cnt_b = {r["benua"]: r["count"] for r in kerja.groupBy("benua").count().collect()}
    beban, peta_b = [0]*N_FOLD, {}
    for b in sorted(cnt_b, key=cnt_b.get, reverse=True):
        f_ = beban.index(min(beban)); peta_b[b] = f_; beban[f_] += cnt_b[b]
    mb = F.create_map([F.lit(x) for kv in peta_b.items() for x in kv])
    kerja = kerja.withColumn("fold_spasial", F.coalesce(mb[F.col("benua")], F.lit(0)))

    q = kerja.approxQuantile("tahun", [i/N_FOLD for i in range(1, N_FOLD)], 0.01)
    e = F.lit(N_FOLD-1)
    for i in range(N_FOLD-2, -1, -1):
        e = F.when(F.col("tahun") <= q[i], F.lit(i)).otherwise(e)
    kerja = kerja.withColumn("fold_temporal", e.cast(IntegerType()))

    kl = kerja.groupBy("komp").count().orderBy(F.desc("count")).toPandas()
    beban_k, peta_k = [0]*N_FOLD, {}
    for _, r in kl.iterrows():
        f_ = int(np.argmin(beban_k))
        peta_k[int(r["komp"])] = f_
        beban_k[f_] += int(r["count"])
    peta_sdf = spark.createDataFrame(
        pd.DataFrame({"komp": list(peta_k.keys()),
                      "fold_cluster": list(peta_k.values())}))
    kerja = (kerja.join(peta_sdf, "komp", "left")
             .withColumn("fold_cluster",
                         F.coalesce(F.col("fold_cluster"), F.lit(0))).cache())
    kerja.count()

FOLDS = ["fold_acak","fold_spasial","fold_temporal","fold_cluster"]
print("  batas periode:", [int(x) for x in q])
print("  benua -> fold:", peta_b)
print()
for kol in FOLDS:
    p = (kerja.groupBy(kol, "inang").count().toPandas()
         .pivot(index=kol, columns="inang", values="count").fillna(0))
    print(f"=== {kol} (%) ===")
    print((100 * p.div(p.sum(axis=1), axis=0)).round(1).to_string())
    print(f"  sel kosong: {int((p==0).sum().sum())}")
    print()

kerja.select("idx","accession","inang","benua","negara","tahun","serotype",
             "komp", *FOLDS).write.mode("overwrite") \
     .parquet(str(STAGE / "fold_lengkap.parquet"))

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("kmer_df", "sisi", "sisi2", "kerja"), catatan="fold")


[BATCH PROCESSING] bangun 4 kolom fold ... mulai
[BATCH PROCESSING] bangun 4 kolom fold -> 0.13 menit, mem 38.07 GB, status OK
  batas periode: [2012, 2017, 2021, 2024]
  benua -> fold: {'Amerika Utara': 0, 'Asia': 1, 'Eropa': 2, 'Oseania': 3, 'Afrika': 4, 'Amerika Selatan': 4, 'Tidak terpetakan': 3}

=== fold_acak (%) ===
inang      babi  manusia  unggas
fold_acak                       
0          12.3     59.4    28.3
1          12.3     59.3    28.4
2          12.4     58.8    28.8
3          12.4     58.7    28.9
4          12.6     58.6    28.7
  sel kosong: 0

=== fold_spasial (%) ===
inang         babi  manusia  unggas
fold_spasial                       
0             14.3     65.8    19.9
1              4.1     34.1    61.8
2             24.5     53.4    22.1
3              1.0     93.6     5.4
4              7.9     47.0    45.0
  sel kosong: 0

=== fold_temporal (%) ===
inang          babi  manusia  unggas
fold_temporal                       
0              10.7     53.8    3

## LAPISAN 9 — ANALYTICS ENGINE C: SCALABLE MACHINE LEARNING (Spark MLlib) | 5 model x 4 skema x 5 fold

In [24]:
from pyspark.ml.classification import (LogisticRegression, LinearSVC, NaiveBayes,
    RandomForestClassifier, MultilayerPerceptronClassifier, OneVsRest)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

ev_f1  = MulticlassClassificationEvaluator(labelCol="label",
            predictionCol="prediction", metricName="f1")
ev_acc = MulticlassClassificationEvaluator(labelCol="label",
            predictionCol="prediction", metricName="accuracy")

DIM_MLP = 3000

def buat_model(nama, dim_in, n_kelas):
    if nama == "LogisticRegression":
        return LogisticRegression(featuresCol="fitur", labelCol="label",
                                  maxIter=30, regParam=0.01, elasticNetParam=0.0)
    if nama == "LinearSVC":
        return OneVsRest(classifier=LinearSVC(featuresCol="fitur",
                         labelCol="label", maxIter=20, regParam=0.01),
                         featuresCol="fitur", labelCol="label")
    if nama == "NaiveBayes":
        return NaiveBayes(featuresCol="fitur", labelCol="label",
                          modelType="multinomial", smoothing=1.0)
    if nama == "RandomForest":
        return RandomForestClassifier(featuresCol="fitur", labelCol="label",
                                      numTrees=60, maxDepth=12, seed=SEED)
    if nama == "MLP":
        return MultilayerPerceptronClassifier(
            featuresCol="fitur_sel", labelCol="label",
            layers=[dim_in, 128, 64, n_kelas],
            blockSize=32, maxIter=120, stepSize=0.03, solver="l-bfgs", seed=SEED)
    raise ValueError(nama)

def metrik_lengkap(pred, n_kelas, nama_kelas):
    per = {}
    hit = pred.groupBy("label", "prediction").count().toPandas()
    for i, nm in enumerate(nama_kelas):
        tp = int(hit[(hit.label == i) & (hit.prediction == i)]["count"].sum())
        fp = int(hit[(hit.label != i) & (hit.prediction == i)]["count"].sum())
        fn = int(hit[(hit.label == i) & (hit.prediction != i)]["count"].sum())
        prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
        per[nm] = 2 * prec * rec / max(prec + rec, 1e-9)
    return per, hit

MODEL_DIPAKAI = ["LogisticRegression","LinearSVC","NaiveBayes","RandomForest","MLP"]
SKEMA = {"1. Random k-fold": "fold_acak",
         "2. Blok spasial":  "fold_spasial",
         "3. Blok temporal": "fold_temporal",
         "4. Blok klaster":  "fold_cluster"}
print("Model :", MODEL_DIPAKAI)
print("Skema :", list(SKEMA))
print("Total run:", len(MODEL_DIPAKAI) * len(SKEMA) * N_FOLD)

Model : ['LogisticRegression', 'LinearSVC', 'NaiveBayes', 'RandomForest', 'MLP']
Skema : ['1. Random k-fold', '2. Blok spasial', '3. Blok temporal', '4. Blok klaster']
Total run: 100


In [25]:
from pyspark.ml.functions import vector_to_array

HASIL_FOLD, KONFUSI, LOSS_KURVA, LAPORAN_KELAS = [], [], [], []
EPS = 1e-15


def _log_loss_dari_prob(pred, kol_prob="probability"):
    d = (pred.withColumn("p_arr", vector_to_array(F.col(kol_prob)))
             .withColumn("p_true", F.element_at(F.col("p_arr"),
                                                (F.col("label") + 1).cast("int"))))
    return float(d.agg(F.avg(-F.log(F.greatest(F.col("p_true"),
                                               F.lit(EPS))))).first()[0])


def _log_loss_dari_raw(pred, kol_raw="rawPrediction"):
    d = (pred.withColumn("r", vector_to_array(F.col(kol_raw)))
             .withColumn("r_maks", F.array_max(F.col("r")))
             .withColumn("e", F.transform(F.col("r"),
                                          lambda x: F.exp(x - F.col("r_maks"))))
             .withColumn("z", F.aggregate(F.col("e"), F.lit(0.0),
                                          lambda acc, x: acc + x))
             .withColumn("e_true", F.element_at(F.col("e"),
                                                (F.col("label") + 1).cast("int")))
             .withColumn("p_true",
                         F.col("e_true") / F.greatest(F.col("z"), F.lit(EPS))))
    return float(d.agg(F.avg(-F.log(F.greatest(F.col("p_true"),
                                               F.lit(EPS))))).first()[0])


def hitung_log_loss(pred):
    """Log loss seragam untuk semua model. Memakai kolom probability bila ada,
    kalau tidak jatuh ke softmax(rawPrediction) -- kasus LinearSVC."""
    kolom = pred.columns
    if "probability" in kolom:
        return _log_loss_dari_prob(pred), "log_loss"
    if "rawPrediction" in kolom:
        return _log_loss_dari_raw(pred), "log_loss(softmax)"
    return float("nan"), "-"


def laporan_klasifikasi(hit_pdf, nama_kelas):
    baris, total = [], int(hit_pdf["count"].sum())
    for i, nm in enumerate(nama_kelas):
        tp = int(hit_pdf[(hit_pdf.label == i) & (hit_pdf.prediction == i)]["count"].sum())
        fp = int(hit_pdf[(hit_pdf.label != i) & (hit_pdf.prediction == i)]["count"].sum())
        fn = int(hit_pdf[(hit_pdf.label == i) & (hit_pdf.prediction != i)]["count"].sum())
        prec = tp / max(tp + fp, 1)
        rec  = tp / max(tp + fn, 1)
        baris.append({"kelas": nm, "precision": prec, "recall": rec,
                      "f1": 2 * prec * rec / max(prec + rec, 1e-12),
                      "support": tp + fn, "tp": tp, "fp": fp, "fn": fn})
    return pd.DataFrame(baris), total


with Tahap("latih semua model x skema x fold", "ANALYTICS/ML"):
    for nama_skema, kol_fold in SKEMA.items():
        for nama_model in MODEL_DIPAKAI:
            for f_ in range(N_FOLD):
                latih = kerja.filter(F.col(kol_fold) != f_)
                uji   = kerja.filter(F.col(kol_fold) == f_)
                n_l, n_u = latih.count(), uji.count()
                kelas_l = sorted(r["label"] for r in
                                 latih.select("label").distinct().collect())

                if n_u == 0 or len(kelas_l) < 2:
                    print(f"  [{nama_skema[:12]}|{nama_model}|f{f_}] dilewati")
                    continue
                if kelas_l != [float(i) for i in range(len(kelas_l))]:
                    print(f"  [{nama_skema[:12]}|{nama_model}|f{f_}] label tak kontinu")
                    continue

                t0 = time.time()
                kurva, loss_uji, jenis_loss = [], float("nan"), "-"
                try:
                    if nama_model == "MLP":
                        sel = UnivariateFeatureSelector(
                            featuresCol="fitur", outputCol="fitur_sel",
                            labelCol="label", featureType="continuous",
                            labelType="categorical",
                            selectionMode="numTopFeatures"
                        ).setSelectionThreshold(min(DIM_MLP, N_FITUR))
                        sel_m = sel.fit(latih)
                        m = buat_model("MLP", min(DIM_MLP, N_FITUR),
                                       len(kelas_l)).fit(sel_m.transform(latih))
                        pred = m.transform(sel_m.transform(uji)).cache()
                    else:
                        m = buat_model(nama_model, N_FITUR, len(kelas_l)).fit(latih)
                        pred = m.transform(uji).cache()

                    try:
                        if nama_model == "LogisticRegression" and m.hasSummary:
                            kurva = [float(x) for x in m.summary.objectiveHistory]
                    except Exception:
                        kurva = []

                    loss_uji, jenis_loss = hitung_log_loss(pred)

                    f1w = ev_f1.evaluate(pred)
                    acc = ev_acc.evaluate(pred)
                    hit = pred.groupBy("label", "prediction").count().toPandas()
                    rep, _tot = laporan_klasifikasi(hit, LABEL_NAMA)
                    macro = float(rep["f1"].mean())
                    per = dict(zip(rep["kelas"], rep["f1"]))
                    pred.unpersist()
                    status = "OK"
                except Exception as ex:
                    f1w = acc = macro = float("nan")
                    per, hit, rep = {}, None, None
                    status = type(ex).__name__ + ": " + str(ex)[:110]

                dt = (time.time() - t0) / 60
                HASIL_FOLD.append({
                    "mesin": "Spark", "skema": nama_skema, "model": nama_model,
                    "fold": f_, "n_latih": n_l, "n_uji": n_u,
                    "macro_f1": macro, "f1_weighted": f1w, "akurasi": acc,
                    "loss_uji": loss_uji, "jenis_loss": jenis_loss,
                    "loss_latih_akhir": kurva[-1] if kurva else np.nan,
                    "n_iterasi": len(kurva), "menit": round(dt, 3),
                    "status": status,
                    **{"f1_" + k_: v_ for k_, v_ in per.items()}})

                for it_, nilai in enumerate(kurva):
                    LOSS_KURVA.append({"mesin": "Spark", "skema": nama_skema,
                                       "model": nama_model, "fold": f_,
                                       "iterasi": it_, "loss": nilai})
                if hit is not None:
                    KONFUSI.append(hit.assign(mesin="Spark", skema=nama_skema,
                                              model=nama_model, fold=f_))
                if rep is not None:
                    LAPORAN_KELAS.append(rep.assign(mesin="Spark", skema=nama_skema,
                                                    model=nama_model, fold=f_))

                print(f"  [{nama_skema[:12]:<12}|{nama_model:<18}|f{f_}] "
                      f"macroF1={macro:.4f} acc={acc:.4f} "
                      f"log_loss={loss_uji:.4f} {dt:.1f}m {status[:18]}")

per_fold = pd.DataFrame(HASIL_FOLD)
per_fold.to_csv(OUT / "metrik_per_fold.csv", index=False)
if KONFUSI:
    pd.concat(KONFUSI, ignore_index=True).to_csv(OUT / "matriks_konfusi.csv", index=False)
if LAPORAN_KELAS:
    pd.concat(LAPORAN_KELAS, ignore_index=True).to_csv(
        OUT / "laporan_klasifikasi.csv", index=False)
if LOSS_KURVA:
    pd.DataFrame(LOSS_KURVA).to_csv(OUT / "kurva_loss.csv", index=False)

print(f"\nBaris hasil: {len(per_fold):,}")
print("Semua model dievaluasi dengan cross-entropy (log loss) pada data uji.")
print("LinearSVC: skor keputusan dikalibrasi softmax agar sebanding.")
audit("ml_selesai", n_run=int(len(per_fold)))

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("kmer_df", "sisi", "sisi2", "kerja"), catatan="ML")


[ANALYTICS/ML] latih semua model x skema x fold ... mulai
  [1. Random k-|LogisticRegression|f0] macroF1=0.9835 acc=0.9888 log_loss=0.0798 1.1m OK
  [1. Random k-|LogisticRegression|f1] macroF1=0.9844 acc=0.9893 log_loss=0.0769 1.2m OK
  [1. Random k-|LogisticRegression|f2] macroF1=0.9865 acc=0.9907 log_loss=0.0494 1.1m OK
  [1. Random k-|LogisticRegression|f3] macroF1=0.9825 acc=0.9880 log_loss=0.0881 0.9m OK
  [1. Random k-|LogisticRegression|f4] macroF1=0.9842 acc=0.9892 log_loss=0.0810 1.0m OK
  [1. Random k-|LinearSVC         |f0] macroF1=0.9850 acc=0.9896 log_loss=0.1103 5.1m OK
  [1. Random k-|LinearSVC         |f1] macroF1=0.9844 acc=0.9892 log_loss=0.1160 5.2m OK
  [1. Random k-|LinearSVC         |f2] macroF1=0.9845 acc=0.9895 log_loss=0.1113 5.1m OK
  [1. Random k-|LinearSVC         |f3] macroF1=0.9829 acc=0.9882 log_loss=0.1118 5.1m OK
  [1. Random k-|LinearSVC         |f4] macroF1=0.9851 acc=0.9898 log_loss=0.1161 5.1m OK
  [1. Random k-|NaiveBayes        |f0] macroF1=0.952

Exception ignored in: <function JavaWrapper.__del__ at 0x000002343B3265C0>
Traceback (most recent call last):
  File "c:\Users\Willy Boen\miniconda3\envs\tf_gpu\Lib\site-packages\pyspark\ml\wrapper.py", line 53, in __del__
    if SparkContext._active_spark_context and self._java_obj is not None:
                                              ^^^^^^^^^^^^^^
AttributeError: 'UnivariateFeatureSelector' object has no attribute '_java_obj'


  [1. Random k-|MLP               |f0] macroF1=nan acc=nan log_loss=nan 0.0m TypeError: Univari


Exception ignored in: <function JavaWrapper.__del__ at 0x000002343B3265C0>
Traceback (most recent call last):
  File "c:\Users\Willy Boen\miniconda3\envs\tf_gpu\Lib\site-packages\pyspark\ml\wrapper.py", line 53, in __del__
    if SparkContext._active_spark_context and self._java_obj is not None:
                                              ^^^^^^^^^^^^^^
AttributeError: 'UnivariateFeatureSelector' object has no attribute '_java_obj'


  [1. Random k-|MLP               |f1] macroF1=nan acc=nan log_loss=nan 0.0m TypeError: Univari


...

Exception ignored in: <function JavaWrapper.__del__ at 0x000002343B3265C0>
Traceback (most recent call last):
  File "c:\Users\Willy Boen\miniconda3\envs\tf_gpu\Lib\site-packages\pyspark\ml\wrapper.py", line 53, in __del__
    if SparkContext._active_spark_context and self._java_obj is not None:
                                              ^^^^^^^^^^^^^^
AttributeError: 'UnivariateFeatureSelector' object has no attribute '_java_obj'


  [4. Blok klas|MLP               |f3] macroF1=nan acc=nan log_loss=nan 0.0m TypeError: Univari


Exception ignored in: <function JavaWrapper.__del__ at 0x000002343B3265C0>
Traceback (most recent call last):
  File "c:\Users\Willy Boen\miniconda3\envs\tf_gpu\Lib\site-packages\pyspark\ml\wrapper.py", line 53, in __del__
    if SparkContext._active_spark_context and self._java_obj is not None:
                                              ^^^^^^^^^^^^^^
AttributeError: 'UnivariateFeatureSelector' object has no attribute '_java_obj'


  [4. Blok klas|MLP               |f4] macroF1=nan acc=nan log_loss=nan 0.0m TypeError: Univari
[ANALYTICS/ML] latih semua model x skema x fold -> 280.27 menit, mem 46.13 GB, status OK

Baris hasil: 100
Semua model dievaluasi dengan cross-entropy (log loss) pada data uji.
LinearSVC: skor keputusan dikalibrasi softmax agar sebanding.


{'ts': '2026-08-27T10:07:42.324599+00:00',
 'run_id': 'run_20260827T052524Z',
 'peristiwa': 'ml_selesai',
 'aktor': 'Willy Boen',
 'n_run': 100}

### LAPORAN KLASIFIKASI + CONFUSION MATRIX (SPARK)

In [26]:
lap = pd.concat(LAPORAN_KELAS, ignore_index=True)
kon = pd.concat(KONFUSI, ignore_index=True)

for nama_skema in SKEMA:
    for nama_model in MODEL_DIPAKAI:
        sub = lap[(lap.skema == nama_skema) & (lap.model == nama_model)]
        if sub.empty:
            continue
        print("=" * 74)
        print(f" {nama_skema}  |  {nama_model}   (rata-rata {N_FOLD} fold)")
        print("=" * 74)

        r = (sub.groupby("kelas")
             .agg(precision=("precision", "mean"), recall=("recall", "mean"),
                  f1=("f1", "mean"), sd_f1=("f1", "std"),
                  support=("support", "sum")).reset_index())
        print(f"{'kelas':<12}{'precision':>10}{'recall':>10}{'f1':>10}"
              f"{'sd f1':>9}{'support':>10}")
        for _, x in r.iterrows():
            sd = 0.0 if pd.isna(x["sd_f1"]) else x["sd_f1"]
            print(f"{x['kelas']:<12}{x['precision']:>10.4f}{x['recall']:>10.4f}"
                  f"{x['f1']:>10.4f}{sd:>9.4f}{int(x['support']):>10,}")

        tot = int(r["support"].sum())
        w = r["support"] / max(tot, 1)
        print("-" * 61)
        print(f"{'macro avg':<12}{r['precision'].mean():>10.4f}"
              f"{r['recall'].mean():>10.4f}{r['f1'].mean():>10.4f}{'':>9}{tot:>10,}")
        print(f"{'weighted avg':<12}{(r['precision']*w).sum():>10.4f}"
              f"{(r['recall']*w).sum():>10.4f}{(r['f1']*w).sum():>10.4f}"
              f"{'':>9}{tot:>10,}")

        k = kon[(kon.skema == nama_skema) & (kon.model == nama_model)]
        M = (k.groupby(["label", "prediction"])["count"].sum()
             .unstack(fill_value=0)
             .reindex(index=range(len(LABEL_NAMA)),
                      columns=range(len(LABEL_NAMA)), fill_value=0))
        M.index   = [f"aktual:{n}" for n in LABEL_NAMA]
        M.columns = [f"pred:{n}" for n in LABEL_NAMA]
        print("\nConfusion matrix (gabungan semua fold):")
        print(M.to_string())
        print("\nDinormalisasi per baris (%):")
        print((100 * M.div(M.sum(axis=1).replace(0, 1), axis=0)).round(1).to_string())

        lo = per_fold[(per_fold.skema == nama_skema) & (per_fold.model == nama_model)]
        print(f"\nLoss uji ({lo['jenis_loss'].iloc[0]}): "
              f"rata2={lo['loss_uji'].mean():.4f}  sd={lo['loss_uji'].std():.4f}  "
              f"min={lo['loss_uji'].min():.4f}  maks={lo['loss_uji'].max():.4f}")
        print()


 1. Random k-fold  |  LogisticRegression   (rata-rata 5 fold)
kelas        precision    recall        f1    sd f1   support
babi            0.9849    0.9509    0.9676   0.0030     4,943
manusia         0.9889    0.9940    0.9915   0.0006    23,506
unggas          0.9914    0.9957    0.9936   0.0015    11,407
-------------------------------------------------------------
macro avg       0.9884    0.9802    0.9842             39,856
weighted avg    0.9892    0.9892    0.9891             39,856

Confusion matrix (gabungan semua fold):
                pred:manusia  pred:unggas  pred:babi
aktual:manusia         23366           78         62
aktual:unggas             39        11358         10
aktual:babi              223           20       4700

Dinormalisasi per baris (%):
                pred:manusia  pred:unggas  pred:babi
aktual:manusia          99.4          0.3        0.3
aktual:unggas            0.3         99.6        0.1
aktual:babi              4.5          0.4       95.1

Loss uji

## LAPISAN 10 — ANALYTICS ENGINE D: STATISTICAL ANALYTICS | Bootstrap CI + uji signifikansi

In [27]:
from scipy import stats

def bootstrap_ci(nilai, n_boot=10000, alpha=0.05, seed=SEED):
    v = np.asarray([x for x in nilai if x == x], dtype=float)
    if len(v) < 2:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    b = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
    return float(np.percentile(b, 100*alpha/2)), float(np.percentile(b, 100*(1-alpha/2)))

baris = []
for (sk, mo), g in per_fold.groupby(["skema","model"]):
    v = g["macro_f1"].dropna().values
    if len(v) == 0:
        continue
    lo, hi = bootstrap_ci(v)
    baris.append({"skema": sk, "model": mo,
                  "macro_f1_rata2": round(float(np.mean(v)), 4),
                  "sd_antar_fold": round(float(np.std(v, ddof=1)), 4) if len(v) > 1 else 0.0,
                  "min": round(float(np.min(v)), 4),
                  "maks": round(float(np.max(v)), 4),
                  "ci95_bawah": round(lo, 4), "ci95_atas": round(hi, 4),
                  "n_fold": int(len(v)),
                  "menit_total": round(float(g["menit"].sum()), 2)})
ringkas_model = pd.DataFrame(baris).sort_values(["model","skema"])
ringkas_model.to_csv(OUT / "ringkasan_model_skema_ci.csv", index=False)
print(ringkas_model.to_string(index=False))

           skema              model  macro_f1_rata2  sd_antar_fold    min   maks  ci95_bawah  ci95_atas  n_fold  menit_total
1. Random k-fold          LinearSVC          0.9844         0.0009 0.9829 0.9851      0.9836     0.9849       5        25.63
 2. Blok spasial          LinearSVC          0.9299         0.0340 0.8734 0.9654      0.8993     0.9537       5        24.69
3. Blok temporal          LinearSVC          0.9507         0.0737 0.8192 0.9893      0.8843     0.9876       5        24.56
 4. Blok klaster          LinearSVC          0.9816         0.0040 0.9759 0.9871      0.9782     0.9848       5        30.23
1. Random k-fold LogisticRegression          0.9842         0.0015 0.9825 0.9865      0.9831     0.9854       5         5.22
 2. Blok spasial LogisticRegression          0.8533         0.1541 0.5790 0.9448      0.7135     0.9319       5         4.53
3. Blok temporal LogisticRegression          0.9516         0.0472 0.8687 0.9847      0.9090     0.9774       5         4.62


In [28]:
PASANG = [("1. Random k-fold","2. Blok spasial"),
          ("1. Random k-fold","3. Blok temporal"),
          ("1. Random k-fold","4. Blok klaster"),
          ("2. Blok spasial", "4. Blok klaster")]

uji = []
for mo in per_fold["model"].unique():
    sub = per_fold[per_fold["model"] == mo]
    for a, b in PASANG:
        va = sub[sub["skema"] == a].sort_values("fold")["macro_f1"].dropna().values
        vb = sub[sub["skema"] == b].sort_values("fold")["macro_f1"].dropna().values
        n = min(len(va), len(vb))
        if n < 3:
            continue
        va, vb = va[:n], vb[:n]
        d_ = va - vb
        try:    _, wp = stats.wilcoxon(va, vb)
        except Exception: wp = float("nan")
        _, tp = stats.ttest_rel(va, vb)
        uji.append({"model": mo, "perbandingan": a + " vs " + b, "n": n,
                    "selisih_rata2": round(float(np.mean(d_)), 4),
                    "penurunan_pct": round(100*float(np.mean(d_))/max(float(np.mean(va)),1e-9), 2),
                    "wilcoxon_p": round(float(wp), 4) if wp == wp else np.nan,
                    "ttest_p": round(float(tp), 4),
                    "cohen_d": round(float(np.mean(d_)/max(np.std(d_, ddof=1), 1e-9)), 3),
                    "signifikan": "ya" if tp < 0.05 else "tidak"})
df_uji = pd.DataFrame(uji)
df_uji.to_csv(OUT / "uji_statistik.csv", index=False)
print(df_uji.to_string(index=False))

             model                         perbandingan  n  selisih_rata2  penurunan_pct  wilcoxon_p  ttest_p  cohen_d signifikan
LogisticRegression  1. Random k-fold vs 2. Blok spasial  5         0.1309          13.30      0.0625   0.1306    0.849      tidak
LogisticRegression 1. Random k-fold vs 3. Blok temporal  5         0.0326           3.31      0.1250   0.1936    0.698      tidak
LogisticRegression  1. Random k-fold vs 4. Blok klaster  5         0.0114           1.16      0.1250   0.1164    0.893      tidak
LogisticRegression   2. Blok spasial vs 4. Blok klaster  5        -0.1195         -14.00      0.0625   0.1723   -0.742      tidak
         LinearSVC  1. Random k-fold vs 2. Blok spasial  5         0.0545           5.53      0.0625   0.0234    1.597         ya
         LinearSVC 1. Random k-fold vs 3. Blok temporal  5         0.0337           3.42      0.6250   0.3671    0.454      tidak
         LinearSVC  1. Random k-fold vs 4. Blok klaster  5         0.0028           0.28  

## LAPISAN 11 — PEMBANDING TANPA SPARK | scikit-learn single-node, uji batas skalabilitas

In [29]:
from sklearn.feature_extraction.text import CountVectorizer as SkCV, TfidfTransformer
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.svm import LinearSVC as SkSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier as SkRF
from sklearn.neural_network import MLPClassifier as SkMLP
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import (f1_score, accuracy_score, log_loss,
                             classification_report, confusion_matrix)
import scipy.sparse as sp

# Kurangi daftar ini bila memori driver terbatas:
SK_MODEL = ["LogisticRegression", "LinearSVC", "NaiveBayes", "RandomForest", "MLP"]
SK_HASIL, SK_KONFUSI, SK_LOSS, SK_LAPORAN = [], [], [], []
IDX_KELAS = list(range(len(LABEL_NAMA)))


def softmax_baris(skor):
    """Kalibrasi skor keputusan -> probabilitas, agar log loss sebanding."""
    skor = np.asarray(skor, dtype=float)
    if skor.ndim == 1:
        skor = np.column_stack([-skor, skor])
    skor = skor - skor.max(axis=1, keepdims=True)
    e = np.exp(skor)
    return e / e.sum(axis=1, keepdims=True)


with Tahap("ekspor ke pandas", "ANALYTICS/NON-SPARK"):
    pdf_all = (kerja.select("idx", "inang", *FOLDS)
               .join(kmer_df.select("idx", "kmers"), "idx").toPandas())
    pdf_all["doc"] = pdf_all["kmers"].map(lambda a: " ".join(a))
    pdf_all = pdf_all.drop(columns=["kmers"])
    print(f"  baris di driver: {len(pdf_all):,}")

with Tahap("vektorisasi sklearn", "ANALYTICS/NON-SPARK"):
    t0 = time.time()
    cv = SkCV(analyzer=lambda s: s.split(), min_df=5, max_features=VOCAB)
    X_all = TfidfTransformer().fit_transform(cv.fit_transform(pdf_all["doc"]))
    y_all = pdf_all["inang"].map({n: i for i, n in enumerate(LABEL_NAMA)}).values
    menit_fitur_sk = (time.time() - t0) / 60
    print(f"  matriks {X_all.shape}, nnz={X_all.nnz:,}, {menit_fitur_sk:.2f} menit")


def kurva_loss_lr(Xtr, ytr, n_iter=30):
    clf, kur = SkLR(max_iter=1, warm_start=True, n_jobs=-1), []
    for _ in range(n_iter):
        clf.fit(Xtr, ytr)
        kur.append(float(log_loss(ytr, clf.predict_proba(Xtr), labels=IDX_KELAS)))
    return clf, kur


with Tahap("latih model sklearn per fold", "ANALYTICS/NON-SPARK"):
    for nama_skema, kol_fold in SKEMA.items():
        fold_arr = pdf_all[kol_fold].values
        for nama_model in SK_MODEL:
            for f_ in range(N_FOLD):
                m_uji = fold_arr == f_
                if m_uji.sum() == 0 or len(np.unique(y_all[~m_uji])) < 2:
                    continue
                Xtr, ytr = X_all[~m_uji], y_all[~m_uji]
                Xte, yte = X_all[m_uji], y_all[m_uji]

                t0, kur = time.time(), []
                loss_uji, jenis = float("nan"), "log_loss"
                try:
                    if nama_model == "LogisticRegression":
                        clf, kur = kurva_loss_lr(Xtr, ytr)
                        yp = clf.predict(Xte)
                        loss_uji = float(log_loss(yte, clf.predict_proba(Xte),
                                                  labels=IDX_KELAS))
                    elif nama_model == "LinearSVC":
                        clf = SkSVC(max_iter=2000).fit(Xtr, ytr)
                        yp = clf.predict(Xte)
                        loss_uji = float(log_loss(
                            yte, softmax_baris(clf.decision_function(Xte)),
                            labels=IDX_KELAS))
                        jenis = "log_loss(softmax)"
                    elif nama_model == "NaiveBayes":
                        clf = MultinomialNB().fit(Xtr, ytr)
                        yp = clf.predict(Xte)
                        loss_uji = float(log_loss(yte, clf.predict_proba(Xte),
                                                  labels=IDX_KELAS))
                    elif nama_model == "RandomForest":
                        clf = SkRF(n_estimators=60, max_depth=12, n_jobs=-1,
                                   random_state=SEED).fit(Xtr, ytr)
                        yp = clf.predict(Xte)
                        loss_uji = float(log_loss(yte, clf.predict_proba(Xte),
                                                  labels=IDX_KELAS))
                    else:
                        skb = SelectKBest(chi2, k=min(DIM_MLP, Xtr.shape[1]))
                        Xtr2 = skb.fit_transform(Xtr, ytr)
                        Xte2 = skb.transform(Xte)
                        clf = SkMLP(hidden_layer_sizes=(128, 64), max_iter=120,
                                    learning_rate_init=0.03, random_state=SEED)
                        clf.fit(Xtr2, ytr)
                        kur = [float(x) for x in clf.loss_curve_]
                        yp = clf.predict(Xte2)
                        loss_uji = float(log_loss(yte, clf.predict_proba(Xte2),
                                                  labels=IDX_KELAS))

                    macro = float(f1_score(yte, yp, average="macro"))
                    acc = float(accuracy_score(yte, yp))
                    rep_d = classification_report(yte, yp, labels=IDX_KELAS,
                                                  target_names=LABEL_NAMA,
                                                  output_dict=True, zero_division=0)
                    cm = confusion_matrix(yte, yp, labels=IDX_KELAS)
                    status = "OK"
                except Exception as ex:
                    macro = acc = float("nan"); rep_d, cm = None, None
                    status = type(ex).__name__ + ": " + str(ex)[:90]

                dt = (time.time() - t0) / 60
                SK_HASIL.append({
                    "mesin": "sklearn", "skema": nama_skema, "model": nama_model,
                    "fold": f_, "n_latih": int((~m_uji).sum()),
                    "n_uji": int(m_uji.sum()), "macro_f1": macro, "akurasi": acc,
                    "loss_uji": loss_uji, "jenis_loss": jenis,
                    "loss_latih_akhir": kur[-1] if kur else np.nan,
                    "n_iterasi": len(kur), "menit": round(dt, 3), "status": status})

                for it_, v_ in enumerate(kur):
                    SK_LOSS.append({"mesin": "sklearn", "skema": nama_skema,
                                    "model": nama_model, "fold": f_,
                                    "iterasi": it_, "loss": v_})
                if rep_d:
                    for nm in LABEL_NAMA:
                        d = rep_d[nm]
                        SK_LAPORAN.append({
                            "mesin": "sklearn", "skema": nama_skema,
                            "model": nama_model, "fold": f_, "kelas": nm,
                            "precision": d["precision"], "recall": d["recall"],
                            "f1": d["f1-score"], "support": int(d["support"])})
                if cm is not None:
                    for i in IDX_KELAS:
                        for j in IDX_KELAS:
                            SK_KONFUSI.append({
                                "mesin": "sklearn", "skema": nama_skema,
                                "model": nama_model, "fold": f_,
                                "label": i, "prediction": j,
                                "count": int(cm[i, j])})

                print(f"  [{nama_skema[:12]:<12}|{nama_model:<18}|f{f_}] "
                      f"macroF1={macro:.4f} acc={acc:.4f} "
                      f"log_loss={loss_uji:.4f} {dt:.1f}m {status[:18]}")

sk_per_fold = pd.DataFrame(SK_HASIL)
sk_per_fold.to_csv(OUT / "metrik_per_fold_sklearn.csv", index=False)
pd.DataFrame(SK_LAPORAN).to_csv(OUT / "laporan_klasifikasi_sklearn.csv", index=False)
pd.DataFrame(SK_KONFUSI).to_csv(OUT / "matriks_konfusi_sklearn.csv", index=False)
pd.DataFrame(SK_LOSS).to_csv(OUT / "kurva_loss_sklearn.csv", index=False)
print(f"\nBaris hasil sklearn: {len(sk_per_fold):,}")

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
del X_all, pdf_all
bersihkan(kecuali=("kmer_df", "sisi", "sisi2", "kerja"), catatan="sklearn")


[ANALYTICS/NON-SPARK] ekspor ke pandas ... mulai
  baris di driver: 39,856
[ANALYTICS/NON-SPARK] ekspor ke pandas -> 0.30 menit, mem 48.09 GB, status OK
[ANALYTICS/NON-SPARK] vektorisasi sklearn ... mulai
  matriks (39856, 210221), nnz=67,374,560, 0.66 menit
[ANALYTICS/NON-SPARK] vektorisasi sklearn -> 0.66 menit, mem 48.44 GB, status OK
[ANALYTICS/NON-SPARK] latih model sklearn per fold ... mulai
  [1. Random k-|LogisticRegression|f0] macroF1=0.9446 acc=0.9619 log_loss=0.1820 2.8m OK
  [1. Random k-|LogisticRegression|f1] macroF1=0.9410 acc=0.9596 log_loss=0.1848 1.4m OK
  [1. Random k-|LogisticRegression|f2] macroF1=0.9433 acc=0.9608 log_loss=0.1855 1.4m OK
  [1. Random k-|LogisticRegression|f3] macroF1=0.9399 acc=0.9582 log_loss=0.1914 1.4m OK
  [1. Random k-|LogisticRegression|f4] macroF1=0.9438 acc=0.9613 log_loss=0.1828 1.3m OK
  [1. Random k-|LinearSVC         |f0] macroF1=0.9837 acc=0.9888 log_loss=0.2062 0.2m OK
  [1. Random k-|LinearSVC         |f1] macroF1=0.9844 acc=0.9893 

### PERBANDINGAN SPARK vs TANPA SPARK | loss, metrik, confusion matrix

In [30]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

gabung = pd.concat([per_fold, sk_per_fold], ignore_index=True)
gabung.to_csv(OUT / "metrik_gabungan.csv", index=False)

ringkas_mesin = (gabung.groupby(["mesin", "skema", "model"])
                 .agg(macro_f1=("macro_f1", "mean"), sd_f1=("macro_f1", "std"),
                      akurasi=("akurasi", "mean"), loss_uji=("loss_uji", "mean"),
                      menit=("menit", "sum")).round(4).reset_index())
ringkas_mesin.to_csv(OUT / "ringkasan_spark_vs_sklearn.csv", index=False)

print("=" * 100)
print(" SPARK vs SCIKIT-LEARN")
print("=" * 100)
print(ringkas_mesin.to_string(index=False))

piv = ringkas_mesin.pivot_table(index=["skema", "model"], columns="mesin",
                                values=["macro_f1", "loss_uji", "menit"])
if ("macro_f1", "Spark") in piv.columns and ("macro_f1", "sklearn") in piv.columns:
    d = (piv[("macro_f1", "Spark")] - piv[("macro_f1", "sklearn")]).round(4)
    print("\nSelisih macro-F1 (Spark - sklearn):")
    print(d.to_string())
    print(f"\nSelisih absolut rata-rata: {d.abs().mean():.5f}")
    print("  -> mendekati nol berarti kedua implementasi konsisten")

kl = pd.concat([pd.DataFrame(LOSS_KURVA), pd.DataFrame(SK_LOSS)], ignore_index=True)
kl.to_csv(OUT / "kurva_loss_gabungan.csv", index=False)

fig, ax = plt.subplots(2, 3, figsize=(20, 11))

sp_lr = kl[(kl.mesin == "Spark") & (kl.model == "LogisticRegression")]
for s in SKEMA:
    g = sp_lr[sp_lr.skema == s].groupby("iterasi")["loss"].mean()
    if len(g):
        ax[0, 0].plot(g.index, g.values, marker="o", ms=3, label=s)
ax[0, 0].set_xlabel("iterasi L-BFGS"); ax[0, 0].set_ylabel("loss pelatihan")
ax[0, 0].set_title("Kurva loss LogisticRegression (Spark)")
ax[0, 0].legend(fontsize=7); ax[0, 0].grid(alpha=.25)

for mesin, gaya in [("Spark", "-"), ("sklearn", "--")]:
    g = (kl[(kl.mesin == mesin) & (kl.model == "LogisticRegression") &
            (kl.skema == list(SKEMA)[0])].groupby("iterasi")["loss"].mean())
    if len(g):
        ax[0, 1].plot(g.index, g.values, gaya, marker="o", ms=3, label=mesin)
ax[0, 1].set_xlabel("iterasi"); ax[0, 1].set_ylabel("loss pelatihan")
ax[0, 1].set_title("Konvergensi: Spark vs sklearn (random k-fold)")
ax[0, 1].legend(); ax[0, 1].grid(alpha=.25)

gabung.pivot_table(index="model", columns="mesin", values="loss_uji",
                   aggfunc="mean").plot(kind="bar", ax=ax[0, 2], rot=25)
ax[0, 2].set_ylabel("loss data uji"); ax[0, 2].set_title("Loss uji per model")
ax[0, 2].grid(alpha=.25, axis="y")

gabung.pivot_table(index="model", columns="mesin", values="macro_f1",
                   aggfunc="mean").plot(kind="bar", ax=ax[1, 0], rot=25)
ax[1, 0].set_ylabel("macro-F1"); ax[1, 0].set_title("macro-F1: Spark vs sklearn")
ax[1, 0].grid(alpha=.25, axis="y")

gabung.pivot_table(index="model", columns="mesin", values="menit",
                   aggfunc="sum").plot(kind="bar", ax=ax[1, 1], rot=25, logy=True)
ax[1, 1].set_ylabel("total menit (log)"); ax[1, 1].set_title("Waktu komputasi")
ax[1, 1].grid(alpha=.25, axis="y")

kon_all = pd.concat(KONFUSI, ignore_index=True)
kk = kon_all[(kon_all.model == "LogisticRegression") &
             (kon_all.skema == list(SKEMA)[-1])]
M = (kk.groupby(["label", "prediction"])["count"].sum().unstack(fill_value=0)
     .reindex(index=range(len(LABEL_NAMA)), columns=range(len(LABEL_NAMA)),
              fill_value=0))
Mn = 100 * M.div(M.sum(axis=1).replace(0, 1), axis=0)
im = ax[1, 2].imshow(Mn.values, cmap="Blues", vmin=0, vmax=100)
ax[1, 2].set_xticks(range(len(LABEL_NAMA)))
ax[1, 2].set_xticklabels(LABEL_NAMA, rotation=30)
ax[1, 2].set_yticks(range(len(LABEL_NAMA)))
ax[1, 2].set_yticklabels(LABEL_NAMA)
for i in range(len(LABEL_NAMA)):
    for j in range(len(LABEL_NAMA)):
        ax[1, 2].text(j, i, f"{Mn.values[i, j]:.1f}", ha="center", va="center",
                      color="white" if Mn.values[i, j] > 50 else "black", fontsize=9)
ax[1, 2].set_xlabel("prediksi"); ax[1, 2].set_ylabel("aktual")
ax[1, 2].set_title(f"Confusion matrix % — LR, {list(SKEMA)[-1]}")
plt.colorbar(im, ax=ax[1, 2], fraction=0.046)

plt.tight_layout()
plt.savefig(OUT / "loss_dan_konfusi.png", dpi=130)
plt.close()
print(f"\nTersimpan: {OUT / 'loss_dan_konfusi.png'}")


 SPARK vs SCIKIT-LEARN
  mesin            skema              model  macro_f1  sd_f1  akurasi  loss_uji  menit
  Spark 1. Random k-fold          LinearSVC    0.9844 0.0009   0.9893    0.1131 25.630
  Spark 1. Random k-fold LogisticRegression    0.9842 0.0015   0.9892    0.0751  5.216
  Spark 1. Random k-fold                MLP       NaN    NaN      NaN       NaN  0.000
  Spark 1. Random k-fold         NaiveBayes    0.9536 0.0032   0.9697    1.0439  0.578
  Spark 1. Random k-fold       RandomForest    0.9737 0.0018   0.9830    0.0756 32.100
  Spark  2. Blok spasial          LinearSVC    0.9299 0.0340   0.9663    0.1984 24.689
  Spark  2. Blok spasial LogisticRegression    0.8533 0.1541   0.8999    1.8380  4.526
  Spark  2. Blok spasial                MLP       NaN    NaN      NaN       NaN  0.000
  Spark  2. Blok spasial         NaiveBayes    0.8524 0.1106   0.9405    2.0527  0.593
  Spark  2. Blok spasial       RandomForest    0.8603 0.0491   0.9424    0.1755 36.079
  Spark 3. Blok temp

## LAPISAN 12 — ANALYTICS ENGINE E: VALIDASI BIOLOGIS | Translasi ORF, penomoran H3, pemetaan k-mer ke situs fungsional

In [31]:
KODON = {}
_b = "TCAG"
_aa = ("FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG")
_i = 0
for a in _b:
    for b in _b:
        for c in _b:
            KODON[a+b+c] = _aa[_i]; _i += 1

def translasi(nt, frame=0):
    s = nt[frame:]
    s = s[:len(s) - len(s) % 3]
    return "".join(KODON.get(s[i:i+3], "X") for i in range(0, len(s), 3))

def orf_terpanjang(nt):
    terbaik, info = "", (0, 0)
    for fr in range(3):
        prot = translasi(nt, fr)
        for seg_ in prot.split("*"):
            if len(seg_) > len(terbaik):
                terbaik = seg_
                info = (fr, prot.find(seg_))
    return terbaik, info

# Motif jangkar konservatif pada HA1/HA2 untuk penomoran relatif
MOTIF_JANGKAR = {
    "sinyal_awal":   "MKA",
    "fusi_HA2":      "GLFGAIAGFIE",
    "cys_HA1":       "CPKYVKQNTLKLAT",
}

# Situs fungsional dalam penomoran H3 mature HA1
SITUS_H3 = {
    "RBS_190_helix":  list(range(186, 196)),
    "RBS_220_loop":   list(range(221, 229)),
    "RBS_130_loop":   list(range(133, 139)),
    "Sa_antigenik":   list(range(124, 130)) + list(range(155, 161)),
    "Sb_antigenik":   list(range(186, 195)),
    "glikosilasi":    [8, 22, 38, 165, 246, 285],
    "posisi_kunci":   [98, 136, 153, 183, 190, 194, 195, 225, 226, 227, 228],
}
POSISI_KRITIS = set(SITUS_H3["RBS_190_helix"] + SITUS_H3["RBS_220_loop"] +
                    SITUS_H3["RBS_130_loop"] + SITUS_H3["posisi_kunci"])
print(f"Kodon tabel: {len(KODON)}  |  posisi kritis H3: {len(POSISI_KRITIS)}")

Kodon tabel: 64  |  posisi kritis H3: 27


In [32]:
with Tahap("translasi ORF batch", "ANALYTICS/BIOLOGI"):
    prot_pdf = (ha.select("idx","accession","seq","inang","benua","tahun","serotype")
                .toPandas())
    hasil = prot_pdf["seq"].map(orf_terpanjang)
    prot_pdf["protein"] = hasil.map(lambda t: t[0])
    prot_pdf["frame"]   = hasil.map(lambda t: t[1][0])
    prot_pdf["awal_aa"] = hasil.map(lambda t: t[1][1])
    prot_pdf["len_prot"] = prot_pdf["protein"].str.len()
    prot_pdf = prot_pdf[prot_pdf["len_prot"].between(480, 600)]
    print(f"  protein valid: {len(prot_pdf):,}")
    print(prot_pdf["len_prot"].describe().to_string())

def cari_jangkar(prot):
    for nama, motif in MOTIF_JANGKAR.items():
        p = prot.find(motif)
        if p >= 0:
            return nama, p
    return None, -1

_j = prot_pdf["protein"].map(cari_jangkar)
prot_pdf["jangkar"]     = _j.map(lambda t: t[0])
prot_pdf["pos_jangkar"] = _j.map(lambda t: t[1])
print()
print(prot_pdf["jangkar"].value_counts(dropna=False).to_string())

[ANALYTICS/BIOLOGI] translasi ORF batch ... mulai
  protein valid: 39,852
count    39852.000000
mean       565.810373
std         10.771741
min        489.000000
25%        566.000000
50%        566.000000
75%        571.000000
max        590.000000
[ANALYTICS/BIOLOGI] translasi ORF batch -> 0.29 menit, mem 29.07 GB, status OK

jangkar
sinyal_awal    14918
fusi_HA2       13352
None           11541
cys_HA1           41


In [33]:
with Tahap("konsensus referensi + peta penomoran H3", "ANALYTICS/BIOLOGI"):
    ref_pool = prot_pdf[(prot_pdf["serotype"] == "H3N2") &
                        (prot_pdf["inang"] == "manusia")]
    if len(ref_pool) < 50:
        ref_pool = prot_pdf[prot_pdf["inang"] == "manusia"]
    panjang_modus = int(ref_pool["len_prot"].mode().iloc[0])
    ref_sub = ref_pool[ref_pool["len_prot"] == panjang_modus]["protein"]
    konsensus = "".join(
        collections.Counter(kol).most_common(1)[0][0]
        for kol in zip(*ref_sub.head(3000)))
    print(f"  referensi dari {len(ref_sub):,} sekuens, panjang {panjang_modus}")

    # HA1 mature dimulai setelah peptida sinyal (~16 aa pada H3)
    OFFSET_SINYAL = 16
    def ke_h3(pos_prot):
        return pos_prot - OFFSET_SINYAL + 1

    KONS = konsensus
    print(f"  konsensus 60 aa awal: {KONS[:60]}")
    audit("referensi_konsensus", panjang=len(KONS), n_sumber=int(len(ref_sub)))

[ANALYTICS/BIOLOGI] konsensus referensi + peta penomoran H3 ... mulai
  referensi dari 5,545 sekuens, panjang 571
  konsensus 60 aa awal: IILLTMKTIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTITNDRIEVTNA
[ANALYTICS/BIOLOGI] konsensus referensi + peta penomoran H3 -> 0.00 menit, mem 29.09 GB, status OK


In [34]:
with Tahap("lokalisasi k-mer diskriminatif", "ANALYTICS/BIOLOGI"):
    lr_best = None
    for nama_skema, kol_fold in [("4. Blok klaster","fold_cluster"),
                                 ("2. Blok spasial","fold_spasial")]:
        try:
            latih = kerja.filter(F.col(kol_fold) != 0)
            lr_best = LogisticRegression(featuresCol="fitur", labelCol="label",
                        maxIter=30, regParam=0.01).fit(latih)
            SKEMA_KOEF = nama_skema
            break
        except Exception as ex:
            print("  gagal:", ex)
    koef = lr_best.coefficientMatrix.toArray()
    print(f"  koefisien dari {SKEMA_KOEF}, bentuk {koef.shape}")

    TOP = 200
    kmer_top = []
    for i, nm in enumerate(LABEL_NAMA):
        urut = np.argsort(koef[i])[::-1][:TOP]
        for j in urut:
            if j < len(KOSAKATA):
                kmer_top.append({"inang": nm, "kmer": KOSAKATA[j],
                                 "koefisien": float(koef[i][j])})
    df_kmer = pd.DataFrame(kmer_top)
    df_kmer.to_csv(OUT / "kmer_penting.csv", index=False)
    print(f"  k-mer top tersimpan: {len(df_kmer):,}")

    sampel = prot_pdf.sample(min(6000, len(prot_pdf)), random_state=SEED)
    peta_pos = []
    for _, r in sampel.iterrows():
        nt, prot, fr, awal = r["seq"], r["protein"], r["frame"], r["awal_aa"]
        for _, km in df_kmer.iterrows():
            p = nt.find(km["kmer"])
            if p < 0:
                continue
            pos_aa = (p - fr) // 3 - awal
            if 0 <= pos_aa < len(prot):
                peta_pos.append({"inang_kmer": km["inang"], "kmer": km["kmer"],
                                 "koefisien": km["koefisien"],
                                 "pos_protein": pos_aa,
                                 "pos_h3": ke_h3(pos_aa),
                                 "inang_seq": r["inang"]})
    df_pos = pd.DataFrame(peta_pos)
    df_pos.to_csv(OUT / "posisi_kmer_h3.csv", index=False)
    print(f"  pemetaan posisi: {len(df_pos):,} baris")

[ANALYTICS/BIOLOGI] lokalisasi k-mer diskriminatif ... mulai
  koefisien dari 4. Blok klaster, bentuk (3, 210221)
  k-mer top tersimpan: 600
  pemetaan posisi: 1,223 baris
[ANALYTICS/BIOLOGI] lokalisasi k-mer diskriminatif -> 4.98 menit, mem 33.99 GB, status OK


In [35]:
with Tahap("uji pengayaan permutasi", "ANALYTICS/STATISTIK"):
    N_PERM = 5000
    panjang_h3 = int(np.nanpercentile(df_pos["pos_h3"], 99)) if len(df_pos) else 550
    hasil_enrich = []

    for nama_situs, posisi in SITUS_H3.items():
        target = set(posisi)
        for inang_ in LABEL_NAMA:
            obs_pos = df_pos[df_pos["inang_kmer"] == inang_]["pos_h3"].dropna().astype(int)
            if len(obs_pos) < 20:
                continue
            n_hit = int(obs_pos.isin(target).sum())
            frak_obs = n_hit / len(obs_pos)
            rng = np.random.default_rng(SEED)
            null = rng.integers(1, panjang_h3 + 1, size=(N_PERM, len(obs_pos)))
            frak_null = np.isin(null, list(target)).mean(axis=1)
            p_emp = float((frak_null >= frak_obs).mean())
            hasil_enrich.append({
                "situs": nama_situs, "inang": inang_,
                "n_kmer_terpeta": int(len(obs_pos)), "n_hit": n_hit,
                "frak_observasi": round(frak_obs, 5),
                "frak_null_rata2": round(float(frak_null.mean()), 5),
                "rasio_pengayaan": round(frak_obs / max(float(frak_null.mean()), 1e-9), 3),
                "p_permutasi": round(p_emp, 5),
                "signifikan": "ya" if p_emp < 0.05 else "tidak"})

    df_enrich = pd.DataFrame(hasil_enrich)
    df_enrich.to_csv(OUT / "pengayaan_situs_fungsional.csv", index=False)

print(df_enrich.to_string(index=False))
audit("pengayaan", n_uji=int(len(df_enrich)),
      signifikan=int((df_enrich["signifikan"] == "ya").sum()))

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "kerja"), catatan="biologi")


[ANALYTICS/STATISTIK] uji pengayaan permutasi ... mulai
[ANALYTICS/STATISTIK] uji pengayaan permutasi -> 0.01 menit, mem 33.99 GB, status OK
        situs   inang  n_kmer_terpeta  n_hit  frak_observasi  frak_null_rata2  rasio_pengayaan  p_permutasi signifikan
RBS_190_helix manusia             279      3         0.01075          0.01887            0.570       0.8902      tidak
RBS_190_helix  unggas             613     17         0.02773          0.01878            1.477       0.0726      tidak
RBS_190_helix    babi             331      2         0.00604          0.01882            0.321       0.9856      tidak
 RBS_220_loop manusia             279      3         0.01075          0.01517            0.709       0.7934      tidak
 RBS_220_loop  unggas             613      1         0.00163          0.01513            0.108       0.9998      tidak
 RBS_220_loop    babi             331      0         0.00000          0.01511            0.000       1.0000      tidak
 RBS_130_loop manusia     

{'ts': '2026-08-27T13:03:43.649057+00:00',
 'run_id': 'run_20260827T052524Z',
 'peristiwa': 'pengayaan',
 'aktor': 'Willy Boen',
 'n_uji': 21,
 'signifikan': 5}

## LAPISAN 13 — ANALYTICS ENGINE F: HOLDOUT ZOONOTIK EKSTERNAL + GENERALISASI LINTAS SEGMEN

In [36]:
SEROTIPE_ZOONOTIK = ["H5N1","H7N9","H9N2","H5N6","H7N7","H10N8","H5N8"]

with Tahap("holdout zoonotik", "ANALYTICS/ML"):
    zoo = kerja.filter(F.col("serotype").isin(SEROTIPE_ZOONOTIK)).cache()
    non = kerja.filter(~F.coalesce(F.col("serotype"), F.lit("NA"))
                       .isin(SEROTIPE_ZOONOTIK)).cache()
    n_zoo, n_non = zoo.count(), non.count()
    print(f"  zoonotik: {n_zoo:,}  |  latih: {n_non:,}")

    hasil_zoo = []
    if n_zoo > 100 and non.select("label").distinct().count() >= 2:
        m_zoo = LogisticRegression(featuresCol="fitur", labelCol="label",
                                   maxIter=30, regParam=0.01).fit(non)
        pred_zoo = m_zoo.transform(zoo).cache()
        acc = ev_acc.evaluate(pred_zoo)
        f1w = ev_f1.evaluate(pred_zoo)
        per, hit = metrik_lengkap(pred_zoo, N_KELAS, LABEL_NAMA)
        print(f"  akurasi holdout: {acc:.4f}  macro-F1: {np.mean(list(per.values())):.4f}")

        rincian = (pred_zoo.groupBy("serotype","inang","prediction").count()
                   .toPandas())
        rincian["prediksi_label"] = rincian["prediction"].map(
            lambda i: LABEL_NAMA[int(i)])
        rincian.to_csv(OUT / "holdout_zoonotik_rincian.csv", index=False)
        hasil_zoo.append({"n_zoonotik": n_zoo, "akurasi": round(acc, 4),
                          "f1_weighted": round(f1w, 4),
                          "macro_f1": round(float(np.mean(list(per.values()))), 4)})
        print()
        pv = (rincian.groupby(["serotype","inang","prediksi_label"])["count"]
              .sum().unstack(fill_value=0))
        print(pv.to_string())
        pred_zoo.unpersist()
    else:
        print("  sampel zoonotik tidak cukup")
    pd.DataFrame(hasil_zoo).to_csv(OUT / "holdout_zoonotik.csv", index=False)

[ANALYTICS/ML] holdout zoonotik ... mulai
  zoonotik: 6,833  |  latih: 33,023
  akurasi holdout: 0.9814  macro-F1: 0.3302

prediksi_label    babi  unggas
serotype inang                
H10N8    manusia     0       2
         unggas      0      18
H5N1     babi        0       9
         manusia     0      76
         unggas      0    3683
H5N6     unggas      0     191
H5N8     unggas      0     207
H7N7     unggas      0      66
H7N9     manusia     0      31
         unggas      0     185
H9N2     babi        0       5
         manusia     0       3
         unggas      1    2356
[ANALYTICS/ML] holdout zoonotik -> 1.49 menit, mem 34.96 GB, status OK


In [37]:
with Tahap("generalisasi lintas segmen NA", "ANALYTICS/ML"):
    na = (stage_all.filter((F.col("segmen") == "NA") &
                           F.col("inang").isNotNull() &
                           (F.col("inang") != "lain"))
          .withColumn("idx", F.monotonically_increasing_id()))
    if CFG["subsampel"]:
        n0 = na.count()
        if CFG["subsampel"] < n0:
            na = na.sample(False, CFG["subsampel"] / n0, seed=7)
    na = na.cache(); n_na = na.count()
    print(f"  NA berlabel: {n_na:,}")

    na_k = (na.repartition(CFG["shuffle_part"]).withColumn("kmers",
             F.expr(f"filter(transform(sequence(0, length(seq) - {K}), "
                    f"i -> substring(seq, i+1, {K})), x -> x rlike '^[ACGT]+$')"))
            .select("idx","accession","inang","benua","tahun","serotype","kmers")
            .filter(F.size("kmers") > 50))

    fm_na = Pipeline(stages=[
        StringIndexer(inputCol="inang", outputCol="label"),
        CountVectorizer(inputCol="kmers", outputCol="tf", vocabSize=VOCAB, minDF=5.0),
        IDF(inputCol="tf", outputCol="fitur")]).fit(na_k)
    ds_na = fm_na.transform(na_k).select("fitur","label","inang","benua","tahun").cache()
    ds_na.count()
    LABEL_NA = list(fm_na.stages[0].labels)

    cnt_bn = {r["benua"]: r["count"] for r in ds_na.groupBy("benua").count().collect()}
    beban_n, peta_n = [0]*N_FOLD, {}
    for b in sorted(cnt_bn, key=cnt_bn.get, reverse=True):
        f_ = beban_n.index(min(beban_n)); peta_n[b] = f_; beban_n[f_] += cnt_bn[b]
    mbn = F.create_map([F.lit(x) for kv in peta_n.items() for x in kv])
    ds_na = (ds_na.withColumn("fold_acak", (F.rand(seed=SEED)*N_FOLD).cast(IntegerType()))
                  .withColumn("fold_spasial", F.coalesce(mbn[F.col("benua")], F.lit(0))))

    hasil_na = []
    for nama_sk, kol in [("1. Random k-fold","fold_acak"), ("2. Blok spasial","fold_spasial")]:
        for f_ in range(N_FOLD):
            tr = ds_na.filter(F.col(kol) != f_); te = ds_na.filter(F.col(kol) == f_)
            if te.count() == 0 or tr.select("label").distinct().count() < 2:
                continue
            m = LogisticRegression(featuresCol="fitur", labelCol="label",
                                   maxIter=30, regParam=0.01).fit(tr)
            pr = m.transform(te).cache()
            per, _ = metrik_lengkap(pr, len(LABEL_NA), LABEL_NA)
            hasil_na.append({"segmen":"NA","skema":nama_sk,"fold":f_,
                             "macro_f1": round(float(np.mean(list(per.values()))),4),
                             "akurasi": round(ev_acc.evaluate(pr),4)})
            pr.unpersist()
            print(f"  NA [{nama_sk}] f{f_}: macroF1={hasil_na[-1]['macro_f1']}")

    df_na = pd.DataFrame(hasil_na)
    df_na.to_csv(OUT / "lintas_segmen_na.csv", index=False)

if len(df_na):
    print()
    print(df_na.groupby("skema")["macro_f1"].agg(["mean","std"]).round(4).to_string())

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "kerja"), catatan="zoonotik+NA")


[ANALYTICS/ML] generalisasi lintas segmen NA ... mulai
  NA berlabel: 39,868
  NA [1. Random k-fold] f0: macroF1=0.9865
  NA [1. Random k-fold] f1: macroF1=0.9862
  NA [1. Random k-fold] f2: macroF1=0.9811
  NA [1. Random k-fold] f3: macroF1=0.9821
  NA [1. Random k-fold] f4: macroF1=0.9832
  NA [2. Blok spasial] f0: macroF1=0.8944
  NA [2. Blok spasial] f1: macroF1=0.9015
  NA [2. Blok spasial] f2: macroF1=0.8558
  NA [2. Blok spasial] f3: macroF1=0.8206
  NA [2. Blok spasial] f4: macroF1=0.8878
[ANALYTICS/ML] generalisasi lintas segmen NA -> 92.10 menit, mem 41.44 GB, status OK

                    mean     std
skema                           
1. Random k-fold  0.9838  0.0024
2. Blok spasial   0.8720  0.0336


## LAPISAN 14 — ANALYTICS ENGINE G: GRAPH ANALYTICS | Derajat, PageRank, komponen, komunitas, asortativitas

In [ ]:
AMBANG = CFG["ambang_jaccard"]; TOP_K = CFG["lsh_top_k"]
SISI_PATH  = STAGE / f"graf_kemiripan_j{int(AMBANG*100)}_k{TOP_K}.parquet"
SISI2_PATH = STAGE / "sisi_dua_arah.parquet"
VEK_PATH   = STAGE / "vek_biner.parquet"
LP_DIR     = STAGE / "lp_iter"
FOLD_PATH  = STAGE / "fold_lengkap.parquet"

# graf kecil: 64 partisi, bukan 600
mode_graf()

with Tahap("restore graf dari disk", "ANALYTICS/GRAPH"):
    sisi = spark.read.parquet(str(SISI_PATH)).cache()
    n_sisi = sisi.count()

    sisi2 = spark.read.parquet(str(SISI2_PATH)).cache()
    sisi2.count()

    vek_df = spark.read.parquet(str(VEK_PATH))
    n_vek = vek_df.count()

    it_ada = sorted(int(d.name.split("=")[1]) for d in LP_DIR.glob("iter=*")
                    if any(d.rglob("*.parquet")))
    label_kmp = (spark.read.parquet(str(LP_DIR / f"iter={max(it_ada)}"))
                 .select("id", "komp"))
    n_klaster = label_kmp.select("komp").distinct().count()
    RASIO_PSEUDO = n_vek / max(n_klaster, 1)

    if FOLD_PATH.exists() and any(FOLD_PATH.rglob("*.parquet")):
        kerja_dasar = spark.read.parquet(str(FOLD_PATH))
        print("  fold_lengkap.parquet ditemukan")
    else:
        kerja_dasar = None
        print("  !! fold_lengkap.parquet TIDAK ADA -> jalankan ulang LAPISAN 8")

print(f"  simpul   : {n_vek:,}")
print(f"  sisi     : {n_sisi:,}")
print(f"  komponen : {n_klaster:,}  (rasio {RASIO_PSEUDO:.2f}x)")
print(f"  LP iterasi terakhir: {max(it_ada)}")
print("  partisi shuffle -> 64 untuk operasi graf iteratif")

In [38]:
with Tahap("metrik derajat & komponen", "ANALYTICS/GRAPH"):
    derajat = (sisi2.groupBy("src").agg(F.count("*").alias("derajat"))
               .withColumnRenamed("src", "id"))
    atribut = kerja.select(F.col("idx").alias("id"), "accession", "inang",
                           "benua", "tahun", "serotype", "komp")
    simpul_df = (atribut.join(derajat, "id", "left")
                 .fillna({"derajat": 0}).cache())
    simpul_df.count()

    stat_der = simpul_df.select(
        F.avg("derajat").alias("rata"), F.max("derajat").alias("maks"),
        F.expr("percentile_approx(derajat, 0.5)").alias("median")).first()
    print(f"  derajat rata2 {stat_der['rata']:.2f} | median {stat_der['median']} "
          f"| maks {stat_der['maks']}")

    ukuran_komp = (simpul_df.groupBy("komp").agg(
        F.count("*").alias("ukuran"),
        F.countDistinct("inang").alias("n_inang"),
        F.countDistinct("benua").alias("n_benua"),
        F.min("tahun").alias("tahun_min"),
        F.max("tahun").alias("tahun_maks")).cache())
    ukuran_komp.count()

    top_komp = ukuran_komp.orderBy(F.desc("ukuran")).limit(20).toPandas()
    print()
    print(top_komp.to_string(index=False))
    ukuran_komp.toPandas().to_csv(OUT / "graf_komponen.csv", index=False)

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "kerja", "simpul_df"), catatan="derajat")


[ANALYTICS/GRAPH] metrik derajat & komponen ... mulai
  derajat rata2 19.12 | median 7 | maks 338

       komp  ukuran  n_inang  n_benua  tahun_min  tahun_maks
17179869287    2486        2        4       2022        2026
17179869815    1811        2        6       2022        2025
17179880815    1693        2        7          0        2018
17179869304    1071        3        1       2022        2026
17179869184     844        2        5       2025        2026
17179869185     817        2        2       2021        2024
17179870143     806        1        5       2016        2019
17179870365     763        2        5       2022        2025
17179873969     702        1        6       2013        2018
17179872745     674        1        1       2021        2022
17179870605     587        1        3       2024        2025
17179871472     552        1        4       2021        2023
17179870920     517        3        6       2015        2019
17179871460     515        2        3       202

In [39]:
with Tahap("PageRank iteratif di Spark", "ANALYTICS/GRAPH"):
    ITER_PR, DAMP = 12, 0.85
    n_simpul = simpul_df.count()

    keluar = sisi2.groupBy("src").agg(F.count("*").alias("n_keluar"))
    tepi   = sisi2.join(keluar, "src").cache(); tepi.count()

    pr = simpul_df.select(F.col("id"), F.lit(1.0 / n_simpul).alias("pr"))
    for it in range(ITER_PR):
        kontrib = (tepi.join(pr, tepi.src == pr.id)
                   .select(F.col("dst").alias("id"),
                           (F.col("pr") / F.col("n_keluar")).alias("k")))
        agg = kontrib.groupBy("id").agg(F.sum("k").alias("s"))
        pr = (simpul_df.select("id").join(agg, "id", "left").fillna({"s": 0.0})
              .select("id", ((1 - DAMP)/n_simpul + DAMP*F.col("s")).alias("pr"))
              .cache())
        if it % 4 == 0:
            print(f"    iterasi {it+1}/{ITER_PR}")

    pr_join = (pr.join(atribut, "id")
               .orderBy(F.desc("pr")).limit(50).toPandas())
    pr_join.to_csv(OUT / "graf_pagerank_top50.csv", index=False)
    print()
    print(pr_join[["accession","inang","benua","tahun","serotype","pr"]]
          .head(15).to_string(index=False))

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "kerja", "simpul_df"), catatan="pagerank")


[ANALYTICS/GRAPH] PageRank iteratif di Spark ... mulai
    iterasi 1/12
    iterasi 5/12
    iterasi 9/12

 accession   inang         benua  tahun serotype       pr
PP302956.1 manusia Amerika Utara   2023     H3N2 0.000181
PP331956.1 manusia Amerika Utara   2023     H3N2 0.000180
PP303345.1 manusia Amerika Utara   2023     H3N2 0.000179
PP684656.1 manusia Amerika Utara   2024     H3N2 0.000178
PP302067.1 manusia Amerika Utara   2023     H3N2 0.000178
PZ486195.1 manusia Amerika Utara   2026     H3N2 0.000166
PX700617.1 manusia          Asia   2025     H3N2 0.000164
PZ861005.1 manusia          Asia   2026     H3N2 0.000163
PZ236593.1 manusia Amerika Utara   2025     H3N2 0.000163
PZ059198.1 manusia         Eropa   2025     H3N2 0.000162
PX912786.1 manusia Amerika Utara   2025     H3N2 0.000161
PZ111978.1 manusia Amerika Utara   2025     H3N2 0.000160
PZ584022.1 manusia         Eropa   2026     H3N2 0.000160
PX951521.1 manusia Amerika Utara   2025     H3N2 0.000160
PX989224.1 manusia Amer

In [40]:
with Tahap("asortativitas & pencampuran", "ANALYTICS/GRAPH"):
    mix_inang = (sisi.groupBy("src_inang","dst_inang").count().toPandas())
    M = mix_inang.pivot(index="src_inang", columns="dst_inang", values="count").fillna(0)
    M = M.reindex(index=LABEL_NAMA, columns=LABEL_NAMA, fill_value=0)
    Msym = M.values + M.values.T
    np.fill_diagonal(Msym, np.diag(M.values + M.values.T) / 2)
    e = Msym / max(Msym.sum(), 1)
    a = e.sum(axis=1); b_ = e.sum(axis=0)
    r_inang = (np.trace(e) - float(a @ b_)) / max(1 - float(a @ b_), 1e-12)

    mix_benua = sisi.groupBy("src_benua","dst_benua").count().toPandas()
    Bn = sorted(set(mix_benua["src_benua"]) | set(mix_benua["dst_benua"]))
    MB = (mix_benua.pivot(index="src_benua", columns="dst_benua", values="count")
          .reindex(index=Bn, columns=Bn, fill_value=0).fillna(0).values)
    MBs = MB + MB.T
    eb = MBs / max(MBs.sum(), 1)
    ab = eb.sum(axis=1); bb = eb.sum(axis=0)
    r_benua = (np.trace(eb) - float(ab @ bb)) / max(1 - float(ab @ bb), 1e-12)

    n_lintas_inang = int(sisi.filter(F.col("src_inang") != F.col("dst_inang")).count())
    n_lintas_benua = int(sisi.filter(F.col("src_benua") != F.col("dst_benua")).count())

GRAF_STAT = {
    "simpul": int(n_vek), "sisi": int(n_sisi),
    "komponen": int(n_klaster),
    "rasio_pseudoreplikasi": round(RASIO_PSEUDO, 3),
    "derajat_rata2": round(float(stat_der["rata"]), 3),
    "derajat_maks": int(stat_der["maks"]),
    "asortativitas_inang": round(float(r_inang), 4),
    "asortativitas_benua": round(float(r_benua), 4),
    "sisi_lintas_inang": n_lintas_inang,
    "sisi_lintas_inang_pct": round(100 * n_lintas_inang / max(int(n_sisi), 1), 2),
    "sisi_lintas_benua": n_lintas_benua,
    "sisi_lintas_benua_pct": round(100 * n_lintas_benua / max(int(n_sisi), 1), 2),
}
json.dump(GRAF_STAT, open(OUT / "graf_statistik.json", "w"), indent=2)
M.to_csv(OUT / "graf_matriks_pencampuran_inang.csv")
print(json.dumps(GRAF_STAT, indent=2))
print()
print("Matriks pencampuran inang (jumlah sisi):")
print(M.to_string())
audit("graf_statistik", **GRAF_STAT)

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "kerja", "simpul_df"), catatan="asortativitas")


[ANALYTICS/GRAPH] asortativitas & pencampuran ... mulai
[ANALYTICS/GRAPH] asortativitas & pencampuran -> 0.01 menit, mem 44.91 GB, status OK
{
  "simpul": 39856,
  "sisi": 381004,
  "komponen": 12554,
  "rasio_pseudoreplikasi": 3.175,
  "derajat_rata2": 19.119,
  "derajat_maks": 338,
  "asortativitas_inang": 0.9576,
  "asortativitas_benua": 0.5598,
  "sisi_lintas_inang": 2153,
  "sisi_lintas_inang_pct": 0.57,
  "sisi_lintas_benua": 50754,
  "sisi_lintas_benua_pct": 13.32
}

Matriks pencampuran inang (jumlah sisi):
dst_inang  manusia  unggas  babi
src_inang                       
manusia     322275      87   798
unggas         210   51284    18
babi          1002      38  5292


{'ts': '2026-08-27T14:48:11.436541+00:00',
 'run_id': 'run_20260827T052524Z',
 'peristiwa': 'graf_statistik',
 'aktor': 'Willy Boen',
 'simpul': 39856,
 'sisi': 381004,
 'komponen': 12554,
 'rasio_pseudoreplikasi': 3.175,
 'derajat_rata2': 19.119,
 'derajat_maks': 338,
 'asortativitas_inang': 0.9576,
 'asortativitas_benua': 0.5598,
 'sisi_lintas_inang': 2153,
 'sisi_lintas_inang_pct': 0.57,
 'sisi_lintas_benua': 50754,
 'sisi_lintas_benua_pct': 13.32}

# G1 — SKEMA PROPERTY GRAPH | vertices + edges bertipe

In [41]:
with Tahap("skema property graph", "ANALYTICS/GRAPH"):
    # --- VERTICES: satu simpul = satu isolat virus ---
    vertices = (kerja.select(
        F.col("idx").alias("id"),
        F.lit("Isolat").alias("label_simpul"),
        F.col("accession").alias("prop_accession"),
        F.col("inang").alias("prop_inang"),
        F.col("benua").alias("prop_benua"),
        F.col("negara").alias("prop_negara"),
        F.col("tahun").alias("prop_tahun"),
        F.col("serotype").alias("prop_serotype"),
        F.col("komp").alias("prop_komponen"))
        .cache())
    n_v = vertices.count()

    # --- EDGES: relasi MIRIP_DENGAN, berbobot Jaccard ---
    edges = (sisi.select(
        F.col("src"), F.col("dst"),
        F.lit("MIRIP_DENGAN").alias("tipe_relasi"),
        F.col("kemiripan").alias("bobot"),
        (F.col("src_inang") != F.col("dst_inang")).alias("lintas_inang"),
        (F.col("src_benua") != F.col("dst_benua")).alias("lintas_benua"),
        F.abs(F.col("src_tahun") - F.col("dst_tahun")).alias("selisih_tahun"))
        .cache())
    n_e = edges.count()

    vertices.write.mode("overwrite").parquet(str(MART / "graph_vertices"))
    edges.write.mode("overwrite").parquet(str(MART / "graph_edges"))

    # --- GraphFrames opsional: dipakai bila paket JVM tersedia ---
    GF = None
    try:
        from graphframes import GraphFrame
        GF = GraphFrame(vertices, edges.withColumnRenamed("bobot", "weight"))
        print("  GraphFrames aktif")
    except Exception:
        print("  GraphFrames tidak tersedia -> memakai implementasi Spark SQL native")

SKEMA_GRAF = pd.DataFrame([
    {"elemen": "Vertex", "label": "Isolat", "jumlah": n_v,
     "properti": "accession, inang, benua, negara, tahun, serotype, komponen"},
    {"elemen": "Edge", "label": "MIRIP_DENGAN", "jumlah": n_e,
     "properti": "bobot (Jaccard), lintas_inang, lintas_benua, selisih_tahun"},
])
SKEMA_GRAF.to_csv(OUT / "graf_skema.csv", index=False)
print()
print(SKEMA_GRAF.to_string(index=False))
print(f"\n  tipe graf : tak berarah, berbobot, sederhana")
print(f"  densitas  : {n_e / max(n_v*(n_v-1)/2, 1):.3e}")

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "kerja", "simpul_df", "vertices", "edges"), catatan="G1")


[ANALYTICS/GRAPH] skema property graph ... mulai
  GraphFrames tidak tersedia -> memakai implementasi Spark SQL native
[ANALYTICS/GRAPH] skema property graph -> 0.05 menit, mem 46.04 GB, status OK

elemen        label  jumlah                                                   properti
Vertex       Isolat   39856 accession, inang, benua, negara, tahun, serotype, komponen
  Edge MIRIP_DENGAN  381004 bobot (Jaccard), lintas_inang, lintas_benua, selisih_tahun

  tipe graf : tak berarah, berbobot, sederhana
  densitas  : 4.797e-04


# G2 — CENTRALITY | degree, eigenvector, PageRank, closeness, betweenness

In [ ]:
with Tahap("centrality analytics", "ANALYTICS/GRAPH"):
    CENT_PATH = STAGE / "centrality.parquet"
    EV_DIR    = STAGE / "ev_iter"
    EV_DIR.mkdir(parents=True, exist_ok=True)

    if CENT_PATH.exists() and any(CENT_PATH.rglob("*.parquet")):
        pusat_pdf = spark.read.parquet(str(CENT_PATH)).toPandas()
        K_PIVOT = 500
        print("  checkpoint centrality ditemukan, baca ulang")
    else:
        # ---------- 1. degree ----------
        deg = (sisi2.groupBy("src").agg(F.count("*").alias("derajat"))
               .withColumnRenamed("src", "id"))
        sent = (vertices.join(deg, "id", "left").fillna({"derajat": 0})
                .withColumn("deg_centrality",
                            F.col("derajat") / F.lit(max(n_v - 1, 1)))
                .repartition(64, "id").cache())
        sent.count()

        # ---------- 2. eigenvector: power iteration, lineage dipotong ----------
        ada_ev = sorted(int(d.name.split("=")[1]) for d in EV_DIR.glob("it=*")
                        if any(d.rglob("*.parquet")))
        if ada_ev:
            ev = spark.read.parquet(str(EV_DIR / f"it={max(ada_ev)}"))
            mulai_ev = max(ada_ev) + 1
            print(f"  eigenvector lanjut dari iterasi {mulai_ev}")
        else:
            ev = sent.select("id", F.lit(1.0).alias("ev"))
            mulai_ev = 0

        for it in range(mulai_ev, 15):
            agg = (sisi2.join(ev, sisi2.src == ev.id)
                   .groupBy("dst").agg(F.sum("ev").alias("s")))
            norm = agg.agg(F.sqrt(F.sum(F.col("s") * F.col("s")))
                           .alias("n")).first()["n"]
            norm = norm if norm and norm > 0 else 1.0

            tujuan = EV_DIR / f"it={it}"
            (sent.select("id").join(agg, sent.id == agg.dst, "left")
             .fillna({"s": 0.0})
             .select("id", (F.col("s") / F.lit(norm)).alias("ev"))
             .repartition(64, "id")
             .write.mode("overwrite").parquet(str(tujuan)))
            ev = spark.read.parquet(str(tujuan))     # lineage terpotong
            if it % 5 == 0:
                print(f"    eigenvector iterasi {it+1}/15")

        # ---------- 3. gabung dengan PageRank ----------
        pusat = (sent.join(ev, "id")
                 .join(pr.withColumnRenamed("pr", "pagerank"), "id"))

        # ---------- 4. closeness & betweenness di komponen terbesar ----------
        import networkx as nx
        kb = (sent.groupBy("prop_komponen").count()
              .orderBy(F.desc("count")).first())
        kid, ksize = kb["prop_komponen"], kb["count"]
        print(f"  komponen terbesar: {ksize:,} simpul")

        BATAS_NX = 15000
        id_komp = [r["id"] for r in
                   sent.filter(F.col("prop_komponen") == kid).select("id").collect()]
        if len(id_komp) > BATAS_NX:
            id_komp = list(np.random.default_rng(SEED)
                           .choice(id_komp, BATAS_NX, replace=False))
        set_id = set(int(x) for x in id_komp)

        e_sub = (sisi.select("src", "dst", "kemiripan").toPandas())
        e_sub = e_sub[e_sub["src"].isin(set_id) & e_sub["dst"].isin(set_id)]

        G = nx.Graph()
        G.add_nodes_from(id_komp)
        G.add_weighted_edges_from(e_sub.itertuples(index=False, name=None))
        print(f"  subgraf NetworkX: {G.number_of_nodes():,} simpul, "
              f"{G.number_of_edges():,} sisi")

        K_PIVOT = min(300, max(G.number_of_nodes() // 20, 50))
        btw = nx.betweenness_centrality(G, k=K_PIVOT, seed=SEED, normalized=True)
        clo = {}
        pv = list(np.random.default_rng(SEED).choice(
            list(G.nodes()), min(800, G.number_of_nodes()), replace=False))
        for nd in pv:
            sp_ = nx.single_source_shortest_path_length(G, nd)
            clo[nd] = (len(sp_) - 1) / max(sum(sp_.values()), 1)

        pusat_pdf = (pusat.select("id", "prop_accession", "prop_inang",
                                  "prop_benua", "prop_tahun", "prop_serotype",
                                  "derajat", "deg_centrality", "ev",
                                  "pagerank").toPandas()
                     .merge(pd.DataFrame({"id": list(btw.keys()),
                                          "betweenness": list(btw.values())}),
                            on="id", how="left")
                     .merge(pd.DataFrame({"id": list(clo.keys()),
                                          "closeness": list(clo.values())}),
                            on="id", how="left"))
        spark.createDataFrame(pusat_pdf.fillna(0)).write.mode("overwrite") \
             .parquet(str(CENT_PATH))

    pusat_pdf.to_csv(OUT / "graf_centrality.csv", index=False)

print()
print("Top 15 PageRank:")
print(pusat_pdf.nlargest(15, "pagerank")[
    ["prop_accession", "prop_inang", "prop_benua", "prop_tahun",
     "prop_serotype", "derajat", "pagerank"]].to_string(index=False))
print()
print("Top 10 betweenness (jembatan antar-lineage):")
print(pusat_pdf.nlargest(10, "betweenness")[
    ["prop_accession", "prop_inang", "prop_benua", "prop_serotype",
     "betweenness"]].to_string(index=False))
print()
print("Korelasi antar ukuran sentralitas:")
print(pusat_pdf[["deg_centrality", "ev", "pagerank",
                 "betweenness", "closeness"]].corr().round(3).to_string())

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "vertices", "edges", "sent"), catatan="G2")


[ANALYTICS/GRAPH] centrality analytics ... mulai
    eigenvector iterasi 1/15
    eigenvector iterasi 6/15
    eigenvector iterasi 11/15
[ANALYTICS/GRAPH] centrality analytics -> 12.79 menit, mem 47.35 GB, status GAGAL


Py4JJavaError: An error occurred while calling o37141.collectToPython.
: java.lang.IllegalStateException: SparkContext has been shutdown
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2385)
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2414)
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2433)
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2458)
	at org.apache.spark.rdd.RDD.$anonfun$collect$1(RDD.scala:1049)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:112)
	at org.apache.spark.rdd.RDD.withScope(RDD.scala:410)
	at org.apache.spark.rdd.RDD.collect(RDD.scala:1048)
	at org.apache.spark.sql.execution.SparkPlan.executeCollectIterator(SparkPlan.scala:455)
	at org.apache.spark.sql.execution.exchange.BroadcastExchangeExec.$anonfun$relationFuture$1(BroadcastExchangeExec.scala:140)
	at org.apache.spark.sql.execution.S

# G3 — COMMUNITY & LOCAL PROPERTIES | LPA, modularitas, triangle, clustering coef

In [43]:
with Tahap("community analytics", "ANALYTICS/GRAPH"):
    # ---------- 1. Label Propagation Algorithm (komunitas, Spark) ----------
    #    Berbeda dari connected components: LPA memecah komponen raksasa
    #    menjadi komunitas padat berdasarkan label mayoritas tetangga.
    komunitas = sent.select("id", F.col("id").alias("kom"))
    ITER_LPA = 10
    for it in range(ITER_LPA):
        suara = (sisi2.join(komunitas, sisi2.src == komunitas.id)
                 .groupBy("dst", "kom").agg(F.count("*").alias("n")))
        w = Window.partitionBy("dst").orderBy(F.desc("n"), F.asc("kom"))
        menang = (suara.withColumn("rk", F.row_number().over(w))
                  .filter(F.col("rk") == 1)
                  .select(F.col("dst").alias("id"), F.col("kom").alias("kom_baru")))
        komunitas = (sent.select("id").join(menang, "id", "left")
                     .select("id", F.coalesce(F.col("kom_baru"),
                                              F.col("id")).alias("kom")).cache())
        if it % 3 == 0:
            print(f"    LPA iterasi {it+1}/{ITER_LPA}")

    n_kom = komunitas.select("kom").distinct().count()
    print(f"  komunitas LPA   : {n_kom:,}")
    print(f"  komponen (CC)   : {n_klaster:,}")

    # ---------- 2. Modularitas Newman ----------
    km = komunitas.cache(); km.count()
    e_kom = (sisi.join(km.withColumnRenamed("id", "src")
                         .withColumnRenamed("kom", "kom_src"), "src")
                 .join(km.withColumnRenamed("id", "dst")
                         .withColumnRenamed("kom", "kom_dst"), "dst"))
    m2 = 2.0 * n_e
    intra = e_kom.filter(F.col("kom_src") == F.col("kom_dst")) \
                 .groupBy(F.col("kom_src").alias("kom")).agg(F.count("*").alias("e_in"))
    dsum = (sent.select("id", "derajat").join(km, "id")
            .groupBy("kom").agg(F.sum("derajat").alias("d_tot")))
    mod_pdf = (dsum.join(intra, "kom", "left").fillna({"e_in": 0}).toPandas())
    Q = float((( mod_pdf["e_in"] / max(n_e, 1))
               - (mod_pdf["d_tot"] / max(m2, 1)) ** 2).sum())
    print(f"  modularitas Q   : {Q:.4f}")

    # ---------- 3. Triangle count & local clustering coefficient (Spark) ----------
    e_kanon = sisi.select("src", "dst")
    tri = (e_kanon.alias("a")
           .join(e_kanon.alias("b"), F.col("a.dst") == F.col("b.src"))
           .join(e_kanon.alias("c"),
                 (F.col("c.src") == F.col("a.src")) &
                 (F.col("c.dst") == F.col("b.dst")))
           .select(F.col("a.src").alias("v1"), F.col("a.dst").alias("v2"),
                   F.col("b.dst").alias("v3")).cache())
    n_tri = tri.count()

    tri_per_simpul = (tri.select(F.col("v1").alias("id"))
                      .union(tri.select(F.col("v2").alias("id")))
                      .union(tri.select(F.col("v3").alias("id")))
                      .groupBy("id").agg(F.count("*").alias("n_segitiga")))

    lokal = (sent.select("id", "derajat", "prop_inang", "prop_benua")
             .join(tri_per_simpul, "id", "left").fillna({"n_segitiga": 0})
             .withColumn("clustering_coef",
                 F.when(F.col("derajat") > 1,
                        2.0 * F.col("n_segitiga") /
                        (F.col("derajat") * (F.col("derajat") - 1)))
                  .otherwise(0.0)).cache())
    cc_global = lokal.agg(F.avg("clustering_coef").alias("c")).first()["c"]

    # ---------- 4. Profil komunitas ----------
    prof = (km.join(sent.select("id", "prop_inang", "prop_benua",
                                "prop_tahun", "prop_serotype"), "id")
            .groupBy("kom").agg(
                F.count("*").alias("ukuran"),
                F.countDistinct("prop_inang").alias("n_inang"),
                F.countDistinct("prop_benua").alias("n_benua"),
                F.countDistinct("prop_serotype").alias("n_serotype"),
                F.min("prop_tahun").alias("tahun_min"),
                F.max("prop_tahun").alias("tahun_maks"))
            .filter(F.col("ukuran") >= 5).cache())
    prof_pdf = prof.orderBy(F.desc("ukuran")).limit(30).toPandas()
    prof.toPandas().to_csv(OUT / "graf_komunitas.csv", index=False)
    lokal.select("id", "derajat", "n_segitiga", "clustering_coef",
                 "prop_inang", "prop_benua").toPandas() \
         .to_csv(OUT / "graf_local_properties.csv", index=False)

    n_kom_murni = int((prof_pdf["n_inang"] == 1).sum())

KOM_STAT = {
    "komunitas_LPA": int(n_kom),
    "komponen_CC": int(n_klaster),
    "modularitas_Q": round(Q, 4),
    "segitiga": int(n_tri),
    "clustering_coef_rata2": round(float(cc_global), 4),
    "komunitas_besar_ge5": int(prof.count()),
    "komunitas_murni_1_inang_pct": round(100 * n_kom_murni / max(len(prof_pdf), 1), 1),
}
json.dump(KOM_STAT, open(OUT / "graf_komunitas_stat.json", "w"), indent=2)
print()
print(json.dumps(KOM_STAT, indent=2))
print()
print("30 komunitas terbesar:")
print(prof_pdf.to_string(index=False))

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "vertices", "edges", "sent"), catatan="G3")


[ANALYTICS/GRAPH] community analytics ... mulai
[ANALYTICS/GRAPH] community analytics -> 0.00 menit, mem 47.81 GB, status GAGAL


Py4JJavaError: An error occurred while calling o37203.cache.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.

# G4 — CONNECTIVITY | k-core, distribusi komponen, ketahanan graf

In [ ]:
with Tahap("connectivity analytics", "ANALYTICS/GRAPH"):
    # ---------- 1. k-core decomposition (Spark iteratif) ----------
    hasil_kcore = []
    aktif = sent.select("id")
    sisi_aktif = sisi2.select("src", "dst")
    k = 1
    while True:
        while True:
            d = sisi_aktif.groupBy("src").agg(F.count("*").alias("d")) \
                          .withColumnRenamed("src", "id")
            tetap = aktif.join(d, "id").filter(F.col("d") >= k).select("id").cache()
            n_tetap = tetap.count()
            if n_tetap == aktif.count() or n_tetap == 0:
                aktif = tetap
                break
            aktif = tetap
            sisi_aktif = (sisi_aktif
                          .join(aktif.withColumnRenamed("id", "src"), "src")
                          .join(aktif.withColumnRenamed("id", "dst"), "dst")
                          .select("src", "dst").cache())
        n_core = aktif.count()
        hasil_kcore.append({"k": k, "simpul_dalam_k_core": n_core,
                            "pct": round(100 * n_core / max(n_v, 1), 2)})
        print(f"    {k}-core: {n_core:,} simpul ({100*n_core/max(n_v,1):.1f}%)")
        if n_core == 0 or k > 40:
            break
        k += 1
    df_kcore = pd.DataFrame(hasil_kcore)
    df_kcore.to_csv(OUT / "graf_kcore.csv", index=False)
    k_maks = int(df_kcore[df_kcore["simpul_dalam_k_core"] > 0]["k"].max())

    # ---------- 2. Distribusi komponen ----------
    dist_komp = (sent.groupBy("prop_komponen").count()
                 .groupBy("count").agg(F.count("*").alias("frekuensi"))
                 .orderBy("count").toPandas())
    dist_komp.columns = ["ukuran_komponen", "frekuensi"]
    dist_komp.to_csv(OUT / "graf_distribusi_komponen.csv", index=False)

    komp_pdf = sent.groupBy("prop_komponen").count().toPandas()
    ukuran_maks = int(komp_pdf["count"].max())
    isolat_tunggal = int((komp_pdf["count"] == 1).sum())

    # ---------- 3. Ketahanan: hapus simpul paling sentral ----------
    urut_pr = pusat_pdf.nlargest(int(0.01 * len(pusat_pdf)) or 1, "pagerank")["id"]
    buang = set(int(x) for x in urut_pr)
    sisa = e_sub[~e_sub["src"].isin(buang) & ~e_sub["dst"].isin(buang)]
    G2 = nx.Graph()
    G2.add_nodes_from([x for x in id_komp if x not in buang])
    G2.add_weighted_edges_from(sisa.itertuples(index=False, name=None))
    komp_sebelum = G.number_of_nodes()
    komp_sesudah = (len(max(nx.connected_components(G2), key=len))
                    if G2.number_of_nodes() else 0)

KONEKTIVITAS = {
    "komponen_total": int(n_klaster),
    "komponen_terbesar": ukuran_maks,
    "komponen_terbesar_pct": round(100 * ukuran_maks / max(n_v, 1), 2),
    "simpul_terisolasi": isolat_tunggal,
    "k_core_maksimum": k_maks,
    "raksasa_sebelum_serangan": int(komp_sebelum),
    "raksasa_setelah_hapus_1pct_hub": int(komp_sesudah),
    "penurunan_konektivitas_pct": round(
        100 * (komp_sebelum - komp_sesudah) / max(komp_sebelum, 1), 2),
}
json.dump(KONEKTIVITAS, open(OUT / "graf_konektivitas.json", "w"), indent=2)
print()
print(json.dumps(KONEKTIVITAS, indent=2))

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "vertices", "edges", "sent"), catatan="G4")


# G5 — PATH ANALYTICS | BFS multi-sumber, distribusi jarak, diameter, jalur zoonotik

In [ ]:
with Tahap("path analytics", "ANALYTICS/GRAPH"):
    # ---------- 1. BFS multi-sumber di Spark (skalabel) ----------
    N_SUMBER = 200
    sumber = [r["id"] for r in
              sent.orderBy(F.desc("derajat")).select("id").limit(N_SUMBER).collect()]
    jarak = (sent.select("id")
             .withColumn("jarak", F.when(F.col("id").isin(sumber), F.lit(0))
                                   .otherwise(F.lit(None).cast("int"))).cache())
    MAKS_HOP = 10
    for hop in range(1, MAKS_HOP + 1):
        depan = jarak.filter(F.col("jarak") == hop - 1).select("id")
        if depan.count() == 0:
            break
        baru = (sisi2.join(depan, sisi2.src == depan.id)
                .select(F.col("dst").alias("id")).distinct())
        jarak = (jarak.join(baru.withColumn("cap", F.lit(hop)), "id", "left")
                 .select("id", F.coalesce(F.col("jarak"), F.col("cap")).alias("jarak"))
                 .cache())
        n_tercapai = jarak.filter(F.col("jarak").isNotNull()).count()
        print(f"    hop {hop}: {n_tercapai:,} simpul terjangkau "
              f"({100*n_tercapai/max(n_v,1):.1f}%)")

    dist_jarak = (jarak.filter(F.col("jarak").isNotNull())
                  .groupBy("jarak").count().orderBy("jarak").toPandas())
    dist_jarak.to_csv(OUT / "graf_distribusi_jarak.csv", index=False)

    # ---------- 2. Average path length & diameter (komponen raksasa) ----------
    n_pivot = min(300, G.number_of_nodes())
    pv = list(np.random.default_rng(SEED).choice(
        list(G.nodes()), n_pivot, replace=False))
    panjang, ecc = [], []
    for nd in pv:
        sp = nx.single_source_shortest_path_length(G, nd)
        v = [x for x in sp.values() if x > 0]
        if v:
            panjang.extend(v)
            ecc.append(max(v))
    apl = float(np.mean(panjang)) if panjang else float("nan")
    diameter_est = int(max(ecc)) if ecc else 0

    # ---------- 3. Jalur lintas inang: rute penularan potensial ----------
    jalur_zoo = []
    unggas = [r["id"] for r in sent.filter(F.col("prop_inang") == "unggas")
              .select("id").limit(4000).collect()]
    manusia = set(r["id"] for r in sent.filter(F.col("prop_inang") == "manusia")
                  .select("id").limit(4000).collect())
    u_in_G = [x for x in unggas if x in G]
    m_in_G = manusia & set(G.nodes())
    for s in u_in_G[:150]:
        try:
            sp = nx.single_source_shortest_path_length(G, s, cutoff=6)
        except Exception:
            continue
        tercapai = {k_: v_ for k_, v_ in sp.items() if k_ in m_in_G}
        if tercapai:
            t = min(tercapai, key=tercapai.get)
            jalur_zoo.append({"sumber_unggas": s, "target_manusia": t,
                              "panjang_jalur": tercapai[t]})
    df_jalur = pd.DataFrame(jalur_zoo)
    if len(df_jalur):
        df_jalur.to_csv(OUT / "graf_jalur_zoonotik.csv", index=False)

PATH_STAT = {
    "sumber_BFS": N_SUMBER,
    "hop_maksimum_tercapai": int(dist_jarak["jarak"].max()) if len(dist_jarak) else 0,
    "cakupan_BFS_pct": round(100 * float(dist_jarak["count"].sum()) / max(n_v, 1), 2),
    "average_path_length": round(apl, 3) if apl == apl else None,
    "diameter_estimasi": diameter_est,
    "pasangan_unggas_manusia_terhubung": int(len(df_jalur)),
    "jalur_terpendek_unggas_ke_manusia": (int(df_jalur["panjang_jalur"].min())
                                          if len(df_jalur) else None),
    "jalur_rata2_unggas_ke_manusia": (round(float(df_jalur["panjang_jalur"].mean()), 2)
                                      if len(df_jalur) else None),
}
json.dump(PATH_STAT, open(OUT / "graf_path_stat.json", "w"), indent=2)
print()
print(json.dumps(PATH_STAT, indent=2))
print()
print("Distribusi jarak dari 200 hub:")
print(dist_jarak.to_string(index=False))

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(kecuali=("sisi", "sisi2", "vertices", "edges", "sent"), catatan="G5")


# G6 — SINTESIS GRAPH ANALYTICS + INTERPRETASI

In [ ]:
GRAF_LENGKAP = {**GRAF_STAT, **KOM_STAT, **KONEKTIVITAS, **PATH_STAT}
json.dump(GRAF_LENGKAP, open(OUT / "graf_lengkap.json", "w"), indent=2, default=str)

PETA_SUBTOPIK = pd.DataFrame([
 ("What is Graph?", "Property graph tak berarah berbobot: Isolat -[MIRIP_DENGAN]-> Isolat",
  f"{n_v:,} vertex, {n_e:,} edge", "SEL G1"),
 ("Centrality Analytics", "Degree, eigenvector, PageRank (Spark); closeness & betweenness (sampel)",
  f"PageRank 12 iterasi, betweenness k={K_PIVOT}", "SEL G2"),
 ("Community Analytics", "Label Propagation + modularitas Newman",
  f"{n_kom:,} komunitas, Q={Q:.4f}", "SEL G3"),
 ("Local Properties", "Triangle count + local clustering coefficient",
  f"{n_tri:,} segitiga, C={cc_global:.4f}", "SEL G3"),
 ("Connectivity Analytics", "Connected components, k-core, uji ketahanan",
  f"{n_klaster:,} komponen, k-core maks {k_maks}", "SEL G4"),
 ("Path Analytics", "BFS multi-sumber, average path length, diameter, jalur zoonotik",
  f"APL={apl:.2f}, diameter~{diameter_est}", "SEL G5"),
], columns=["Subtopik", "Metode", "Hasil", "Lokasi"])
PETA_SUBTOPIK.to_csv(OUT / "graf_peta_subtopik.csv", index=False)

print("=" * 100)
print(" PEMETAAN KE SUBTOPIK GRAPH ANALYTICS")
print("=" * 100)
print(PETA_SUBTOPIK.to_string(index=False))
print("=" * 100)
print()
print("INTERPRETASI:")
print(f"  1. Modularitas Q={Q:.3f}. Nilai > 0,3 berarti graf punya struktur komunitas")
print(f"     nyata, bukan acak. Komunitas ini adalah lineage virus.")
print(f"  2. Asortativitas inang={GRAF_STAT['asortativitas_inang']:.3f}. Positif berarti")
print(f"     isolat cenderung mirip dengan isolat berinang sama -> sinyal inang")
print(f"     memang tertanam dalam struktur graf, bukan artefak.")
print(f"  3. Asortativitas benua={GRAF_STAT['asortativitas_benua']:.3f}. Bandingkan dengan")
print(f"     angka inang: mana yang lebih kuat menjelaskan struktur kemiripan?")
print(f"     Inilah bukti graf untuk pertanyaan penelitian utama.")
print(f"  4. Rasio pseudoreplikasi {RASIO_PSEUDO:.2f}x. Graf membuktikan bahwa")
print(f"     {n_v:,} sekuens hanya setara ~{n_klaster:,} unit independen ->")
print(f"     justifikasi kuantitatif untuk skema fold_cluster.")
print(f"  5. Simpul betweenness tertinggi = isolat jembatan antar-lineage,")
print(f"     kandidat prioritas surveillance.")
print(f"  6. Menghapus 1% hub menurunkan konektivitas komponen raksasa sebesar")
print(f"     {KONEKTIVITAS['penurunan_konektivitas_pct']}% -> graf scale-free,")
print(f"     rapuh terhadap serangan tertarget.")

## LAPISAN 15 — DATA WAREHOUSE LAYER | Tabel mart (star schema) untuk konsumsi analitik dan dashboard

In [ ]:
with Tahap("bangun mart", "WAREHOUSE"):
    # --- dim_sekuens ---
    dim_sekuens = kerja.select("idx","accession","inang","benua","negara",
                               "tahun","serotype","komp", *FOLDS)
    dim_sekuens.write.mode("overwrite").partitionBy("benua") \
        .parquet(str(MART / "dim_sekuens"))

    # --- fakta agregat geografi x inang x tahun ---
    fakta_geo = (kerja.groupBy("benua","negara","inang","tahun")
                 .agg(F.count("*").alias("jumlah"),
                      F.countDistinct("serotype").alias("n_serotipe"),
                      F.countDistinct("komp").alias("n_klaster")))
    fakta_geo.write.mode("overwrite").parquet(str(MART / "fakta_geo"))

    # --- fakta model ---
    fakta_model = spark.createDataFrame(per_fold.fillna(0))
    fakta_model.write.mode("overwrite").parquet(str(MART / "fakta_model"))

    # --- fakta graf ---
    fakta_graf = simpul_df.select("id","accession","inang","benua","tahun",
                                  "serotype","komp","derajat")
    fakta_graf.write.mode("overwrite").parquet(str(MART / "fakta_graf"))

    # --- ekspor CSV untuk lapisan visualisasi ---
    geo_pdf = fakta_geo.toPandas()
    geo_pdf.to_csv(MART / "fakta_geo.csv", index=False)
    per_fold.to_csv(MART / "fakta_model.csv", index=False)
    prof_pdf.to_csv(MART / "fakta_komunitas.csv", index=False)
    pusat_pdf.to_csv(MART / "fakta_centrality.csv", index=False)
    dist_jarak.to_csv(MART / "fakta_jarak.csv", index=False)
    df_kcore.to_csv(MART / "fakta_kcore.csv", index=False)
    uk_komp_pdf = ukuran_komp.toPandas()
    uk_komp_pdf.to_csv(MART / "fakta_komponen.csv", index=False)
    simpul_pdf  = simpul_df.select("accession","inang","benua","tahun",
                                   "serotype","komp","derajat").toPandas()
    simpul_pdf.to_csv(MART / "fakta_graf.csv", index=False)
    SPARK_VERSI = spark.version

    for nama, p in [("dim_sekuens", MART/"dim_sekuens"),
                    ("fakta_geo", MART/"fakta_geo"),
                    ("fakta_model", MART/"fakta_model"),
                    ("fakta_graf", MART/"fakta_graf")]:
        mb = sum(f.stat().st_size for f in p.rglob("*.parquet")) / 1e6
        nf = sum(1 for _ in p.rglob("*.parquet"))
        print(f"  {nama:<14} {mb:>8.1f} MB  {nf:>4} file")
    audit("mart_dibangun", tabel=4)

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(tegas=True, catatan="mart -- semua cache dilepas")
mode_data()


## LAPISAN 16 — MONITORING: BENCHMARK SKALABILITAS | Spark local[1..16] vs single-node

In [ ]:
PARALEL = [1, 2, 4, 8, 16]
N_BENCH = min(40000, N)

with Tahap("benchmark skalabilitas", "MONITORING"):
    dasar_df = (spark.read.parquet(str(STAGE / "flua_seq_bersih.parquet"))
                .filter((F.col("segmen") == "HA") & (F.col("inang") != "lain") &
                        F.col("inang").isNotNull())
                .limit(N_BENCH).cache())
    dasar_df.count()

    bench = []
    for c in PARALEL:
        try:
            spark.conf.set("spark.sql.shuffle.partitions", str(c * 8))
            spark.conf.set("spark.default.parallelism", str(c))
            t0 = time.time()
            d = (dasar_df.repartition(c)
                 .withColumn("kmers",
                     F.expr(f"filter(transform(sequence(0, length(seq) - {K}), "
                            f"i -> substring(seq, i+1, {K})), x -> x rlike '^[ACGT]+$')")))
            m = Pipeline(stages=[
                StringIndexer(inputCol="inang", outputCol="label"),
                CountVectorizer(inputCol="kmers", outputCol="tf",
                                vocabSize=VOCAB, minDF=5.0),
                IDF(inputCol="tf", outputCol="fitur"),
                LogisticRegression(featuresCol="fitur", labelCol="label",
                                   maxIter=20, regParam=0.01)]).fit(d)
            _ = m.transform(d).select(F.max("prediction")).first()
            dt = time.time() - t0
            bench.append({"n_core": c, "n_partisi": c, "detik": round(dt, 2),
                          "menit": round(dt/60, 3), "status": "OK"})
            print(f"  partisi={c:>2} : {dt/60:.2f} menit")
        except Exception as ex:
            bench.append({"n_core": c, "n_partisi": c, "detik": np.nan,
                          "menit": np.nan, "status": type(ex).__name__})
            print(f"  partisi={c:>2} : GAGAL {type(ex).__name__}")

    spark.conf.set("spark.sql.shuffle.partitions", str(CFG["shuffle_part"]))
    dasar_df.unpersist()

    df_bench = pd.DataFrame(bench)
    dsr = df_bench.loc[df_bench["n_core"] == 1, "detik"]
    t1 = float(dsr.iloc[0]) if len(dsr) and dsr.iloc[0] == dsr.iloc[0] else np.nan
    df_bench["speedup"]   = (t1 / df_bench["detik"]).round(3)
    df_bench["efisiensi"] = (df_bench["speedup"] / df_bench["n_core"]).round(3)
    df_bench["amdahl_serial"] = df_bench.apply(
        lambda r: round((1/r["speedup"] - 1/r["n_core"]) / (1 - 1/r["n_core"]), 4)
        if r["n_core"] > 1 and r["speedup"] == r["speedup"] else np.nan, axis=1)
    df_bench.to_csv(OUT / "benchmark_skalabilitas.csv", index=False)
    audit("benchmark", hasil=df_bench.to_dict("records"))

SPARK_VERSI = spark.version
print()
print(df_bench.to_string(index=False))

# --- pembersihan memori agar tahap berikutnya tidak gagal ---
bersihkan(tegas=True, catatan="benchmark")


## LAPISAN 17 — MONITORING: PARSER SPARK EVENT LOG | Karakterisasi beban kerja

In [ ]:
def parse_event_log(logdir):
    tugas, tahap, apl = [], [], []
    berkas = sorted(list(Path(logdir).glob("*")) )
    for f in berkas:
        if f.is_dir():
            continue
        try:
            pembuka = gzip.open if f.suffix in (".gz", ".lz4", ".snappy") else open
            with pembuka(f, "rt", errors="ignore") as fh:
                for baris in fh:
                    baris = baris.strip()
                    if not baris.startswith("{"):
                        continue
                    try: ev = json.loads(baris)
                    except Exception: continue
                    tipe = ev.get("Event", "")
                    if tipe == "SparkListenerApplicationStart":
                        apl.append({"app": ev.get("App Name"),
                                    "app_id": ev.get("App ID"),
                                    "mulai": ev.get("Timestamp")})
                    elif tipe == "SparkListenerStageCompleted":
                        si = ev.get("Stage Info", {})
                        m = si.get("Accumulables", [])
                        def acc(nama):
                            for a in m:
                                if a.get("Name") == nama:
                                    try: return float(a.get("Value", 0))
                                    except Exception: return 0.0
                            return 0.0
                        tahap.append({
                            "berkas": f.name,
                            "stage_id": si.get("Stage ID"),
                            "nama": (si.get("Stage Name") or "")[:60],
                            "n_task": si.get("Number of Tasks"),
                            "submit": si.get("Submission Time"),
                            "selesai": si.get("Completion Time"),
                            "durasi_ms": (si.get("Completion Time") or 0) -
                                         (si.get("Submission Time") or 0),
                            "shuffle_read_mb": round(acc("internal.metrics.shuffle.read.remoteBytesRead")/1e6, 2),
                            "shuffle_write_mb": round(acc("internal.metrics.shuffle.write.bytesWritten")/1e6, 2),
                            "spill_disk_mb": round(acc("internal.metrics.diskBytesSpilled")/1e6, 2),
                            "spill_mem_mb": round(acc("internal.metrics.memoryBytesSpilled")/1e6, 2),
                            "gc_ms": acc("internal.metrics.jvmGCTime"),
                            "cpu_ns": acc("internal.metrics.executorCpuTime"),
                        })
                    elif tipe == "SparkListenerTaskEnd":
                        ti = ev.get("Task Info", {}); tm = ev.get("Task Metrics", {}) or {}
                        tugas.append({
                            "stage_id": ev.get("Stage ID"),
                            "task_id": ti.get("Task ID"),
                            "durasi_ms": ti.get("Finish Time", 0) - ti.get("Launch Time", 0),
                            "gagal": bool(ti.get("Failed")),
                            "gc_ms": tm.get("JVM GC Time", 0),
                            "spill_disk_mb": round(tm.get("Disk Bytes Spilled", 0)/1e6, 3),
                        })
        except Exception as ex:
            print(f"  lewati {f.name}: {ex}")
    return pd.DataFrame(apl), pd.DataFrame(tahap), pd.DataFrame(tugas)

with Tahap("parse event log", "MONITORING"):
    df_apl, df_tahap, df_tugas = parse_event_log(LOGDIR)
    print(f"  aplikasi : {len(df_apl):,}")
    print(f"  stage    : {len(df_tahap):,}")
    print(f"  task     : {len(df_tugas):,}")

    if len(df_tahap):
        df_tahap["durasi_s"] = df_tahap["durasi_ms"] / 1000
        df_tahap.to_csv(OUT / "monitoring_stage.csv", index=False)
        ring_stage = pd.DataFrame([{
            "total_stage": len(df_tahap),
            "total_task": int(df_tahap["n_task"].fillna(0).sum()),
            "durasi_total_menit": round(df_tahap["durasi_s"].sum()/60, 2),
            "stage_terlama_s": round(df_tahap["durasi_s"].max(), 2),
            "shuffle_read_total_mb": round(df_tahap["shuffle_read_mb"].sum(), 1),
            "shuffle_write_total_mb": round(df_tahap["shuffle_write_mb"].sum(), 1),
            "spill_disk_total_mb": round(df_tahap["spill_disk_mb"].sum(), 1),
            "gc_total_s": round(df_tahap["gc_ms"].sum()/1000, 1),
        }])
        ring_stage.to_csv(OUT / "monitoring_ringkasan.csv", index=False)
        print()
        print(ring_stage.T.to_string(header=False))
        print()
        print("10 stage terlama:")
        print(df_tahap.nlargest(10, "durasi_s")[
              ["stage_id","nama","n_task","durasi_s","shuffle_read_mb",
               "spill_disk_mb"]].to_string(index=False))
    if len(df_tugas):
        df_tugas.to_csv(OUT / "monitoring_task.csv", index=False)
        print()
        print(f"  task gagal: {int(df_tugas['gagal'].sum()):,}")
        print(f"  durasi task median: {df_tugas['durasi_ms'].median():.0f} ms")

dfm = simpan_monitor()
print()
print(dfm.to_string(index=False))

## LAPISAN 18 — VISUALIZATION LAYER | Figur statis

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

with Tahap("figur statis", "VISUALIZATION"):
    fig, ax = plt.subplots(2, 3, figsize=(20, 11))

    # (1) sebaran macro-F1 per skema
    sk_urut = list(SKEMA)
    box = [per_fold[per_fold["skema"] == s]["macro_f1"].dropna().values for s in sk_urut]
    bp = ax[0,0].boxplot(box, patch_artist=True,
                         labels=["Random","Spasial","Temporal","Klaster"])
    for pa, c in zip(bp["boxes"], ["tab:red","tab:blue","tab:green","tab:purple"]):
        pa.set_facecolor(c); pa.set_alpha(.6)
    ax[0,0].set_ylabel("macro-F1"); ax[0,0].set_title("Sebaran macro-F1 per skema CV")
    ax[0,0].grid(alpha=.25, axis="y")

    # (2) model x skema
    piv = per_fold.pivot_table(index="model", columns="skema",
                               values="macro_f1", aggfunc="mean")
    piv.plot(kind="bar", ax=ax[0,1], rot=25)
    ax[0,1].set_ylabel("macro-F1"); ax[0,1].set_title("Model x skema")
    ax[0,1].legend(fontsize=7); ax[0,1].grid(alpha=.25, axis="y")

    # (3) skalabilitas
    ok = df_bench.dropna(subset=["speedup"])
    ax[0,2].plot(ok["n_core"], ok["speedup"], "o-", label="terukur")
    ax[0,2].plot(ok["n_core"], ok["n_core"], "--", color="grey", label="ideal linear")
    ax[0,2].set_xlabel("jumlah core"); ax[0,2].set_ylabel("speedup")
    ax[0,2].set_title("Skalabilitas Spark"); ax[0,2].legend(); ax[0,2].grid(alpha=.25)

    # (4) Spark vs tanpa Spark
    df_sk = (sk_per_fold.groupby(["model"], as_index=False)
             .agg(n=("n_uji", "mean"), menit_fitur=("menit", "mean"),
                  menit_latih=("menit", "mean"),
                  status=("status", "first")))
    okk = df_sk[df_sk["status"] == "OK"]
    ax[1,0].plot(okk["n"], okk["menit_fitur"] + okk["menit_latih"], "o-",
                 label="scikit-learn (1 node)")
    gagal = df_sk[df_sk["status"] != "OK"]
    for _, r in gagal.iterrows():
        ax[1,0].axvline(r["n"], color="red", ls=":", alpha=.7)
    ax[1,0].set_xlabel("jumlah sekuens"); ax[1,0].set_ylabel("menit")
    ax[1,0].set_title("Batas single-node (garis merah = gagal)")
    ax[1,0].legend(fontsize=8); ax[1,0].grid(alpha=.25)

    # (5) pengayaan situs fungsional
    if len(df_enrich):
        pv = df_enrich.pivot_table(index="situs", columns="inang",
                                   values="rasio_pengayaan", aggfunc="mean")
        pv.plot(kind="barh", ax=ax[1,1])
        ax[1,1].axvline(1.0, color="grey", ls="--")
        ax[1,1].set_xlabel("rasio pengayaan (obs/null)")
        ax[1,1].set_title("Pengayaan k-mer pada situs fungsional HA")
        ax[1,1].legend(fontsize=7); ax[1,1].grid(alpha=.25, axis="x")

    # (6) distribusi ukuran komponen graf
    uk = uk_komp_pdf["ukuran"]
    ax[1,2].hist(uk, bins=60, log=True, color="tab:purple", alpha=.8)
    ax[1,2].set_xlabel("ukuran komponen"); ax[1,2].set_ylabel("frekuensi (log)")
    ax[1,2].set_title(f"Komponen graf kemiripan (n={len(uk):,})")
    ax[1,2].grid(alpha=.25)

    plt.tight_layout()
    plt.savefig(OUT / "ringkasan_visual.png", dpi=130)
    plt.close()
print("Tersimpan:", OUT / "ringkasan_visual.png")

## LAPISAN 18b — VISUALIZATION LAYER | Dashboard interaktif mandiri (HTML + Plotly)

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

with Tahap("bangun dashboard", "VISUALIZATION"):
    piv_sk = (per_fold.groupby("skema")["macro_f1"].agg(["mean","std"])
              .reindex(list(SKEMA)).reset_index())
    f1_ref = float(piv_sk.loc[0, "mean"])
    piv_sk["penurunan_pct"] = (100 * (f1_ref - piv_sk["mean"]) / max(f1_ref, 1e-9)).round(2)

    g1 = go.Figure()
    g1.add_bar(x=piv_sk["skema"], y=piv_sk["mean"],
               error_y=dict(type="data", array=piv_sk["std"].fillna(0)),
               marker_color=["#E24B4A","#378ADD","#639922","#7F77DD"])
    g1.update_layout(title="Insight 1 — Kebocoran evaluasi: macro-F1 per skema CV",
                     yaxis_title="macro-F1", height=420,
                     template="plotly_white")

    piv_m = per_fold.pivot_table(index="model", columns="skema",
                                 values="macro_f1", aggfunc="mean").reset_index()
    g2 = go.Figure()
    for s in SKEMA:
        if s in piv_m.columns:
            g2.add_bar(name=s, x=piv_m["model"], y=piv_m[s])
    g2.update_layout(barmode="group", title="Insight 2 — Robustness model terhadap skema evaluasi",
                     yaxis_title="macro-F1", height=420, template="plotly_white")

    geo_agg = (geo_pdf.groupby(["benua","inang"])["jumlah"].sum().reset_index())
    g3 = px.bar(geo_agg, x="benua", y="jumlah", color="inang", barmode="stack",
                title="Insight 3 — Confounding geografis: komposisi inang per benua",
                template="plotly_white", height=430)

    uk_pdf = uk_komp_pdf[["ukuran","n_inang","n_benua"]]
    g4 = px.histogram(uk_pdf, x="ukuran", log_y=True, nbins=60,
                      title=f"Insight 4 — Graph analytics: {int(n_klaster):,} komponen dari "
                            f"{int(n_vek):,} sekuens (rasio {RASIO_PSEUDO:.1f}x)",
                      template="plotly_white", height=400)

    okb = df_bench.dropna(subset=["speedup"])
    g5 = go.Figure()
    g5.add_scatter(x=okb["n_core"], y=okb["speedup"], mode="lines+markers", name="terukur")
    g5.add_scatter(x=okb["n_core"], y=okb["n_core"], mode="lines",
                   line=dict(dash="dash", color="grey"), name="ideal linear")
    g5.update_layout(title="Insight 5 — Skalabilitas platform big data",
                     xaxis_title="jumlah core", yaxis_title="speedup",
                     height=400, template="plotly_white")

    if len(df_enrich):
        g6 = px.bar(df_enrich, x="rasio_pengayaan", y="situs", color="inang",
                    orientation="h", barmode="group",
                    title="Insight 6 — Validasi biologis: pengayaan k-mer di situs fungsional HA",
                    template="plotly_white", height=470)
        g6.add_vline(x=1.0, line_dash="dash", line_color="grey")
    else:
        g6 = go.Figure()

    g7 = px.bar(prof_pdf.head(20), x="kom", y="ukuran", color="n_inang",
                title=f"Insight 7 — Community analytics: {n_kom:,} komunitas LPA, "
                      f"modularitas Q={Q:.3f}",
                template="plotly_white", height=400)
    g7.update_xaxes(type="category", title="komunitas")

    g8 = px.bar(dist_jarak, x="jarak", y="count",
                title=f"Insight 8 — Path analytics: distribusi jarak geodesik "
                      f"(APL={apl:.2f}, diameter~{diameter_est})",
                template="plotly_white", height=380)

    KPI = [
        ("Sekuens dianalisis", f"{int(N):,}"),
        ("Token k-mer", f"{VOLUME['token_kmer_total']:,}"),
        ("Ruang fitur", f"{N_FITUR:,}"),
        ("Model dilatih", f"{len(per_fold):,}"),
        ("Sisi graf", f"{int(n_sisi):,}"),
        ("Rasio pseudoreplikasi", f"{RASIO_PSEUDO:.2f}x"),
        ("Komunitas LPA", f"{n_kom:,}"),
        ("Modularitas Q", f"{Q:.3f}"),
        ("Penurunan blok klaster", f"{float(piv_sk.iloc[-1]['penurunan_pct']):.1f}%"),
        ("Speedup 8 core",
         f"{float(okb[okb.n_core==8]['speedup'].iloc[0]):.2f}x" if len(okb[okb.n_core==8]) else "n/a"),
    ]
    kpi_html = "".join(
        f"<div class='kpi'><div class='kpi-v'>{v}</div><div class='kpi-l'>{k_}</div></div>"
        for k_, v in KPI)

    bagian = []
    for judul, fig in [("Kebocoran evaluasi", g1), ("Robustness model", g2),
                       ("Confounding geografis", g3), ("Struktur graf", g4),
                       ("Skalabilitas", g5), ("Validasi biologis", g6),
                       ("Community analytics", g7), ("Path analytics", g8)]:
        bagian.append("<div class='card'>" +
                      pio.to_html(fig, include_plotlyjs=False, full_html=False) +
                      "</div>")

    HTML = (
      "<!DOCTYPE html><html lang='id'><head><meta charset='utf-8'>"
      "<title>Dashboard Influenza A Host Tropism</title>"
      "<script src='https://cdn.plot.ly/plotly-2.27.0.min.js'></script>"
      "<style>"
      "body{font-family:system-ui,sans-serif;margin:0;background:#f4f5f7;color:#1f2933}"
      "header{background:#12263f;color:#fff;padding:28px 40px}"
      "header h1{margin:0;font-size:24px;font-weight:600}"
      "header p{margin:6px 0 0;opacity:.8;font-size:14px}"
      ".kpis{display:flex;flex-wrap:wrap;gap:14px;padding:22px 40px}"
      ".kpi{background:#fff;border-radius:10px;padding:16px 22px;min-width:150px;"
      "box-shadow:0 1px 3px rgba(0,0,0,.08)}"
      ".kpi-v{font-size:22px;font-weight:600;color:#12263f}"
      ".kpi-l{font-size:12px;color:#69707a;margin-top:4px}"
      ".card{background:#fff;border-radius:10px;margin:14px 40px;padding:10px;"
      "box-shadow:0 1px 3px rgba(0,0,0,.08)}"
      "footer{padding:26px 40px;font-size:12px;color:#69707a}"
      "</style></head><body>"
      "<header><h1>Influenza A Host Tropism — Big Data Dashboard</h1>"
      "<p>Run " + CFG["run_id"] + " &middot; Batch processing &middot; Apache Spark " +
      SPARK_VERSI + " &middot; " + str(int(N)) + " sekuens HA</p></header>"
      "<div class='kpis'>" + kpi_html + "</div>" + "".join(bagian) +
      "<footer>Sumber data: NCBI Influenza Virus Database (Entrez E-utilities). "
      "Klasifikasi data dan kontrol akses diterapkan pada Security Layer; "
      "seluruh tahap tercatat pada audit trail " + AUDIT_LOG.name + ".</footer>"
      "</body></html>")

    dash_path = OUT / "dashboard.html"
    dash_path.write_text(HTML, encoding="utf-8")
    audit("dashboard", path=str(dash_path), ukuran_kb=round(len(HTML)/1024, 1))

print("Dashboard :", dash_path)
print(f"Ukuran    : {len(HTML)/1024:.0f} KB")

## LAPISAN 19 — RINGKASAN EKSEKUSI & PEMETAAN ARSITEKTUR

In [ ]:
ARSITEKTUR = [
 ("Data Sources",              "NCBI Entrez E-utilities (FASTA unstructured + metadata semi-terstruktur)", "SEL 1 blok DATA SOURCES"),
 ("Ingestion Layer",           "Batch harvest berjendela WebEnv + checkpoint idempotent + retry eksponensial", "blok INGESTION"),
 ("Hadoop Storage Layer",      "Parquet kolumnar Snappy berpartisi (segmen, benua) pada zona lake/stage/mart", "blok STORAGE"),
 ("Platform Management Layer", "Apache Spark (pengganti MapReduce), Spark SQL (pengganti Hive), Catalyst + AQE", "blok PLATFORM"),
 ("Analytics Engine: Text",    "CountVectorizer + IDF atas token k-mer", "blok ANALYTICS/TEXT"),
 ("Analytics Engine: Search",  "MinHash LSH approximate similarity join", "blok ANALYTICS/GRAPH"),
 ("Analytics Engine: ML",      "Spark MLlib: LR, LinearSVC, NaiveBayes, RandomForest, MLP", "blok ANALYTICS/ML"),
 ("Analytics Engine: Graph",   "Label propagation, PageRank iteratif, asortativitas", "blok ANALYTICS/GRAPH"),
 ("Analytics Engine: Stats",   "Bootstrap CI, Wilcoxon, paired t-test, uji permutasi", "blok ANALYTICS/STATISTIK"),
 ("Data Warehouse",            "Star schema mart: dim_sekuens, fakta_geo, fakta_model, fakta_graf", "blok WAREHOUSE"),
 ("Visualization Layer",       "Matplotlib figur statis + dashboard Plotly HTML mandiri", "blok VISUALIZATION"),
 ("Security Layer",            "Klasifikasi data, RBAC, pseudonimisasi HMAC, pemindaian PII, data quality gate, audit trail", "blok SECURITY"),
 ("Monitoring Layer",          "Instrumentasi tahap, Spark event log parser, benchmark skalabilitas", "blok MONITORING"),
 ("Infrastructure Layer",      "Bare metal workstation, Spark standalone local[N], NVMe Gen4 sebagai shuffle store", "bangun_spark()"),
]
df_arsi = pd.DataFrame(ARSITEKTUR, columns=["Lapisan","Implementasi","Lokasi di notebook"])
df_arsi.to_csv(OUT / "pemetaan_arsitektur.csv", index=False)
print(df_arsi.to_string(index=False))

In [ ]:
f1_sk = per_fold.groupby("skema")["macro_f1"].mean().reindex(list(SKEMA))
f1_r  = float(f1_sk.iloc[0])

RINGKASAN = {
    "run_id": CFG["run_id"],
    "mode": CFG["mode"],
    "spark_versi": SPARK_VERSI,
    "sekuens_HA": int(N),
    "sekuens_NA": int(n_na) if "n_na" in globals() else 0,
    "k_mer": K,
    "token_kmer_total": VOLUME["token_kmer_total"],
    "ruang_fitur": int(N_FITUR),
    "sel_matriks_teoretis": VOLUME["sel_matriks_teoretis"],
    "kelas": ",".join(LABEL_NAMA),
    "n_model_dilatih": int(len(per_fold)),
    "graf_simpul": int(n_vek),
    "graf_sisi": int(n_sisi),
    "graf_komponen": int(n_klaster),
    "rasio_pseudoreplikasi": round(RASIO_PSEUDO, 3),
    "asortativitas_inang": GRAF_STAT["asortativitas_inang"],
    "asortativitas_benua": GRAF_STAT["asortativitas_benua"],
    "komunitas_LPA": KOM_STAT["komunitas_LPA"],
    "modularitas_Q": KOM_STAT["modularitas_Q"],
    "segitiga": KOM_STAT["segitiga"],
    "clustering_coef": KOM_STAT["clustering_coef_rata2"],
    "k_core_maksimum": KONEKTIVITAS["k_core_maksimum"],
    "average_path_length": PATH_STAT["average_path_length"],
    "diameter_estimasi": PATH_STAT["diameter_estimasi"],
    "macroF1_random": round(f1_r, 4),
    "macroF1_spasial": round(float(f1_sk.iloc[1]), 4),
    "macroF1_temporal": round(float(f1_sk.iloc[2]), 4),
    "macroF1_klaster": round(float(f1_sk.iloc[3]), 4),
    "penurunan_spasial_pct": round(100*(f1_r-float(f1_sk.iloc[1]))/max(f1_r,1e-9), 2),
    "penurunan_temporal_pct": round(100*(f1_r-float(f1_sk.iloc[2]))/max(f1_r,1e-9), 2),
    "penurunan_klaster_pct": round(100*(f1_r-float(f1_sk.iloc[3]))/max(f1_r,1e-9), 2),
    "durasi_total_menit": round(float(pd.DataFrame(MONITOR)["menit"].sum()), 2),
    "stage_spark": int(len(df_tahap)),
    "task_spark": int(df_tahap["n_task"].fillna(0).sum()) if len(df_tahap) else 0,
}
df_ring = pd.DataFrame(list(RINGKASAN.items()), columns=["Metrik","Nilai"])
df_ring.to_csv(OUT / "ringkasan_akhir.csv", index=False)
json.dump(RINGKASAN, open(OUT / "ringkasan_akhir.json", "w"), indent=2, default=str)
audit("selesai_run", **{k_: str(v_) for k_, v_ in RINGKASAN.items()})

print("=" * 70)
print(" RINGKASAN EKSEKUSI")
print("=" * 70)
print(df_ring.to_string(index=False))
print("=" * 70)
print()
print("Berkas keluaran di", OUT)
for f in sorted(OUT.glob("*")):
    print(f"  {f.name:<42} {f.stat().st_size/1024:>10,.1f} KB")